# NIFTYBEES Price–NAV Early-Warning Project
## Complete Steps 1–8 Python workflow

This notebook combines all 14 Python scripts used in the final project. It preserves the repository workflow while adapting path handling for Jupyter. Run the cells from top to bottom.

**Study period:** 1 January 2021 to 16 November 2026  
**Primary outcome:** next-trading-day abnormal NIFTYBEES price–NAV dislocation  
**Primary evaluation:** purged walk-forward PR-AUC with robustness, uncertainty, calibration and operational-capacity checks

The notebook expects the original repository directories (`data`, `config`, `outputs`, `figures`, `report`, `docs`, and `scripts`) to remain beside the `notebooks` folder.


## Environment

Recommended Python: 3.11 or 3.12. Install dependencies once from the repository root:

```bash
pip install -r requirements.txt
```

The setup cell locates the repository whether Jupyter starts in the project root or inside `notebooks/`.


In [2]:
from pathlib import Path
import os

CURRENT = Path.cwd().resolve()
if (CURRENT / "scripts").is_dir() and (CURRENT / "data").is_dir():
    PROJECT_ROOT = CURRENT
elif (CURRENT.parent / "scripts").is_dir() and (CURRENT.parent / "data").is_dir():
    PROJECT_ROOT = CURRENT.parent
else:
    raise FileNotFoundError(
        "Repository root not found. Keep this notebook inside the project's notebooks folder "
        "or start Jupyter from the extracted project root."
    )

os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")
print(f"Python workflow ready: {(PROJECT_ROOT / 'requirements.txt').exists()}")

Project root: /workspace/scratch/e2d669a31e97/Niftybees-Price-NAV-Early-Warning
Python workflow ready: True
Completed in 0.0 seconds.


## Step 1 — Merge and audit official data

Original repository script: `scripts/01_merge_and_audit_data.py`


In [4]:
"""Merge the four official cleaned sources and create the analysis panel.

The script is deterministic and uses Date as the only merge key. It also
records missing NAV/VIX observations and the two source-zero VIX anomalies.
"""
from pathlib import Path

import numpy as np
import pandas as pd


ROOT = PROJECT_ROOT
DATA = ROOT / "data" / "processed"


def unique_dates(frame: pd.DataFrame, name: str) -> None:
    if frame["Date"].duplicated().any():
        bad = frame.loc[frame["Date"].duplicated(False), "Date"].dt.strftime("%Y-%m-%d").tolist()
        raise ValueError(f"{name} has duplicate dates: {bad[:10]}")


market = pd.read_csv(DATA / "NIFTYBEES_EQ_Market_Data_2021_to_2026_Clean.csv", parse_dates=["Date"])
nav = pd.read_csv(DATA / "NIFTYBEES_NAV_AMFI_official_2021_to_2026_Updated.csv")
nifty = pd.read_csv(DATA / "NIFTY50_Historical_2021_to_2026_Clean.csv", parse_dates=["Date"])
vix = pd.read_csv(DATA / "INDIA_VIX_Historical_2021_to_2026_Clean.csv", parse_dates=["Date"])

assert len(market) == 1456
assert set(market["Symbol"]) == {"NIFTYBEES"}
assert set(market["Series"]) == {"EQ"}
assert set(nav["Scheme Code"].astype(str)) == {"140084"}

nav["Date"] = pd.to_datetime(nav["AUDIT_DATE"])
for frame, name in [(market, "market"), (nav, "NAV"), (nifty, "NIFTY 50"), (vix, "India VIX")]:
    unique_dates(frame, name)

market = market.rename(columns={
    "Symbol": "ETF_Symbol", "Series": "ETF_Series", "Prev Close": "ETF_Prev_Close",
    "Open Price": "ETF_Open", "High Price": "ETF_High", "Low Price": "ETF_Low",
    "Last Price": "ETF_Last", "Close Price": "ETF_Close", "Average Price": "ETF_Average_Price",
    "Total Traded Quantity": "ETF_Volume", "Turnover ₹": "ETF_Turnover_INR",
    "No. of Trades": "ETF_Trades", "Deliverable Qty": "ETF_Deliverable_Qty",
    "% Dly Qt to Traded Qty": "ETF_Delivery_Pct",
})
market["ETF_Intraday_Range_Pct"] = ((market["ETF_High"] - market["ETF_Low"]) / market["ETF_Close"] * 100).round(4)
market["ETF_Extreme_Range_Flag"] = (market["ETF_Intraday_Range_Pct"] > 5).astype(int)
delivery_check = market["ETF_Deliverable_Qty"] / market["ETF_Volume"] * 100
market["ETF_Delivery_Audit_Flag"] = np.where(
    (delivery_check - market["ETF_Delivery_Pct"]).abs() <= 0.10, "OK", "CHECK_SOURCE"
)

market_cols = [
    "Date", "ETF_Symbol", "ETF_Series", "ETF_Prev_Close", "ETF_Open", "ETF_High", "ETF_Low",
    "ETF_Last", "ETF_Close", "ETF_Average_Price", "ETF_Volume", "ETF_Turnover_INR",
    "ETF_Trades", "ETF_Deliverable_Qty", "ETF_Delivery_Pct", "ETF_Intraday_Range_Pct",
    "ETF_Extreme_Range_Flag", "ETF_Delivery_Audit_Flag",
]
master = market[market_cols].copy()
master = master.merge(nav[["Date", "NAV"]], on="Date", how="left", validate="one_to_one")
master["Price_NAV_Dislocation_Pct"] = ((master["ETF_Close"] - master["NAV"]) / master["NAV"] * 100).round(6)
master["Abs_Dislocation_Pct"] = master["Price_NAV_Dislocation_Pct"].abs().round(6)

nifty = nifty.rename(columns={"Open": "NIFTY50_Open", "High": "NIFTY50_High", "Low": "NIFTY50_Low", "Close": "NIFTY50_Close"})
master = master.merge(nifty[["Date", "NIFTY50_Open", "NIFTY50_High", "NIFTY50_Low", "NIFTY50_Close"]], on="Date", how="left", validate="one_to_one")

vix = vix.rename(columns={
    "Open": "INDIA_VIX_Open", "High": "INDIA_VIX_High", "Low": "INDIA_VIX_Low",
    "Close": "INDIA_VIX_Close", "% Change": "INDIA_VIX_Pct_Change",
    "Data Quality Flag": "VIX_Quality_Flag",
})
master = master.merge(vix[["Date", "INDIA_VIX_Open", "INDIA_VIX_High", "INDIA_VIX_Low", "INDIA_VIX_Close", "INDIA_VIX_Pct_Change", "VIX_Quality_Flag"]], on="Date", how="left", validate="one_to_one")
master["VIX_Quality_Flag"] = master["VIX_Quality_Flag"].fillna("MISSING")
master["NAV_Available"] = master["NAV"].notna().astype(int)
master["VIX_Available"] = master["INDIA_VIX_Close"].notna().astype(int)
master["Complete_Case"] = (
    master["NAV_Available"].eq(1)
    & master["NIFTY50_Close"].notna()
    & master["VIX_Available"].eq(1)
    & master["VIX_Quality_Flag"].eq("OK")
).astype(int)


def quality_note(row: pd.Series) -> str:
    notes = []
    if not row["NAV_Available"]:
        notes.append("NAV_MISSING")
    if not row["VIX_Available"]:
        notes.append("VIX_MISSING")
    elif row["VIX_Quality_Flag"] != "OK":
        notes.append(str(row["VIX_Quality_Flag"]))
    return "|".join(notes) if notes else "OK"


master["Data_Quality_Notes"] = master.apply(quality_note, axis=1)
master = master.sort_values("Date").reset_index(drop=True)
analysis = master.loc[master["Complete_Case"].eq(1)].copy()

master.to_csv(DATA / "ETF_Master_Merged_2021_to_2026.csv", index=False, date_format="%Y-%m-%d", float_format="%.6f")
analysis.to_csv(DATA / "ETF_Analysis_Ready_2021_to_2026.csv", index=False, date_format="%Y-%m-%d", float_format="%.6f")

assert len(master) == 1456
assert len(analysis) == 1446
assert master["NAV_Available"].sum() == 1448
assert (master["VIX_Quality_Flag"] == "SOURCE_ZERO_ANOMALY").sum() == 2
print({"master_rows": len(master), "complete_rows": len(analysis), "first": str(master.Date.min().date()), "last": str(master.Date.max().date())})


{'master_rows': 1456, 'complete_rows': 1446, 'first': '2021-01-01', 'last': '2026-11-16'}
Completed in 0.7 seconds.


## Step 2 — Construct the prospective label and leakage-safe dataset

Original repository script: `scripts/02_build_leakage_safe_dataset.py`


In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

ROOT = PROJECT_ROOT
DATA_DIR = ROOT / 'data' / 'processed'
OUTDIR = ROOT / 'outputs'
OUTDIR.mkdir(parents=True, exist_ok=True)
MODEL_CSV = DATA_DIR / 'ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv'
INFILE = DATA_DIR / 'ETF_Master_Merged_2021_to_2026.csv'
AUDIT_XLSX = OUTDIR / 'ETF_Label_and_Leakage_Audit_2021_to_2026.xlsx'
SUMMARY_JSON = OUTDIR / 'modeling_dataset_summary.json'

m = pd.read_csv(INFILE, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)
# Preserve the actual NSE trading calendar before filtering incomplete observations.
m['Next_Market_Date'] = m['Date'].shift(-1)
m['Next_Complete_Case'] = m['Complete_Case'].shift(-1)
m['Next_Abs_Dislocation_Pct'] = m['Abs_Dislocation_Pct'].shift(-1)
m['Next_Signed_Dislocation_Pct'] = m['Price_NAV_Dislocation_Pct'].shift(-1)

# Feature engineering is performed on complete observations only. Every X value uses date t or earlier.
v = m.loc[m['Complete_Case'].eq(1)].copy().reset_index(drop=False).rename(columns={'index':'Master_Row'})
v['Prediction_Date'] = v['Date']
v['Target_Date'] = v['Next_Market_Date']

# Returns available after the close/NAV publication on t.
v['X_ETF_Return_1D_Pct'] = v['ETF_Close'].pct_change(fill_method=None) * 100
v['X_NAV_Return_1D_Pct'] = v['NAV'].pct_change(fill_method=None) * 100
v['X_NIFTY50_Return_1D_Pct'] = v['NIFTY50_Close'].pct_change(fill_method=None) * 100
v['X_Tracking_Return_Difference_1D_Pct'] = v['X_ETF_Return_1D_Pct'] - v['X_NIFTY50_Return_1D_Pct']
v['X_ETF_Return_5D_Pct'] = (v['ETF_Close'] / v['ETF_Close'].shift(5) - 1) * 100
v['X_NIFTY50_Return_5D_Pct'] = (v['NIFTY50_Close'] / v['NIFTY50_Close'].shift(5) - 1) * 100
v['X_VIX_Change_5D_Pct'] = (v['INDIA_VIX_Close'] / v['INDIA_VIX_Close'].shift(5) - 1) * 100

# Price–NAV state and lags.
v['X_Dislocation_Pct_t'] = v['Price_NAV_Dislocation_Pct']
v['X_Abs_Dislocation_Pct_t'] = v['Abs_Dislocation_Pct']
for lag in (1, 2, 5):
    v[f'X_Dislocation_Lag_{lag}_Pct'] = v['Price_NAV_Dislocation_Pct'].shift(lag)
v['X_Premium_Flag_t'] = (v['Price_NAV_Dislocation_Pct'] > 0).astype(int)
v['X_Abs_Dislocation_Mean_5D_Pct'] = v['Abs_Dislocation_Pct'].rolling(5).mean()
v['X_Abs_Dislocation_Mean_20D_Pct'] = v['Abs_Dislocation_Pct'].rolling(20).mean()
v['X_Abs_Dislocation_Std_20D_Pct'] = v['Abs_Dislocation_Pct'].rolling(20).std(ddof=1)

# Liquidity and market controls known by t.
v['X_Log_ETF_Volume_t'] = np.log1p(v['ETF_Volume'])
v['X_Volume_to_20D_Median'] = v['ETF_Volume'] / v['ETF_Volume'].rolling(20).median()
v['X_Log_ETF_Turnover_t'] = np.log1p(v['ETF_Turnover_INR'])
v['X_Average_Trade_Value_INR_t'] = v['ETF_Turnover_INR'] / v['ETF_Trades'].replace(0, np.nan)
v['X_Delivery_Pct_t'] = v['ETF_Delivery_Pct']
v['X_Intraday_Range_Pct_t'] = v['ETF_Intraday_Range_Pct']
v['X_NIFTY50_Close_t'] = v['NIFTY50_Close']
v['X_INDIA_VIX_Close_t'] = v['INDIA_VIX_Close']
v['X_INDIA_VIX_Change_1D_Pct'] = v['INDIA_VIX_Pct_Change']

# Dynamic label threshold: 95th percentile of the 252 complete observations available through t.
window = 252
v['Rolling_Threshold_95Pct'] = v['Abs_Dislocation_Pct'].rolling(window=window, min_periods=window).quantile(0.95)
v['Threshold_Observation_Count'] = v['Abs_Dislocation_Pct'].rolling(window=window, min_periods=1).count().astype(int)
v['Threshold_Window_Start_Date'] = v['Prediction_Date'].shift(window-1)
v['X_Current_Abs_Gap_to_Threshold_Ratio'] = v['X_Abs_Dislocation_Pct_t'] / v['Rolling_Threshold_95Pct']

# Label is valid only when the immediately following NSE market session is also complete.
v['Target_Abs_Dislocation_Pct'] = v['Next_Abs_Dislocation_Pct']
v['Target_Signed_Dislocation_Pct'] = v['Next_Signed_Dislocation_Pct']
v['Y_Abnormal_Dislocation_95Q'] = (v['Target_Abs_Dislocation_Pct'] > v['Rolling_Threshold_95Pct']).astype('Int64')
v['Y_Above_0_50_Pct'] = (v['Target_Abs_Dislocation_Pct'] > 0.50).astype('Int64')
v['Y_Above_1_00_Pct'] = (v['Target_Abs_Dislocation_Pct'] > 1.00).astype('Int64')

# Eligibility: full historical window, next actual market day complete, all chosen predictors available.
feature_cols = [
    'X_Dislocation_Pct_t','X_Abs_Dislocation_Pct_t','X_Dislocation_Lag_1_Pct',
    'X_Dislocation_Lag_2_Pct','X_Dislocation_Lag_5_Pct','X_Premium_Flag_t',
    'X_Abs_Dislocation_Mean_5D_Pct','X_Abs_Dislocation_Mean_20D_Pct',
    'X_Abs_Dislocation_Std_20D_Pct','X_Current_Abs_Gap_to_Threshold_Ratio',
    'X_ETF_Return_1D_Pct','X_NAV_Return_1D_Pct','X_NIFTY50_Return_1D_Pct',
    'X_Tracking_Return_Difference_1D_Pct','X_ETF_Return_5D_Pct',
    'X_NIFTY50_Return_5D_Pct','X_INDIA_VIX_Close_t','X_INDIA_VIX_Change_1D_Pct',
    'X_VIX_Change_5D_Pct','X_Log_ETF_Volume_t','X_Volume_to_20D_Median',
    'X_Log_ETF_Turnover_t','X_Average_Trade_Value_INR_t','X_Delivery_Pct_t',
    'X_Intraday_Range_Pct_t'
]
v['Has_252_Observation_History'] = v['Rolling_Threshold_95Pct'].notna().astype(int)
v['Next_Actual_Market_Day_Complete'] = v['Next_Complete_Case'].eq(1).astype(int)
v['All_Features_Available'] = v[feature_cols].notna().all(axis=1).astype(int)
v['Initial_Eligible'] = (
    v['Has_252_Observation_History'].eq(1) &
    v['Next_Actual_Market_Day_Complete'].eq(1) &
    v['All_Features_Available'].eq(1) &
    v['Target_Abs_Dislocation_Pct'].notna()
).astype(int)

# Fixed chronological holdout defined by target date.
def split_name(d):
    if pd.isna(d): return 'Not eligible'
    if d <= pd.Timestamp('2024-12-31'): return 'Train'
    if d <= pd.Timestamp('2025-12-31'): return 'Validation'
    return 'Test'
v['Chronological_Split'] = v['Target_Date'].map(split_name)

# Purge one overlapping label interval at each boundary.
v['Purged_Boundary_Overlap'] = 0
eligible_idx = v.index[v['Initial_Eligible'].eq(1)].tolist()
for prior_split, next_split in [('Train','Validation'), ('Validation','Test')]:
    nxt = v.index[(v['Initial_Eligible'].eq(1)) & (v['Chronological_Split'].eq(next_split))]
    prv = v.index[(v['Initial_Eligible'].eq(1)) & (v['Chronological_Split'].eq(prior_split))]
    if len(nxt) and len(prv):
        first_next_prediction = v.loc[nxt[0], 'Prediction_Date']
        overlap = [i for i in prv if v.loc[i, 'Target_Date'] >= first_next_prediction]
        if overlap:
            v.loc[overlap, 'Purged_Boundary_Overlap'] = 1
v['Leakage_Safe_Model_Use'] = (v['Initial_Eligible'].eq(1) & v['Purged_Boundary_Overlap'].eq(0)).astype(int)

# Data provenance and timing audit fields.
v['Prediction_Timestamp_Rule'] = 'After date-t AMFI NAV publication; all date-t market data known'
v['Forecast_Horizon_Trading_Days'] = 1
v['Feature_Cutoff_Date'] = v['Prediction_Date']
v['Label_Window_End_Date'] = v['Target_Date']

model_cols = [
    'Prediction_Date','Target_Date','Prediction_Timestamp_Rule','Forecast_Horizon_Trading_Days',
    'Chronological_Split','Feature_Cutoff_Date','Threshold_Window_Start_Date',
    'Threshold_Observation_Count','Rolling_Threshold_95Pct',
] + feature_cols + [
    'Target_Signed_Dislocation_Pct','Target_Abs_Dislocation_Pct',
    'Y_Abnormal_Dislocation_95Q','Y_Above_0_50_Pct','Y_Above_1_00_Pct'
]
model = v.loc[v['Leakage_Safe_Model_Use'].eq(1), model_cols].copy()
for c in ['Prediction_Date','Target_Date','Feature_Cutoff_Date','Threshold_Window_Start_Date']:
    model[c] = pd.to_datetime(model[c]).dt.strftime('%Y-%m-%d')
model.to_csv(MODEL_CSV, index=False, float_format='%.8f')

# Audit tables.
audit_cols = [
    'Prediction_Date','Target_Date','Master_Row','Has_252_Observation_History',
    'Next_Actual_Market_Day_Complete','All_Features_Available','Initial_Eligible',
    'Chronological_Split','Purged_Boundary_Overlap','Leakage_Safe_Model_Use',
    'Threshold_Window_Start_Date','Threshold_Observation_Count','Rolling_Threshold_95Pct',
    'Target_Abs_Dislocation_Pct','Y_Abnormal_Dislocation_95Q'
]
audit = v[audit_cols].copy()

used = v[v['Leakage_Safe_Model_Use'].eq(1)].copy()
summary = used.groupby('Chronological_Split', sort=False).agg(
    Observations=('Prediction_Date','size'),
    First_Prediction_Date=('Prediction_Date','min'),
    Last_Prediction_Date=('Prediction_Date','max'),
    First_Target_Date=('Target_Date','min'),
    Last_Target_Date=('Target_Date','max'),
    Abnormal_Events=('Y_Abnormal_Dislocation_95Q','sum'),
    Above_0_50_Pct=('Y_Above_0_50_Pct','sum'),
    Above_1_00_Pct=('Y_Above_1_00_Pct','sum'),
).reset_index()
summary['Abnormal_Event_Rate_Pct'] = summary['Abnormal_Events'] / summary['Observations'] * 100
summary.loc[len(summary)] = [
    'Total', len(used), used['Prediction_Date'].min(), used['Prediction_Date'].max(),
    used['Target_Date'].min(), used['Target_Date'].max(),
    int(used['Y_Abnormal_Dislocation_95Q'].sum()), int(used['Y_Above_0_50_Pct'].sum()),
    int(used['Y_Above_1_00_Pct'].sum()), used['Y_Abnormal_Dislocation_95Q'].mean()*100
]

by_year = used.assign(Target_Year=used['Target_Date'].dt.year).groupby('Target_Year').agg(
    Observations=('Prediction_Date','size'),
    Abnormal_Events=('Y_Abnormal_Dislocation_95Q','sum'),
    Mean_Target_Abs_Gap_Pct=('Target_Abs_Dislocation_Pct','mean'),
    Maximum_Target_Abs_Gap_Pct=('Target_Abs_Dislocation_Pct','max')
).reset_index()
by_year['Abnormal_Event_Rate_Pct'] = by_year['Abnormal_Events']/by_year['Observations']*100

events = used.loc[used['Y_Abnormal_Dislocation_95Q'].eq(1), [
    'Prediction_Date','Target_Date','Chronological_Split','Rolling_Threshold_95Pct',
    'Target_Signed_Dislocation_Pct','Target_Abs_Dislocation_Pct'
]].sort_values('Target_Date')

checks = pd.DataFrame([
    ['Feature dates do not exceed prediction date', bool((used['Feature_Cutoff_Date'] <= used['Prediction_Date']).all()), 'Feature_Cutoff_Date equals Prediction_Date'],
    ['Target is the immediately next NSE market date', bool((used['Target_Date'] > used['Prediction_Date']).all()), 'Built from next row of the full market calendar'],
    ['Threshold excludes target day', bool((used['Threshold_Window_Start_Date'] <= used['Prediction_Date']).all()), '252 observations ending on prediction date t'],
    ['Exactly 252 observations in threshold', bool((used['Threshold_Observation_Count'] == 252).all()), 'Required before a row enters modelling data'],
    ['No missing selected predictors', bool(used[feature_cols].notna().all().all()), 'All feature columns complete'],
    ['No missing primary labels', bool(used['Y_Abnormal_Dislocation_95Q'].notna().all()), 'Primary label complete'],
    ['Purged rows excluded', bool((used['Purged_Boundary_Overlap'] == 0).all()), 'One-step overlapping intervals removed at split boundaries'],
    ['Prediction dates unique', bool(used['Prediction_Date'].is_unique), 'One forecast per eligible market date'],
], columns=['Leakage_or_Quality_Check','Passed','Evidence'])

methodology = [
    ('Objective','Predict whether NIFTYBEES will experience an abnormal absolute price–NAV dislocation on the next actual NSE trading day.'),
    ('Prediction timestamp','After AMFI publishes NAV for trading date t. ETF, NIFTY 50 and India VIX closing information for t is then known.'),
    ('Forecast horizon','One actual NSE trading day: t+1.'),
    ('Primary label','1 when absolute dislocation on t+1 is greater than the rolling 95th percentile calculated from 252 complete observations ending on t; otherwise 0.'),
    ('Dislocation formula','100 × (ETF closing market price − NAV) ÷ NAV.'),
    ('Why label is leakage-safe','The target-day dislocation is compared with a threshold frozen on t. It is never used to calculate its own threshold.'),
    ('Split rule','Train: target dates through 31-Dec-2024; Validation: target dates in 2025; Test: target dates in 2026.'),
    ('Purging rule','At each split boundary, any earlier-split observation whose label interval reaches the first prediction date of the next split is excluded.'),
    ('Sensitivity labels','Separate fixed thresholds above 0.50% and above 1.00% are retained for robustness checks.'),
    ('Missing-data rule','A row is excluded if date t is incomplete, the immediately next market session is incomplete, 252 observations are unavailable, or a selected feature is missing.'),
]

dictionary_rows = []
for c in model_cols:
    role = 'Identifier / timing'
    if c.startswith('X_'): role = 'Predictor available by date t'
    elif c.startswith('Y_'): role = 'Future outcome label'
    elif c.startswith('Target_'): role = 'Future outcome audit value'
    elif c in ['Rolling_Threshold_95Pct','Threshold_Observation_Count','Threshold_Window_Start_Date']: role = 'Label construction / audit'
    dictionary_rows.append((c, role))
data_dictionary = pd.DataFrame(dictionary_rows, columns=['Column','Role'])

# Create formatted workbook.
wb = Workbook()
wb.remove(wb.active)
navy='17365D'; blue='4472C4'; light='D9EAF7'; green='E2F0D9'; red='FCE4D6'; white='FFFFFF'; gray='E7E6E6'
thin=Side(style='thin', color='B7B7B7')

def add_df_sheet(name, df, freeze='A2', autofilter=True):
    ws=wb.create_sheet(name)
    for j, col in enumerate(df.columns, 1):
        cell=ws.cell(1,j,col)
        cell.fill=PatternFill('solid', fgColor=navy)
        cell.font=Font(color=white,bold=True)
        cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
        cell.border=Border(bottom=thin)
    for i, row in enumerate(df.itertuples(index=False, name=None), 2):
        for j, val in enumerate(row,1):
            if isinstance(val, pd.Timestamp): val=val.to_pydatetime()
            if pd.isna(val): val=None
            ws.cell(i,j,val)
    if freeze: ws.freeze_panes=freeze
    if autofilter and len(df): ws.auto_filter.ref=f'A1:{get_column_letter(len(df.columns))}{len(df)+1}'
    for j,col in enumerate(df.columns,1):
        vals=[str(col)] + [str(x) for x in df.iloc[:500,j-1].dropna().tolist()]
        width=min(max(max(map(len,vals))+2,12),38)
        ws.column_dimensions[get_column_letter(j)].width=width
    ws.sheet_view.showGridLines=False
    return ws

meth_df=pd.DataFrame(methodology,columns=['Item','Decision'])
ws=add_df_sheet('Methodology',meth_df,autofilter=False)
ws.column_dimensions['A'].width=27; ws.column_dimensions['B'].width=115
for row in ws.iter_rows(min_row=2):
    row[0].font=Font(bold=True,color=navy); row[1].alignment=Alignment(wrap_text=True,vertical='top')
    row[0].alignment=Alignment(wrap_text=True,vertical='top')
    ws.row_dimensions[row[0].row].height=42

ws=add_df_sheet('Split Summary',summary)
for c in range(2,7):
    for r in range(2,len(summary)+2): ws.cell(r,c).number_format='dd-mmm-yyyy' if c in (3,4,5,6) else '0'
for r in range(2,len(summary)+2): ws.cell(r,10).number_format='0.00%'
# Correct percentage display since values are already percentage units.
for r in range(2,len(summary)+2): ws.cell(r,10).number_format='0.00'
add_df_sheet('Year Summary',by_year)
add_df_sheet('Abnormal Events',events)
ws=add_df_sheet('Leakage Checks',checks)
for r in range(2,len(checks)+2):
    ws.cell(r,2).fill=PatternFill('solid',fgColor=green if ws.cell(r,2).value else red)
    ws.cell(r,2).font=Font(bold=True)
add_df_sheet('Candidate Audit',audit)
add_df_sheet('Data Dictionary',data_dictionary)
wb.save(AUDIT_XLSX)

result = {
    'input_rows': int(len(m)),
    'complete_rows': int(len(v)),
    'model_rows_after_purge': int(len(model)),
    'first_prediction_date': str(model['Prediction_Date'].min()),
    'last_prediction_date': str(model['Prediction_Date'].max()),
    'first_target_date': str(model['Target_Date'].min()),
    'last_target_date': str(model['Target_Date'].max()),
    'primary_events': int(model['Y_Abnormal_Dislocation_95Q'].sum()),
    'primary_event_rate_pct': float(model['Y_Abnormal_Dislocation_95Q'].mean()*100),
    'purged_boundary_rows': int(v['Purged_Boundary_Overlap'].sum()),
    'all_checks_passed': bool(checks['Passed'].all()),
    'split_summary': summary.astype(str).to_dict(orient='records'),
    'model_csv': str(MODEL_CSV.relative_to(ROOT)),
    'audit_xlsx': str(AUDIT_XLSX.relative_to(ROOT)),
}
SUMMARY_JSON.write_text(json.dumps(result,indent=2))
print(json.dumps(result,indent=2))


{
  "input_rows": 1456,
  "complete_rows": 1446,
  "model_rows_after_purge": 1185,
  "first_prediction_date": "2022-01-11",
  "last_prediction_date": "2026-11-13",
  "first_target_date": "2022-01-12",
  "last_target_date": "2026-11-16",
  "primary_events": 80,
  "primary_event_rate_pct": 6.751054852320674,
  "purged_boundary_rows": 2,
  "all_checks_passed": true,
  "split_summary": [
    {
      "Chronological_Split": "Train",
      "Observations": "725",
      "First_Prediction_Date": "2022-01-11",
      "Last_Prediction_Date": "2024-12-27",
      "First_Target_Date": "2022-01-12",
      "Last_Target_Date": "2024-12-30",
      "Abnormal_Events": "41",
      "Above_0_50_Pct": "4",
      "Above_1_00_Pct": "2",
      "Abnormal_Event_Rate_Pct": "5.655172413793103"
    },
    {
      "Chronological_Split": "Validation",
      "Observations": "246",
      "First_Prediction_Date": "2024-12-31",
      "Last_Prediction_Date": "2025-12-29",
      "First_Target_Date": "2025-01-01",
      "Last_T

## Step 3 — Train and compare benchmark and machine-learning models

Original repository script: `scripts/03_train_and_evaluate_models.py`


In [8]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, log_loss, precision_score,
    recall_score, roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text

warnings.filterwarnings('ignore')

ROOT = PROJECT_ROOT
OUT = ROOT / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)
INPUT = ROOT / 'data' / 'processed' / 'ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv'
PREDICTIONS = OUT / 'ETF_Model_Comparison_Predictions_2022_to_2026.csv'
RESULTS_JSON = OUT / 'model_results.json'

df = pd.read_csv(INPUT, parse_dates=['Prediction_Date', 'Target_Date'])
feature_cols = [c for c in df.columns if c.startswith('X_')]
target = 'Y_Abnormal_Dislocation_95Q'

train = df[df['Chronological_Split'].eq('Train')].copy()
valid = df[df['Chronological_Split'].eq('Validation')].copy()
test = df[df['Chronological_Split'].eq('Test')].copy()

X_train, y_train = train[feature_cols], train[target].astype(int)
X_valid, y_valid = valid[feature_cols], valid[target].astype(int)
X_test, y_test = test[feature_cols], test[target].astype(int)

def safe_auc(y, score):
    return float(roc_auc_score(y, score)) if len(np.unique(y)) > 1 else None

def top_budget_metrics(y, score, budget=0.10):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    k = max(1, int(math.ceil(len(y) * budget)))
    order = np.argsort(-score, kind='stable')
    alert = np.zeros(len(y), dtype=int)
    alert[order[:k]] = 1
    tp = int(((alert == 1) & (y == 1)).sum())
    return {
        'Budget_Pct': budget * 100,
        'Alerts': int(k),
        'True_Positives': tp,
        'Precision_at_Budget': float(tp / k),
        'Recall_at_Budget': float(tp / max(1, y.sum())),
        'Alert_Flag': alert,
    }

def fixed_threshold_metrics(y, score, threshold):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    alert = (score >= threshold).astype(int)
    tp = int(((alert == 1) & (y == 1)).sum())
    alerts = int(alert.sum())
    return {
        'Locked_Threshold': float(threshold),
        'Alerts': alerts,
        'Alert_Rate_Pct': float(alert.mean() * 100),
        'True_Positives': tp,
        'Precision': float(tp / alerts) if alerts else 0.0,
        'Recall': float(tp / max(1, y.sum())),
        'Alert_Flag': alert,
    }

def validation_budget_threshold(score, budget=0.10):
    score = np.asarray(score, dtype=float)
    k = max(1, int(math.ceil(len(score) * budget)))
    return float(np.sort(score)[::-1][k - 1])

def calibration_stats(y, prob):
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    logit = np.log(p / (1 - p)).reshape(-1, 1)
    slope_model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=5000)
    slope_model.fit(logit, y)
    # Equal-frequency expected calibration error.
    q = min(5, len(np.unique(p)))
    if q < 2:
        ece = float(abs(y.mean() - p.mean()))
    else:
        bins = pd.qcut(p, q=q, duplicates='drop')
        tmp = pd.DataFrame({'y': y, 'p': p, 'bin': bins})
        agg = tmp.groupby('bin', observed=True).agg(n=('y','size'), actual=('y','mean'), predicted=('p','mean'))
        ece = float(((agg['n']/len(tmp)) * (agg['actual']-agg['predicted']).abs()).sum())
    return {
        'Brier_Score': float(brier_score_loss(y, p)),
        'Log_Loss': float(log_loss(y, p, labels=[0,1])),
        'Mean_Predicted_Probability': float(p.mean()),
        'Actual_Event_Rate': float(y.mean()),
        'Calibration_Intercept': float(slope_model.intercept_[0]),
        'Calibration_Slope': float(slope_model.coef_[0][0]),
        'ECE_5_Bins': ece,
    }

def calibration_table(y, prob, model, split):
    temp = pd.DataFrame({'Actual': np.asarray(y, dtype=int), 'Probability': np.asarray(prob, dtype=float)})
    temp['Bin'] = pd.qcut(temp['Probability'].rank(method='first'), q=5, labels=['1 Lowest','2','3','4','5 Highest'])
    out = temp.groupby('Bin', observed=False).agg(
        Observations=('Actual','size'), Actual_Events=('Actual','sum'),
        Actual_Event_Rate=('Actual','mean'), Mean_Predicted_Probability=('Probability','mean'),
        Min_Probability=('Probability','min'), Max_Probability=('Probability','max')
    ).reset_index()
    out.insert(0, 'Split', split)
    out.insert(0, 'Model', model)
    return out

models = {}
tuning_rows = []

# Logistic regression: tune C using validation PR-AUC only.
for C in [0.01, 0.1, 1.0, 10.0]:
    model = Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(C=C, penalty='l2', solver='liblinear', max_iter=5000, random_state=42))
    ])
    model.fit(X_train, y_train)
    pv = model.predict_proba(X_valid)[:, 1]
    tuning_rows.append({'Model':'Logistic Regression','Parameters':f'C={C}','Validation_PR_AUC':average_precision_score(y_valid,pv),'Validation_Brier':brier_score_loss(y_valid,pv)})
best_log = max([r for r in tuning_rows if r['Model']=='Logistic Regression'], key=lambda r:(r['Validation_PR_AUC'],-r['Validation_Brier']))
best_C = float(best_log['Parameters'].split('=')[1])
log_model = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(C=best_C, penalty='l2', solver='liblinear', max_iter=5000, random_state=42))
])
log_model.fit(X_train, y_train)
models['Logistic Regression'] = log_model

# Small decision tree: transparent classifier with tuning limited to depth and leaf size.
for depth in [2, 3, 4, 5]:
    for leaf in [10, 25, 50]:
        for weight in [None, 'balanced']:
            model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, class_weight=weight, random_state=42)
            model.fit(X_train, y_train)
            pv = model.predict_proba(X_valid)[:, 1]
            tuning_rows.append({'Model':'Decision Tree','Parameters':f'depth={depth}; leaf={leaf}; weight={weight}','Validation_PR_AUC':average_precision_score(y_valid,pv),'Validation_Brier':brier_score_loss(y_valid,pv)})
best_tree = max([r for r in tuning_rows if r['Model']=='Decision Tree'], key=lambda r:(r['Validation_PR_AUC'],-r['Validation_Brier']))
parts = dict(p.split('=') for p in best_tree['Parameters'].split('; '))
tree_model = DecisionTreeClassifier(
    max_depth=int(parts['depth']), min_samples_leaf=int(parts['leaf']),
    class_weight=None if parts['weight']=='None' else parts['weight'], random_state=42
)
tree_model.fit(X_train, y_train)
models['Decision Tree'] = tree_model

# Random forest: modest grid suitable for a college project.
for depth in [3, 5, None]:
    for leaf in [5, 10, 20]:
        for max_features in ['sqrt', 0.5]:
            for weight in [None, 'balanced_subsample']:
                model = RandomForestClassifier(
                    n_estimators=400, max_depth=depth, min_samples_leaf=leaf,
                    max_features=max_features, class_weight=weight,
                    random_state=42, n_jobs=-1
                )
                model.fit(X_train, y_train)
                pv = model.predict_proba(X_valid)[:, 1]
                tuning_rows.append({
                    'Model':'Random Forest',
                    'Parameters':f'depth={depth}; leaf={leaf}; max_features={max_features}; weight={weight}',
                    'Validation_PR_AUC':average_precision_score(y_valid,pv),
                    'Validation_Brier':brier_score_loss(y_valid,pv)
                })
best_rf = max([r for r in tuning_rows if r['Model']=='Random Forest'], key=lambda r:(r['Validation_PR_AUC'],-r['Validation_Brier']))
parts = dict(p.split('=') for p in best_rf['Parameters'].split('; '))
rf_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=None if parts['depth']=='None' else int(parts['depth']),
    min_samples_leaf=int(parts['leaf']),
    max_features='sqrt' if parts['max_features']=='sqrt' else float(parts['max_features']),
    class_weight=None if parts['weight']=='None' else parts['weight'],
    random_state=42, n_jobs=-1
)
rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model

scores = {}
prevalence = float(y_train.mean())
for split_name, split_df in [('Train', train), ('Validation', valid), ('Test', test)]:
    scores[('Historical Prevalence', split_name)] = np.repeat(prevalence, len(split_df))
    scores[('Current Gap Rule', split_name)] = split_df['X_Current_Abs_Gap_to_Threshold_Ratio'].to_numpy(float)
    for name, model in models.items():
        scores[(name, split_name)] = model.predict_proba(split_df[feature_cols])[:, 1]

metric_rows = []
budget_rows = []
locked_rows = []
calibration_rows = []
probability_models = ['Historical Prevalence', 'Logistic Regression', 'Decision Tree', 'Random Forest']
all_models = ['Historical Prevalence', 'Current Gap Rule', 'Logistic Regression', 'Decision Tree', 'Random Forest']
operational_models = ['Current Gap Rule', 'Logistic Regression', 'Decision Tree', 'Random Forest']
split_map = {'Train':train, 'Validation':valid, 'Test':test}

for name in all_models:
    valid_score = scores[(name, 'Validation')]
    locked_threshold = validation_budget_threshold(valid_score, 0.10)
    for split_name in ['Validation', 'Test']:
        split_df = split_map[split_name]
        y = split_df[target].astype(int).to_numpy()
        score = scores[(name, split_name)]
        row = {
            'Model': name, 'Split': split_name, 'Observations':len(y), 'Events':int(y.sum()),
            'Event_Rate_Pct':float(y.mean()*100),
            'PR_AUC':float(average_precision_score(y, score)),
            'ROC_AUC':safe_auc(y, score),
        }
        if name in probability_models:
            row.update(calibration_stats(y, score))
            calibration_rows.append(calibration_table(y, score, name, split_name))
        else:
            row.update({k:None for k in ['Brier_Score','Log_Loss','Mean_Predicted_Probability','Actual_Event_Rate','Calibration_Intercept','Calibration_Slope','ECE_5_Bins']})
        metric_rows.append(row)

        if name in operational_models:
            bm = top_budget_metrics(y, score, 0.10)
            budget_rows.append({k:v for k,v in {'Model':name,'Split':split_name,**bm}.items() if k!='Alert_Flag'})
            lm = fixed_threshold_metrics(y, score, locked_threshold)
            locked_rows.append({k:v for k,v in {'Model':name,'Split':split_name,**lm}.items() if k!='Alert_Flag'})

metrics = pd.DataFrame(metric_rows)
budgets = pd.DataFrame(budget_rows)
locked = pd.DataFrame(locked_rows)
calibration = pd.concat(calibration_rows, ignore_index=True)
tuning = pd.DataFrame(tuning_rows).sort_values(['Model','Validation_PR_AUC','Validation_Brier'], ascending=[True,False,True])

# Select the classifier on validation PR-AUC only. The test period remains untouched.
classifier_names = list(models.keys())
validation_classifier_metrics = metrics[(metrics['Split']=='Validation') & metrics['Model'].isin(classifier_names)]
selected_model = validation_classifier_metrics.sort_values(['PR_AUC','Brier_Score'], ascending=[False,True]).iloc[0]['Model']

# Feature explanations from the fitted training-only models.
log_coef = pd.DataFrame({
    'Feature': feature_cols,
    'Standardized_Coefficient': log_model.named_steps['model'].coef_[0]
})
log_coef['Absolute_Coefficient'] = log_coef['Standardized_Coefficient'].abs()
log_coef['Direction'] = np.where(log_coef['Standardized_Coefficient'] >= 0, 'Higher predicted risk', 'Lower predicted risk')
log_coef = log_coef.sort_values('Absolute_Coefficient', ascending=False)

tree_importance = pd.DataFrame({'Feature':feature_cols, 'Importance':tree_model.feature_importances_}).sort_values('Importance',ascending=False)
rf_importance = pd.DataFrame({'Feature':feature_cols, 'Importance':rf_model.feature_importances_}).sort_values('Importance',ascending=False)
tree_rules = export_text(tree_model, feature_names=feature_cols, decimals=4)

# Row-level prediction output with exact 10% ranking alerts and locked-threshold alerts.
pred = df[['Prediction_Date','Target_Date','Chronological_Split','Target_Abs_Dislocation_Pct',target]].copy()
pred['Historical_Prevalence_Probability'] = prevalence
pred['Current_Gap_Rule_Score'] = df['X_Current_Abs_Gap_to_Threshold_Ratio']
for name, model in models.items():
    key = name.replace(' ', '_')
    pred[f'{key}_Probability'] = model.predict_proba(df[feature_cols])[:,1]

for name in operational_models:
    score_col = {
        'Historical Prevalence':'Historical_Prevalence_Probability',
        'Current Gap Rule':'Current_Gap_Rule_Score',
        'Logistic Regression':'Logistic_Regression_Probability',
        'Decision Tree':'Decision_Tree_Probability',
        'Random Forest':'Random_Forest_Probability',
    }[name]
    valid_threshold = validation_budget_threshold(pred.loc[pred['Chronological_Split']=='Validation',score_col], 0.10)
    pred[f'{name.replace(" ","_")}_Locked_Alert'] = (pred[score_col] >= valid_threshold).astype(int)
    pred[f'{name.replace(" ","_")}_Top10Pct_Alert'] = 0
    for split_name in ['Validation','Test']:
        idx = pred.index[pred['Chronological_Split'].eq(split_name)]
        k = max(1, math.ceil(len(idx)*0.10))
        top_idx = pred.loc[idx].sort_values(score_col, ascending=False, kind='stable').head(k).index
        pred.loc[top_idx, f'{name.replace(" ","_")}_Top10Pct_Alert'] = 1

for c in ['Prediction_Date','Target_Date']:
    pred[c] = pd.to_datetime(pred[c]).dt.strftime('%Y-%m-%d')
pred.to_csv(PREDICTIONS, index=False, float_format='%.8f')

summary = {
    'selected_model': selected_model,
    'selection_rule': 'Highest validation PR-AUC; Brier score breaks ties',
    'train_rows': len(train), 'validation_rows':len(valid), 'test_rows':len(test),
    'train_events':int(y_train.sum()), 'validation_events':int(y_valid.sum()), 'test_events':int(y_test.sum()),
    'train_prevalence':prevalence,
    'best_parameters': {
        'Logistic Regression': best_log['Parameters'],
        'Decision Tree': best_tree['Parameters'],
        'Random Forest': best_rf['Parameters'],
    },
    'metrics': metrics.where(pd.notna(metrics), None).to_dict(orient='records'),
    'budget_metrics': budgets.to_dict(orient='records'),
    'locked_threshold_metrics': locked.to_dict(orient='records'),
    'calibration_bins': calibration.to_dict(orient='records'),
    'tuning': tuning.to_dict(orient='records'),
    'logistic_coefficients': log_coef.to_dict(orient='records'),
    'decision_tree_importance': tree_importance.to_dict(orient='records'),
    'random_forest_importance': rf_importance.to_dict(orient='records'),
    'decision_tree_rules': tree_rules,
    'feature_columns': feature_cols,
    'predictions_csv': str(PREDICTIONS.relative_to(ROOT)),
}
RESULTS_JSON.parent.mkdir(parents=True, exist_ok=True)
RESULTS_JSON.write_text(json.dumps(summary, indent=2, default=str))

print(json.dumps({
    'selected_model':selected_model,
    'best_parameters':summary['best_parameters'],
    'validation_metrics':metrics[metrics.Split.eq('Validation')][['Model','PR_AUC','Brier_Score','ECE_5_Bins']].where(pd.notna(metrics),None).to_dict(orient='records'),
    'test_metrics':metrics[metrics.Split.eq('Test')][['Model','PR_AUC','Brier_Score','ECE_5_Bins']].where(pd.notna(metrics),None).to_dict(orient='records'),
    'test_budget':budgets[budgets.Split.eq('Test')].to_dict(orient='records'),
}, indent=2))


{
  "selected_model": "Random Forest",
  "best_parameters": {
    "Logistic Regression": "C=0.1",
    "Decision Tree": "depth=5; leaf=25; weight=None",
    "Random Forest": "depth=5; leaf=5; max_features=0.5; weight=balanced_subsample"
  },
  "validation_metrics": [
    {
      "Model": "Historical Prevalence",
      "PR_AUC": 0.02032520325203252,
      "Brier_Score": 0.021224450180292523,
      "ECE_5_Bins": 0.036226520885898526
    },
    {
      "Model": "Current Gap Rule",
      "PR_AUC": 0.11593063408156237,
      "Brier_Score": NaN,
      "ECE_5_Bins": NaN
    },
    {
      "Model": "Logistic Regression",
      "PR_AUC": 0.13381120495270118,
      "Brier_Score": 0.028286921016903662,
      "ECE_5_Bins": 0.09242059719655717
    },
    {
      "Model": "Decision Tree",
      "PR_AUC": 0.1341540507551329,
      "Brier_Score": 0.01911085985217658,
      "ECE_5_Bins": 0.019374345576194302
    },
    {
      "Model": "Random Forest",
      "PR_AUC": 0.276138147566719,
      "Brier_Sco

## Step 4 — Run purged walk-forward validation

Original repository script: `scripts/07_walk_forward_validation.py`


In [10]:
"""Purged expanding-window validation for the enhanced final study.

The fold definitions and reporting rules are frozen in docs/ANALYSIS_PLAN_V2.md.
Models are fitted on each fold's training period, hyperparameters are selected
using validation PR-AUC only, and the subsequent calendar period is evaluated
once. No test outcome enters tuning or feature selection.
"""
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore")
ROOT = Path(__file__).resolve().parents[1]
DATA = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

INPUT = DATA / "ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv"
RESULTS_JSON = OUT / "walk_forward_results.json"
METRICS_CSV = OUT / "ETF_Walk_Forward_Metrics.csv"
PREDICTIONS_CSV = OUT / "ETF_Walk_Forward_Predictions.csv"
TUNING_CSV = OUT / "ETF_Walk_Forward_Tuning.csv"
COEFFICIENTS_CSV = OUT / "ETF_Logistic_Standardized_Coefficients.csv"

df = pd.read_csv(INPUT, parse_dates=["Prediction_Date", "Target_Date"])
features = [c for c in df.columns if c.startswith("X_")]
target = "Y_Abnormal_Dislocation_95Q"

folds = [
    {"Fold": "Fold 1", "Train_End": "2022-12-31", "Validation_Year": 2023, "Test_Year": 2024},
    {"Fold": "Fold 2", "Train_End": "2023-12-31", "Validation_Year": 2024, "Test_Year": 2025},
    {"Fold": "Fold 3", "Train_End": "2024-12-31", "Validation_Year": 2025, "Test_Year": 2026},
]


def safe_auc(y, score):
    return float(roc_auc_score(y, score)) if len(np.unique(y)) > 1 else None


def top_budget(y, score, budget=0.10):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    k = max(1, int(math.ceil(len(y) * budget)))
    order = np.argsort(-score, kind="stable")
    alert = np.zeros(len(y), dtype=int)
    alert[order[:k]] = 1
    tp = int(((alert == 1) & (y == 1)).sum())
    precision = tp / k
    prevalence = y.mean()
    return {
        "Budget_Pct": budget * 100,
        "Alerts": k,
        "True_Positives": tp,
        "Precision": float(precision),
        "Recall": float(tp / max(1, y.sum())),
        "Lift": float(precision / prevalence) if prevalence else None,
        "Number_Needed_to_Review": float(k / tp) if tp else None,
        "Alert_Flag": alert,
    }


def no_skill_budget(y, budget=0.10):
    """Expected performance from random review at the same capacity.

    A constant prevalence score contains no ranking information, so choosing
    arbitrary tied rows would make its realised precision depend on row order.
    """
    y = np.asarray(y, dtype=int)
    k = max(1, int(math.ceil(len(y) * budget)))
    prevalence = float(y.mean())
    return {
        "Budget_Pct": budget * 100,
        "Alerts": k,
        "True_Positives": float(k * prevalence),
        "Precision": prevalence,
        "Recall": float(k / len(y)),
        "Lift": 1.0,
        "Number_Needed_to_Review": float(1 / prevalence) if prevalence else None,
        "Alert_Flag": np.zeros(len(y), dtype=int),
    }


def purge_prior(prior, following):
    """Remove any prior sample whose label reaches the next split's feature date."""
    if prior.empty or following.empty:
        return prior.copy(), 0
    first_next_prediction = following["Prediction_Date"].min()
    keep = prior["Target_Date"] < first_next_prediction
    return prior.loc[keep].copy(), int((~keep).sum())


def metric_row(fold, model, split, y, score, probability=False):
    row = {
        "Fold": fold,
        "Model": model,
        "Split": split,
        "Observations": int(len(y)),
        "Events": int(np.asarray(y).sum()),
        "Event_Rate": float(np.asarray(y).mean()),
        "PR_AUC": float(average_precision_score(y, score)),
        "ROC_AUC": safe_auc(y, score),
        "Brier_Score": None,
        "Log_Loss": None,
    }
    if probability:
        p = np.clip(np.asarray(score, dtype=float), 1e-6, 1 - 1e-6)
        row["Brier_Score"] = float(brier_score_loss(y, p))
        row["Log_Loss"] = float(log_loss(y, p, labels=[0, 1]))
    return row


metric_rows, tuning_rows, coefficient_rows, prediction_parts, fold_audit = [], [], [], [], []

for spec in folds:
    fold = spec["Fold"]
    train = df[df["Target_Date"] <= pd.Timestamp(spec["Train_End"])].copy()
    valid = df[df["Target_Date"].dt.year.eq(spec["Validation_Year"])].copy()
    test = df[df["Target_Date"].dt.year.eq(spec["Test_Year"])].copy()

    train, purge_train = purge_prior(train, valid)
    valid, purge_valid = purge_prior(valid, test)

    counts = {
        "Fold": fold,
        "Train_Target_End": spec["Train_End"],
        "Validation_Year": spec["Validation_Year"],
        "Test_Year": spec["Test_Year"],
        "Train_Rows": len(train), "Train_Events": int(train[target].sum()),
        "Validation_Rows": len(valid), "Validation_Events": int(valid[target].sum()),
        "Test_Rows": len(test), "Test_Events": int(test[target].sum()),
        "Purged_Train_Boundary_Rows": purge_train,
        "Purged_Validation_Boundary_Rows": purge_valid,
        "Feasible": True,
        "Reason": "",
    }
    if any(part[target].nunique() < 2 for part in [train, valid, test]):
        counts["Feasible"] = False
        counts["Reason"] = "At least one partition contains only one outcome class"
        fold_audit.append(counts)
        continue
    fold_audit.append(counts)

    Xtr, ytr = train[features], train[target].astype(int)
    Xva, yva = valid[features], valid[target].astype(int)
    Xte, yte = test[features], test[target].astype(int)

    best = {}

    # Logistic regression.
    candidates = []
    for c in [0.01, 0.1, 1.0, 10.0]:
        model = Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(C=c, penalty="l2", solver="liblinear", max_iter=5000, random_state=42)),
        ])
        model.fit(Xtr, ytr)
        p = model.predict_proba(Xva)[:, 1]
        row = {"Fold": fold, "Model": "Logistic Regression", "Parameters": f"C={c}",
               "Validation_PR_AUC": average_precision_score(yva, p), "Validation_Brier": brier_score_loss(yva, p)}
        tuning_rows.append(row); candidates.append((row, model))
    best_row, log_model = max(candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"]))
    best["Logistic Regression"] = best_row["Parameters"]

    # Coefficients are on standardized training features because the selected
    # logistic model is a StandardScaler -> LogisticRegression pipeline.
    # Saving every fold makes the paper's explanation table reproducible.
    fold_coefficients = log_model.named_steps["model"].coef_[0]
    coefficient_order = np.argsort(-np.abs(fold_coefficients))
    for rank, feature_index in enumerate(coefficient_order, start=1):
        coefficient_rows.append({
            "Fold": fold,
            "Test_Year": spec["Test_Year"],
            "Selected_C": float(best_row["Parameters"].split("=")[1]),
            "Feature": features[feature_index],
            "Standardized_Coefficient": float(fold_coefficients[feature_index]),
            "Absolute_Coefficient": float(abs(fold_coefficients[feature_index])),
            "Absolute_Rank_within_Fold": rank,
        })

    # Bounded decision tree.
    candidates = []
    for depth in [2, 3, 4, 5]:
        for leaf in [10, 25, 50]:
            for weight in [None, "balanced"]:
                model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, class_weight=weight, random_state=42)
                model.fit(Xtr, ytr)
                p = model.predict_proba(Xva)[:, 1]
                row = {"Fold": fold, "Model": "Decision Tree",
                       "Parameters": f"depth={depth}; leaf={leaf}; weight={weight}",
                       "Validation_PR_AUC": average_precision_score(yva, p), "Validation_Brier": brier_score_loss(yva, p)}
                tuning_rows.append(row); candidates.append((row, model))
    best_row, tree_model = max(candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"]))
    best["Decision Tree"] = best_row["Parameters"]

    # Random forest.
    candidates = []
    for depth in [3, 5, None]:
        for leaf in [5, 10, 20]:
            for max_features in ["sqrt", 0.5]:
                for weight in [None, "balanced_subsample"]:
                    model = RandomForestClassifier(
                        n_estimators=400, max_depth=depth, min_samples_leaf=leaf,
                        max_features=max_features, class_weight=weight,
                        random_state=42, n_jobs=-1,
                    )
                    model.fit(Xtr, ytr)
                    p = model.predict_proba(Xva)[:, 1]
                    row = {"Fold": fold, "Model": "Random Forest",
                           "Parameters": f"depth={depth}; leaf={leaf}; max_features={max_features}; weight={weight}",
                           "Validation_PR_AUC": average_precision_score(yva, p), "Validation_Brier": brier_score_loss(yva, p)}
                    tuning_rows.append(row); candidates.append((row, model))
    best_row, rf_model = max(candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"]))
    best["Random Forest"] = best_row["Parameters"]

    models = {
        "Logistic Regression": log_model,
        "Decision Tree": tree_model,
        "Random Forest": rf_model,
    }
    train_prevalence = float(ytr.mean())

    for split_name, part, y in [("Validation", valid, yva), ("Test", test, yte)]:
        scores = {
            "Historical Prevalence": np.repeat(train_prevalence, len(part)),
            "Current Absolute Gap": part["X_Abs_Dislocation_Pct_t"].to_numpy(float),
            "Current Gap Ratio": part["X_Current_Abs_Gap_to_Threshold_Ratio"].to_numpy(float),
            "Previous Event Persistence": (part["X_Abs_Dislocation_Pct_t"] > part["Rolling_Threshold_95Pct"]).astype(float).to_numpy(),
        }
        for name, model in models.items():
            scores[name] = model.predict_proba(part[features])[:, 1]

        for name, score in scores.items():
            probability = name in {"Historical Prevalence", "Logistic Regression", "Decision Tree", "Random Forest"}
            row = metric_row(fold, name, split_name, y, score, probability)
            budget = no_skill_budget(y, 0.10) if name == "Historical Prevalence" else top_budget(y, score, 0.10)
            row.update({k: v for k, v in budget.items() if k != "Alert_Flag"})
            metric_rows.append(row)

        if split_name == "Test":
            p = part[["Prediction_Date", "Target_Date", target, "Target_Abs_Dislocation_Pct"]].copy()
            p.insert(0, "Fold", fold)
            for name, score in scores.items():
                key = name.replace(" ", "_")
                p[f"{key}_Score"] = score
                budget_result = no_skill_budget(y, 0.10) if name == "Historical Prevalence" else top_budget(y, score, 0.10)
                p[f"{key}_Top10Pct_Alert"] = budget_result["Alert_Flag"]
            prediction_parts.append(p)

    # Add selected classifier to fold audit using validation results.
    fold_val = [r for r in metric_rows if r["Fold"] == fold and r["Split"] == "Validation" and r["Model"] in models]
    selected = sorted(fold_val, key=lambda r: (r["PR_AUC"], -(r["Brier_Score"] or 999)), reverse=True)[0]["Model"]
    fold_audit[-1]["Selected_Classifier"] = selected
    fold_audit[-1]["Best_Parameters"] = best


metrics = pd.DataFrame(metric_rows)
tuning = pd.DataFrame(tuning_rows)
predictions = pd.concat(prediction_parts, ignore_index=True) if prediction_parts else pd.DataFrame()
coefficients = pd.DataFrame(coefficient_rows)

test_metrics = metrics[metrics["Split"].eq("Test")].copy()
summary = test_metrics.groupby("Model", as_index=False).agg(
    Folds=("Fold", "nunique"),
    Mean_PR_AUC=("PR_AUC", "mean"),
    SD_PR_AUC=("PR_AUC", "std"),
    Minimum_PR_AUC=("PR_AUC", "min"),
    Maximum_PR_AUC=("PR_AUC", "max"),
    Mean_Precision_at_10Pct=("Precision", "mean"),
    Mean_Recall_at_10Pct=("Recall", "mean"),
    Mean_Lift_at_10Pct=("Lift", "mean"),
)

if not test_metrics.empty:
    winners = test_metrics.loc[test_metrics.groupby("Fold")["PR_AUC"].idxmax(), ["Fold", "Model"]]
    win_counts = winners["Model"].value_counts()
    summary["Fold_Wins"] = summary["Model"].map(win_counts).fillna(0).astype(int)

for frame in [metrics, tuning, predictions, coefficients]:
    for col in ["Prediction_Date", "Target_Date"]:
        if col in frame:
            frame[col] = pd.to_datetime(frame[col]).dt.strftime("%Y-%m-%d")

metrics.to_csv(METRICS_CSV, index=False, float_format="%.17g")
tuning.to_csv(TUNING_CSV, index=False, float_format="%.17g")
# Preserve near-tied tree probabilities so recomputed ranking metrics are exact.
predictions.to_csv(PREDICTIONS_CSV, index=False, float_format="%.17g")
coefficients.to_csv(COEFFICIENTS_CSV, index=False, float_format="%.17g")

metrics_clean = metrics.replace([np.inf, -np.inf], np.nan)
summary_clean = summary.replace([np.inf, -np.inf], np.nan)
result = {
    "analysis": "Purged expanding-window walk-forward evaluation",
    "fold_audit": fold_audit,
    "metrics": metrics_clean.astype(object).where(pd.notna(metrics_clean), None).to_dict("records"),
    "test_stability_summary": summary_clean.astype(object).where(pd.notna(summary_clean), None).to_dict("records"),
    "tuning_csv": str(TUNING_CSV.relative_to(ROOT)),
    "predictions_csv": str(PREDICTIONS_CSV.relative_to(ROOT)),
    "standardized_logistic_coefficients_csv": str(COEFFICIENTS_CSV.relative_to(ROOT)),
}
RESULTS_JSON.write_text(json.dumps(result, indent=2, default=str), encoding="utf-8")

# Figure 1: test PR-AUC by fold.
order = ["Historical Prevalence", "Current Absolute Gap", "Current Gap Ratio",
         "Previous Event Persistence", "Logistic Regression", "Decision Tree", "Random Forest"]
plot = test_metrics.pivot(index="Fold", columns="Model", values="PR_AUC").reindex(columns=order)
ax = plot.plot(kind="bar", figsize=(11, 5.4), width=0.82, color=["#A5A5A5", "#5B9BD5", "#1F4E78", "#9EADBA", "#4472C4", "#70AD47", "#ED7D31"])
ax.set_title("Purged walk-forward test PR-AUC", pad=58)
ax.set_ylabel("PR-AUC"); ax.set_xlabel(""); ax.grid(axis="y", alpha=.25)
ax.legend(frameon=False, ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 1.16))
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(FIG / "ETF_Walk_Forward_PR_AUC.png", dpi=220, bbox_inches="tight"); plt.close()

# Figure 2: mean and range across test folds.
s = summary.sort_values("Mean_PR_AUC", ascending=True)
lower = s["Mean_PR_AUC"] - s["Minimum_PR_AUC"]
upper = s["Maximum_PR_AUC"] - s["Mean_PR_AUC"]
fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.errorbar(s["Mean_PR_AUC"], s["Model"], xerr=[lower, upper], fmt="o", color="#1F4E78", ecolor="#7F8C8D", capsize=4)
ax.set_title("Walk-forward PR-AUC stability: mean and observed range")
ax.set_xlabel("Test PR-AUC"); ax.grid(axis="x", alpha=.25)
fig.tight_layout(); fig.savefig(FIG / "ETF_Walk_Forward_Stability.png", dpi=220, bbox_inches="tight"); plt.close(fig)

print(json.dumps({"fold_audit": fold_audit, "test_stability_summary": result["test_stability_summary"]}, indent=2, default=str))


Completed purged expanding-window validation for 3 folds.
Validation events: 2023=2, 2024=26, 2025=5.
Mean test PR-AUC: gap ratio=0.233, logistic=0.169, tree=0.156, random forest=0.184.
Saved outputs/ETF_Logistic_Standardized_Coefficients.csv (75 rows; 25 features × 3 folds).


## Step 5 — Estimate dependent-data uncertainty with a moving-block bootstrap

Original repository script: `scripts/08_block_bootstrap_uncertainty.py`


In [12]:
"""Paired moving-block bootstrap uncertainty for walk-forward test predictions.

The procedure follows docs/ANALYSIS_PLAN_V2.md: 2,000 valid resamples,
five-observation primary blocks, ten-observation sensitivity blocks, seed 42.
Each fold is resampled independently, while the same indices are applied to
every model so model differences remain paired.
"""
from __future__ import annotations

from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score


ROOT = PROJECT_ROOT
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

INPUT = OUT / "ETF_Walk_Forward_Predictions.csv"
INTERVALS_CSV = OUT / "ETF_Block_Bootstrap_Intervals.csv"
DIFFERENCES_CSV = OUT / "ETF_Paired_PR_AUC_Differences.csv"
RESULTS_JSON = OUT / "block_bootstrap_results.json"

df = pd.read_csv(INPUT, parse_dates=["Prediction_Date", "Target_Date"])
target = "Y_Abnormal_Dislocation_95Q"
folds = list(df["Fold"].drop_duplicates())
models = {
    "Historical Prevalence": "Historical_Prevalence_Score",
    "Current Absolute Gap": "Current_Absolute_Gap_Score",
    "Current Gap Ratio": "Current_Gap_Ratio_Score",
    "Previous Event Persistence": "Previous_Event_Persistence_Score",
    "Logistic Regression": "Logistic_Regression_Score",
    "Decision Tree": "Decision_Tree_Score",
    "Random Forest": "Random_Forest_Score",
}
comparators = ["Logistic Regression", "Decision Tree", "Random Forest"]
resamples = 2000


def circular_block_indices(n, block_length, rng):
    blocks = int(math.ceil(n / block_length))
    starts = rng.integers(0, n, size=blocks)
    return np.concatenate([(start + np.arange(block_length)) % n for start in starts])[:n]


def ranked_budget(y, score, budget=0.10, no_skill=False):
    y = np.asarray(y, dtype=int)
    k = max(1, int(math.ceil(len(y) * budget)))
    if no_skill:
        prevalence = float(y.mean())
        return prevalence, float(k / len(y))
    order = np.argsort(-np.asarray(score, dtype=float), kind="stable")
    chosen = y[order[:k]]
    return float(chosen.mean()), float(chosen.sum() / y.sum())


def point_metrics(part, model, score_column):
    y = part[target].to_numpy(int)
    score = part[score_column].to_numpy(float)
    precision, recall = ranked_budget(y, score, 0.10, no_skill=model == "Historical Prevalence")
    return {
        "PR_AUC": float(average_precision_score(y, score)),
        "Precision_at_10Pct": precision,
        "Recall_at_10Pct": recall,
    }


interval_rows = []
difference_rows = []
json_by_block = {}

for block_length in [5, 10]:
    rng = np.random.default_rng(42 + block_length)
    observed_fold = {
        fold: {model: point_metrics(df[df["Fold"].eq(fold)], model, column) for model, column in models.items()}
        for fold in folds
    }
    observed_mean = {
        model: {metric: float(np.mean([observed_fold[fold][model][metric] for fold in folds]))
                for metric in ["PR_AUC", "Precision_at_10Pct", "Recall_at_10Pct"]}
        for model in models
    }

    fold_dist = {
        fold: {model: {metric: [] for metric in ["PR_AUC", "Precision_at_10Pct", "Recall_at_10Pct"]}
               for model in models}
        for fold in folds
    }
    mean_dist = {model: {metric: [] for metric in ["PR_AUC", "Precision_at_10Pct", "Recall_at_10Pct"]}
                 for model in models}

    accepted = 0
    attempts = 0
    while accepted < resamples and attempts < resamples * 20:
        attempts += 1
        sampled = {}
        valid = True
        for fold in folds:
            part = df[df["Fold"].eq(fold)].reset_index(drop=True)
            idx = circular_block_indices(len(part), block_length, rng)
            boot = part.iloc[idx].reset_index(drop=True)
            if boot[target].nunique() < 2:
                valid = False
                break
            sampled[fold] = boot
        if not valid:
            continue

        for fold in folds:
            for model, column in models.items():
                values = point_metrics(sampled[fold], model, column)
                for metric, value in values.items():
                    fold_dist[fold][model][metric].append(value)

        for model in models:
            for metric in ["PR_AUC", "Precision_at_10Pct", "Recall_at_10Pct"]:
                mean_dist[model][metric].append(float(np.mean([
                    fold_dist[fold][model][metric][-1] for fold in folds
                ])))
        accepted += 1

    if accepted < resamples:
        raise RuntimeError(f"Only {accepted} valid resamples for block length {block_length}")

    for fold in folds:
        for model in models:
            for metric in ["PR_AUC", "Precision_at_10Pct", "Recall_at_10Pct"]:
                values = np.asarray(fold_dist[fold][model][metric])
                interval_rows.append({
                    "Block_Length": block_length,
                    "Scope": "Individual fold",
                    "Fold": fold,
                    "Model": model,
                    "Metric": metric,
                    "Observed": observed_fold[fold][model][metric],
                    "CI_Lower_95Pct": float(np.percentile(values, 2.5)),
                    "CI_Upper_95Pct": float(np.percentile(values, 97.5)),
                    "Valid_Resamples": accepted,
                })

    for model in models:
        for metric in ["PR_AUC", "Precision_at_10Pct", "Recall_at_10Pct"]:
            values = np.asarray(mean_dist[model][metric])
            interval_rows.append({
                "Block_Length": block_length,
                "Scope": "Mean across folds",
                "Fold": "All folds",
                "Model": model,
                "Metric": metric,
                "Observed": observed_mean[model][metric],
                "CI_Lower_95Pct": float(np.percentile(values, 2.5)),
                "CI_Upper_95Pct": float(np.percentile(values, 97.5)),
                "Valid_Resamples": accepted,
            })

    for comparator in comparators:
        gap = np.asarray(mean_dist["Current Gap Ratio"]["PR_AUC"])
        other = np.asarray(mean_dist[comparator]["PR_AUC"])
        diff = gap - other
        observed_diff = observed_mean["Current Gap Ratio"]["PR_AUC"] - observed_mean[comparator]["PR_AUC"]
        difference_rows.append({
            "Block_Length": block_length,
            "Reference_Model": "Current Gap Ratio",
            "Comparator": comparator,
            "Observed_PR_AUC_Difference": observed_diff,
            "CI_Lower_95Pct": float(np.percentile(diff, 2.5)),
            "CI_Upper_95Pct": float(np.percentile(diff, 97.5)),
            "Bootstrap_Share_Difference_GT_0": float(np.mean(diff > 0)),
            "Valid_Resamples": accepted,
        })

    json_by_block[str(block_length)] = {
        "accepted_resamples": accepted,
        "attempts": attempts,
        "observed_mean": observed_mean,
    }

intervals = pd.DataFrame(interval_rows)
differences = pd.DataFrame(difference_rows)
intervals.to_csv(INTERVALS_CSV, index=False, float_format="%.8f")
differences.to_csv(DIFFERENCES_CSV, index=False, float_format="%.8f")

result = {
    "method": "Paired circular moving-block bootstrap within each walk-forward test fold",
    "resamples": resamples,
    "random_seed_rule": "42 + block length",
    "block_lengths": [5, 10],
    "blocks": json_by_block,
    "model_intervals": intervals.to_dict("records"),
    "paired_pr_auc_differences": differences.to_dict("records"),
}
RESULTS_JSON.write_text(json.dumps(result, indent=2), encoding="utf-8")

# Primary five-day-block figures.
plot = intervals[(intervals["Block_Length"].eq(5)) & (intervals["Scope"].eq("Mean across folds")) & (intervals["Metric"].eq("PR_AUC"))].copy()
order = ["Current Gap Ratio", "Current Absolute Gap", "Random Forest", "Logistic Regression", "Decision Tree", "Previous Event Persistence", "Historical Prevalence"]
plot["Order"] = plot["Model"].map({name: i for i, name in enumerate(order)})
plot = plot.sort_values("Order", ascending=False)
lower = plot["Observed"] - plot["CI_Lower_95Pct"]
upper = plot["CI_Upper_95Pct"] - plot["Observed"]
fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.errorbar(plot["Observed"], plot["Model"], xerr=[lower, upper], fmt="o", color="#1F4E78", ecolor="#7F8C8D", capsize=4)
ax.set_title("Mean walk-forward PR-AUC with 95% block-bootstrap intervals")
ax.set_xlabel("Mean test PR-AUC"); ax.grid(axis="x", alpha=.25)
fig.tight_layout(); fig.savefig(FIG / "ETF_Bootstrap_PR_AUC_Intervals.png", dpi=220, bbox_inches="tight"); plt.close(fig)

diff_plot = differences[differences["Block_Length"].eq(5)].copy().sort_values("Observed_PR_AUC_Difference")
lower = diff_plot["Observed_PR_AUC_Difference"] - diff_plot["CI_Lower_95Pct"]
upper = diff_plot["CI_Upper_95Pct"] - diff_plot["Observed_PR_AUC_Difference"]
fig, ax = plt.subplots(figsize=(8.2, 3.8))
ax.errorbar(diff_plot["Observed_PR_AUC_Difference"], diff_plot["Comparator"], xerr=[lower, upper], fmt="o", color="#1F4E78", ecolor="#7F8C8D", capsize=4)
ax.axvline(0, color="#C00000", linestyle="--", linewidth=1)
ax.set_title("Paired PR-AUC difference: gap ratio minus classifier")
ax.set_xlabel("Mean PR-AUC difference with 95% interval"); ax.grid(axis="x", alpha=.25)
fig.tight_layout(); fig.savefig(FIG / "ETF_Bootstrap_Paired_Differences.png", dpi=220, bbox_inches="tight"); plt.close(fig)

primary_intervals = intervals[(intervals.Block_Length.eq(5)) & (intervals.Scope.eq("Mean across folds"))]
print(json.dumps({
    "primary_mean_intervals": primary_intervals.to_dict("records"),
    "paired_differences": differences[differences.Block_Length.eq(5)].to_dict("records"),
    "accepted_resamples": json_by_block["5"]["accepted_resamples"],
}, indent=2))


{
  "primary_mean_intervals": [
    {
      "Block_Length": 5,
      "Scope": "Mean across folds",
      "Fold": "All folds",
      "Model": "Historical Prevalence",
      "Metric": "PR_AUC",
      "Observed": 0.09554724148461845,
      "CI_Lower_95Pct": 0.07021213806213909,
      "CI_Upper_95Pct": 0.12219312454381977,
      "Valid_Resamples": 2000
    },
    {
      "Block_Length": 5,
      "Scope": "Mean across folds",
      "Fold": "All folds",
      "Model": "Historical Prevalence",
      "Metric": "Precision_at_10Pct",
      "Observed": 0.09554724148461845,
      "CI_Lower_95Pct": 0.07021213806213909,
      "CI_Upper_95Pct": 0.12219312454381977,
      "Valid_Resamples": 2000
    },
    {
      "Block_Length": 5,
      "Scope": "Mean across folds",
      "Fold": "All folds",
      "Model": "Historical Prevalence",
      "Metric": "Recall_at_10Pct",
      "Observed": 0.10257851323396218,
      "CI_Lower_95Pct": 0.10257851323396218,
      "CI_Upper_95Pct": 0.10257851323396218,
      

## Step 6A — Test alternative event-label definitions

Original repository script: `scripts/09_label_sensitivity.py`


In [14]:
"""Prespecified abnormal-event label sensitivity analysis.

Re-runs the purged walk-forward pipeline for rolling 90th, 95th and 97.5th
percentile labels, and audits feasibility for fixed 0.50% and 1.00% labels.
Every rolling threshold uses the 252 complete observations ending on prediction
date t, so the target date never contributes to its own label threshold.
"""
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore")
ROOT = PROJECT_ROOT
DATA = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

MODEL_INPUT = DATA / "ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv"
MASTER_INPUT = DATA / "ETF_Master_Merged_2021_to_2026.csv"
METRICS_CSV = OUT / "ETF_Label_Sensitivity_Metrics.csv"
SUMMARY_CSV = OUT / "ETF_Label_Sensitivity_Summary.csv"
AUDIT_CSV = OUT / "ETF_Label_Sensitivity_Feasibility.csv"
TUNING_CSV = OUT / "ETF_Label_Sensitivity_Tuning.csv"
RESULTS_JSON = OUT / "label_sensitivity_results.json"

folds = [
    {"Fold": "Fold 1", "Train_End": "2022-12-31", "Validation_Year": 2023, "Test_Year": 2024},
    {"Fold": "Fold 2", "Train_End": "2023-12-31", "Validation_Year": 2024, "Test_Year": 2025},
    {"Fold": "Fold 3", "Train_End": "2024-12-31", "Validation_Year": 2025, "Test_Year": 2026},
]

scenarios = [
    {"Label_Key": "Rolling_90Q", "Label": "Rolling 90th percentile", "kind": "rolling", "value": 0.90},
    {"Label_Key": "Rolling_95Q", "Label": "Rolling 95th percentile", "kind": "rolling", "value": 0.95},
    {"Label_Key": "Rolling_97_5Q", "Label": "Rolling 97.5th percentile", "kind": "rolling", "value": 0.975},
    {"Label_Key": "Fixed_0_50", "Label": "Fixed 0.50%", "kind": "fixed", "value": 0.50},
    {"Label_Key": "Fixed_1_00", "Label": "Fixed 1.00%", "kind": "fixed", "value": 1.00},
]


def purge_prior(prior: pd.DataFrame, following: pd.DataFrame):
    if prior.empty or following.empty:
        return prior.copy(), 0
    first_next_prediction = following["Prediction_Date"].min()
    keep = prior["Target_Date"] < first_next_prediction
    return prior.loc[keep].copy(), int((~keep).sum())


def top_budget(y, score, budget=0.10):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    k = max(1, int(math.ceil(len(y) * budget)))
    order = np.argsort(-score, kind="stable")
    alert = np.zeros(len(y), dtype=int)
    alert[order[:k]] = 1
    tp = int(((alert == 1) & (y == 1)).sum())
    prevalence = float(y.mean())
    return {
        "Alerts": k,
        "True_Positives": tp,
        "Precision": float(tp / k),
        "Recall": float(tp / max(1, y.sum())),
        "Lift": float((tp / k) / prevalence) if prevalence else None,
    }


def no_skill_budget(y, budget=0.10):
    y = np.asarray(y, dtype=int)
    k = max(1, int(math.ceil(len(y) * budget)))
    prevalence = float(y.mean())
    return {
        "Alerts": k,
        "True_Positives": float(k * prevalence),
        "Precision": prevalence,
        "Recall": float(k / len(y)),
        "Lift": 1.0,
    }


def metric_row(label, fold, model, split, y, score, probability=False):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    row = {
        "Label_Definition": label,
        "Fold": fold,
        "Model": model,
        "Split": split,
        "Observations": int(len(y)),
        "Events": int(y.sum()),
        "Event_Rate": float(y.mean()),
        "PR_AUC": float(average_precision_score(y, score)),
        "ROC_AUC": float(roc_auc_score(y, score)) if len(np.unique(y)) > 1 else None,
        "Brier_Score": None,
    }
    if probability:
        row["Brier_Score"] = float(brier_score_loss(y, np.clip(score, 1e-6, 1 - 1e-6)))
    budget = no_skill_budget(y) if model == "Historical Prevalence" else top_budget(y, score)
    row.update(budget)
    return row


def fit_models(train, valid, features, target, label, fold, tuning_rows):
    Xtr, ytr = train[features], train[target].astype(int)
    Xva, yva = valid[features], valid[target].astype(int)
    chosen = {}

    candidates = []
    for c in [0.01, 0.1, 1.0, 10.0]:
        model = Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(C=c, penalty="l2", solver="liblinear", max_iter=5000, random_state=42)),
        ])
        model.fit(Xtr, ytr)
        p = model.predict_proba(Xva)[:, 1]
        row = {"Label_Definition": label, "Fold": fold, "Model": "Logistic Regression",
               "Parameters": f"C={c}", "Validation_PR_AUC": float(average_precision_score(yva, p)),
               "Validation_Brier": float(brier_score_loss(yva, p))}
        tuning_rows.append(row); candidates.append((row, model))
    best_row, chosen["Logistic Regression"] = max(
        candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"])
    )

    candidates = []
    for depth in [2, 3, 4, 5]:
        for leaf in [10, 25, 50]:
            for weight in [None, "balanced"]:
                model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf,
                                               class_weight=weight, random_state=42)
                model.fit(Xtr, ytr)
                p = model.predict_proba(Xva)[:, 1]
                row = {"Label_Definition": label, "Fold": fold, "Model": "Decision Tree",
                       "Parameters": f"depth={depth}; leaf={leaf}; weight={weight}",
                       "Validation_PR_AUC": float(average_precision_score(yva, p)),
                       "Validation_Brier": float(brier_score_loss(yva, p))}
                tuning_rows.append(row); candidates.append((row, model))
    best_row, chosen["Decision Tree"] = max(
        candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"])
    )

    candidates = []
    for depth in [3, 5, None]:
        for leaf in [5, 10, 20]:
            for max_features in ["sqrt", 0.5]:
                for weight in [None, "balanced_subsample"]:
                    model = RandomForestClassifier(
                        n_estimators=400, max_depth=depth, min_samples_leaf=leaf,
                        max_features=max_features, class_weight=weight,
                        random_state=42, n_jobs=-1,
                    )
                    model.fit(Xtr, ytr)
                    p = model.predict_proba(Xva)[:, 1]
                    row = {"Label_Definition": label, "Fold": fold, "Model": "Random Forest",
                           "Parameters": f"depth={depth}; leaf={leaf}; max_features={max_features}; weight={weight}",
                           "Validation_PR_AUC": float(average_precision_score(yva, p)),
                           "Validation_Brier": float(brier_score_loss(yva, p))}
                    tuning_rows.append(row); candidates.append((row, model))
    best_row, chosen["Random Forest"] = max(
        candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"])
    )
    return chosen


model_base = pd.read_csv(MODEL_INPUT, parse_dates=["Prediction_Date", "Target_Date"])
master = pd.read_csv(MASTER_INPUT, parse_dates=["Date"]).sort_values("Date")
complete = master.loc[master["Complete_Case"].eq(1), ["Date", "Abs_Dislocation_Pct"]].copy()
for q, col in [(0.90, "Threshold_90Q"), (0.95, "Threshold_95Q"), (0.975, "Threshold_97_5Q")]:
    complete[col] = complete["Abs_Dislocation_Pct"].rolling(252, min_periods=252).quantile(q)
thresholds = complete.drop(columns="Abs_Dislocation_Pct")
model_base = model_base.merge(thresholds, left_on="Prediction_Date", right_on="Date", how="left", validate="one_to_one")
assert np.allclose(model_base["Threshold_95Q"], model_base["Rolling_Threshold_95Pct"], atol=1e-8)

base_features = [c for c in model_base.columns if c.startswith("X_")]
metric_rows, tuning_rows, audit_rows, yearly_rows = [], [], [], []

for scenario in scenarios:
    label = scenario["Label"]
    data = model_base.copy()
    if scenario["kind"] == "rolling":
        threshold_col = {0.90: "Threshold_90Q", 0.95: "Threshold_95Q", 0.975: "Threshold_97_5Q"}[scenario["value"]]
        data["Scenario_Threshold"] = data[threshold_col]
    else:
        data["Scenario_Threshold"] = float(scenario["value"])
    data["Scenario_Target"] = (data["Target_Abs_Dislocation_Pct"] > data["Scenario_Threshold"]).astype(int)
    data["Scenario_Persistence"] = (data["X_Abs_Dislocation_Pct_t"] > data["Scenario_Threshold"]).astype(float)
    # Replace the label-dependent ratio feature so every scenario uses its own frozen threshold.
    data["X_Current_Abs_Gap_to_Threshold_Ratio"] = (
        data["X_Abs_Dislocation_Pct_t"] / data["Scenario_Threshold"]
    )

    for year, part in data.groupby(data["Target_Date"].dt.year):
        yearly_rows.append({"Label_Definition": label, "Target_Year": int(year),
                            "Observations": int(len(part)), "Events": int(part["Scenario_Target"].sum()),
                            "Event_Rate": float(part["Scenario_Target"].mean())})

    for spec in folds:
        fold = spec["Fold"]
        train = data[data["Target_Date"] <= pd.Timestamp(spec["Train_End"])].copy()
        valid = data[data["Target_Date"].dt.year.eq(spec["Validation_Year"])].copy()
        test = data[data["Target_Date"].dt.year.eq(spec["Test_Year"])].copy()
        train, purge_train = purge_prior(train, valid)
        valid, purge_valid = purge_prior(valid, test)
        split_counts = {
            "Train": (len(train), int(train["Scenario_Target"].sum())),
            "Validation": (len(valid), int(valid["Scenario_Target"].sum())),
            "Test": (len(test), int(test["Scenario_Target"].sum())),
        }
        bad = [name for name, (n, events) in split_counts.items() if events == 0 or events == n]
        feasible = not bad
        audit_rows.append({
            "Label_Definition": label, "Fold": fold,
            "Total_Events": int(data["Scenario_Target"].sum()),
            "Train_Rows": split_counts["Train"][0], "Train_Events": split_counts["Train"][1],
            "Validation_Rows": split_counts["Validation"][0], "Validation_Events": split_counts["Validation"][1],
            "Test_Rows": split_counts["Test"][0], "Test_Events": split_counts["Test"][1],
            "Purged_Train_Rows": purge_train, "Purged_Validation_Rows": purge_valid,
            "Feasible": feasible,
            "Reason": "" if feasible else f"No usable class variation in: {', '.join(bad)}",
        })
        if not feasible:
            continue

        models = fit_models(train, valid, base_features, "Scenario_Target", label, fold, tuning_rows)
        train_prevalence = float(train["Scenario_Target"].mean())
        for split_name, part in [("Validation", valid), ("Test", test)]:
            y = part["Scenario_Target"].astype(int).to_numpy()
            scores = {
                "Historical Prevalence": np.repeat(train_prevalence, len(part)),
                "Current Absolute Gap": part["X_Abs_Dislocation_Pct_t"].to_numpy(float),
                "Current Gap Ratio": part["X_Current_Abs_Gap_to_Threshold_Ratio"].to_numpy(float),
                "Previous Event Persistence": part["Scenario_Persistence"].to_numpy(float),
            }
            for name, model in models.items():
                scores[name] = model.predict_proba(part[base_features])[:, 1]
            for name, score in scores.items():
                metric_rows.append(metric_row(
                    label, fold, name, split_name, y, score,
                    probability=name in {"Historical Prevalence", "Logistic Regression", "Decision Tree", "Random Forest"},
                ))

metrics = pd.DataFrame(metric_rows)
audit = pd.DataFrame(audit_rows)
yearly = pd.DataFrame(yearly_rows)
tuning = pd.DataFrame(tuning_rows)

if not metrics.empty:
    test = metrics[metrics["Split"].eq("Test")].copy()
    summary = test.groupby(["Label_Definition", "Model"], as_index=False).agg(
        Feasible_Folds=("Fold", "nunique"),
        Mean_PR_AUC=("PR_AUC", "mean"),
        Minimum_PR_AUC=("PR_AUC", "min"),
        Maximum_PR_AUC=("PR_AUC", "max"),
        Mean_Precision_at_10Pct=("Precision", "mean"),
        Mean_Recall_at_10Pct=("Recall", "mean"),
        Mean_Lift_at_10Pct=("Lift", "mean"),
    )
    summary["Rank_within_Label"] = summary.groupby("Label_Definition")["Mean_PR_AUC"].rank(method="min", ascending=False).astype(int)
else:
    summary = pd.DataFrame()

metrics.to_csv(METRICS_CSV, index=False, float_format="%.17g")
summary.to_csv(SUMMARY_CSV, index=False, float_format="%.17g")
audit.to_csv(AUDIT_CSV, index=False)
tuning.to_csv(TUNING_CSV, index=False, float_format="%.17g")
yearly.to_csv(OUT / "ETF_Label_Sensitivity_Yearly_Events.csv", index=False, float_format="%.17g")

result = {
    "analysis": "Prespecified label-definition sensitivity using the purged walk-forward pipeline",
    "threshold_window": 252,
    "feasibility_rule": "Every train, validation and test partition must contain both outcome classes",
    "feature_rule": "The current-gap ratio and persistence benchmark use each scenario's threshold frozen on prediction date t",
    "feasibility": audit.replace({np.nan: None}).to_dict("records"),
    "yearly_event_counts": yearly.replace({np.nan: None}).to_dict("records"),
    "test_summary": summary.replace({np.nan: None}).to_dict("records"),
}
RESULTS_JSON.write_text(json.dumps(result, indent=2), encoding="utf-8")

# Event-count figure for all prespecified definitions.
event_totals = yearly.groupby("Label_Definition", as_index=False)["Events"].sum()
order_labels = [s["Label"] for s in scenarios]
event_totals["Label_Definition"] = pd.Categorical(event_totals["Label_Definition"], order_labels, ordered=True)
event_totals = event_totals.sort_values("Label_Definition")
fig, ax = plt.subplots(figsize=(9.2, 4.8))
bars = ax.bar(event_totals["Label_Definition"].astype(str), event_totals["Events"], color=["#5B9BD5", "#1F4E78", "#4472C4", "#A5A5A5", "#7F7F7F"])
ax.bar_label(bars, padding=3)
ax.set_title("Abnormal-event counts under prespecified label definitions")
ax.set_ylabel("Events in 1,185 modelling observations")
ax.tick_params(axis="x", rotation=18)
ax.grid(axis="y", alpha=.22)
fig.tight_layout(); fig.savefig(FIG / "ETF_Label_Sensitivity_Event_Counts.png", dpi=220, bbox_inches="tight"); plt.close(fig)

# Model ranking plot for feasible rolling labels.
plot_models = ["Current Gap Ratio", "Logistic Regression", "Decision Tree", "Random Forest"]
plot = summary[summary["Model"].isin(plot_models)].pivot(index="Label_Definition", columns="Model", values="Mean_PR_AUC")
plot = plot.reindex([s["Label"] for s in scenarios if s["kind"] == "rolling"])
ax = plot.plot(kind="bar", figsize=(10.2, 5.2), color=["#1F4E78", "#70AD47", "#4472C4", "#ED7D31"], width=.78)
ax.set_title("Mean walk-forward test PR-AUC across label definitions")
ax.set_ylabel("Mean test PR-AUC"); ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0); ax.grid(axis="y", alpha=.22)
ax.legend(frameon=False, ncol=2, loc="upper left")
fig = ax.get_figure(); fig.tight_layout(); fig.savefig(FIG / "ETF_Label_Sensitivity_PR_AUC.png", dpi=220, bbox_inches="tight"); plt.close(fig)

print(json.dumps({
    "event_totals": {str(r.Label_Definition): int(r.Events) for r in event_totals.itertuples()},
    "feasible_folds": audit.groupby("Label_Definition")["Feasible"].sum().astype(int).to_dict(),
    "top_models": summary.loc[summary.groupby("Label_Definition")["Mean_PR_AUC"].idxmax(),
                              ["Label_Definition", "Model", "Mean_PR_AUC"]].to_dict("records") if not summary.empty else [],
}, indent=2))


{
  "event_totals": {
    "Rolling 90th percentile": 138,
    "Rolling 95th percentile": 80,
    "Rolling 97.5th percentile": 41,
    "Fixed 0.50%": 12,
    "Fixed 1.00%": 2
  },
  "feasible_folds": {
    "Fixed 0.50%": 0,
    "Fixed 1.00%": 0,
    "Rolling 90th percentile": 3,
    "Rolling 95th percentile": 3,
    "Rolling 97.5th percentile": 3
  },
  "top_models": [
    {
      "Label_Definition": "Rolling 90th percentile",
      "Model": "Random Forest",
      "Mean_PR_AUC": 0.29854017547511763
    },
    {
      "Label_Definition": "Rolling 95th percentile",
      "Model": "Current Gap Ratio",
      "Mean_PR_AUC": 0.23326676949033884
    },
    {
      "Label_Definition": "Rolling 97.5th percentile",
      "Model": "Random Forest",
      "Mean_PR_AUC": 0.2864187382718785
    }
  ]
}
Completed in 359.8 seconds.


## Step 6B — Evaluate operational alert capacities

Original repository script: `scripts/10_alert_budget_sensitivity.py`


In [16]:
"""Operational alert-budget sensitivity for walk-forward test predictions.

Evaluates the highest-scored 5%, 10% and 20% of dates separately within each
test fold. When multiple observations tie at the capacity cutoff, the primary
metrics use the expected number of positives under random selection within the
cutoff tie. A deterministic stable-order count is retained only as an audit.
"""
from __future__ import annotations

from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


ROOT = PROJECT_ROOT
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
INPUT = OUT / "ETF_Walk_Forward_Predictions.csv"
FOLD_CSV = OUT / "ETF_Alert_Budget_Fold_Metrics.csv"
SUMMARY_CSV = OUT / "ETF_Alert_Budget_Summary.csv"
RESULTS_JSON = OUT / "alert_budget_sensitivity_results.json"

target = "Y_Abnormal_Dislocation_95Q"
budgets = [0.05, 0.10, 0.20]
model_order = [
    "Historical Prevalence", "Current Absolute Gap", "Current Gap Ratio",
    "Previous Event Persistence", "Logistic Regression", "Decision Tree", "Random Forest",
]


def display_name(score_col: str) -> str:
    return score_col.removesuffix("_Score").replace("_", " ")


def tie_adjusted_budget(y, score, budget):
    """Exact capacity with expected positives when the cutoff score is tied."""
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    n = len(y)
    k = max(1, int(math.ceil(n * budget)))
    order = np.argsort(-score, kind="stable")
    cutoff = float(score[order[k - 1]])
    above = score > cutoff
    tied = score == cutoff
    above_n = int(above.sum())
    tie_n = int(tied.sum())
    slots_from_tie = int(k - above_n)
    above_tp = int(y[above].sum())
    tie_events = int(y[tied].sum())
    expected_tie_tp = slots_from_tie * tie_events / tie_n
    expected_tp = float(above_tp + expected_tie_tp)
    exact_idx = order[:k]
    exact_tp = int(y[exact_idx].sum())
    prevalence = float(y.mean())
    precision = expected_tp / k
    recall = expected_tp / max(1, int(y.sum()))
    return {
        "Observations": n,
        "Events": int(y.sum()),
        "Event_Rate": prevalence,
        "Budget_Pct": budget * 100,
        "Alerts": k,
        "Expected_True_Positives": expected_tp,
        "Exact_True_Positives_Stable_Tiebreak": exact_tp,
        "Precision": float(precision),
        "Recall": float(recall),
        "Lift": float(precision / prevalence) if prevalence else None,
        "Number_Needed_to_Review": float(k / expected_tp) if expected_tp else None,
        "Cutoff_Score": cutoff,
        "Rows_Above_Cutoff": above_n,
        "Rows_Tied_at_Cutoff": tie_n,
        "Slots_Selected_from_Tie": slots_from_tie,
        "Events_in_Cutoff_Tie": tie_events,
        "Tie_Adjustment_Used": bool(tie_n > slots_from_tie),
    }


pred = pd.read_csv(INPUT)
score_cols = [c for c in pred.columns if c.endswith("_Score")]
score_map = {display_name(c): c for c in score_cols}
assert set(model_order).issubset(score_map), "Expected score columns are missing"

fold_rows = []
for budget in budgets:
    for fold, part in pred.groupby("Fold", sort=False):
        y = part[target].astype(int).to_numpy()
        for model in model_order:
            row = {"Fold": fold, "Model": model}
            row.update(tie_adjusted_budget(y, part[score_map[model]].to_numpy(float), budget))
            fold_rows.append(row)

fold_metrics = pd.DataFrame(fold_rows)
summary_rows = []
for (budget_pct, model), part in fold_metrics.groupby(["Budget_Pct", "Model"], sort=False):
    alerts = int(part["Alerts"].sum())
    expected_tp = float(part["Expected_True_Positives"].sum())
    events = int(part["Events"].sum())
    observations = int(part["Observations"].sum())
    precision = expected_tp / alerts
    recall = expected_tp / events
    prevalence = events / observations
    summary_rows.append({
        "Budget_Pct": float(budget_pct),
        "Model": model,
        "Test_Folds": int(part["Fold"].nunique()),
        "Observations": observations,
        "Events": events,
        "Total_Alerts": alerts,
        "Expected_True_Positives": expected_tp,
        "Precision": precision,
        "Recall": recall,
        "Lift": precision / prevalence,
        "Number_Needed_to_Review": alerts / expected_tp if expected_tp else None,
        "Mean_Fold_Precision": float(part["Precision"].mean()),
        "Mean_Fold_Recall": float(part["Recall"].mean()),
        "Folds_with_Cutoff_Ties": int(part["Tie_Adjustment_Used"].sum()),
    })

summary = pd.DataFrame(summary_rows)
summary["Precision_Rank_within_Budget"] = summary.groupby("Budget_Pct")["Precision"].rank(method="min", ascending=False).astype(int)

# Diminishing-return audit for the primary gap-ratio rule.
gap = summary[summary["Model"].eq("Current Gap Ratio")].sort_values("Budget_Pct").copy()
gap["Additional_Alerts_vs_Prior_Budget"] = gap["Total_Alerts"].diff()
gap["Additional_Expected_TP_vs_Prior_Budget"] = gap["Expected_True_Positives"].diff()
gap["Marginal_Precision_vs_Prior_Budget"] = (
    gap["Additional_Expected_TP_vs_Prior_Budget"] / gap["Additional_Alerts_vs_Prior_Budget"]
)

fold_metrics.to_csv(FOLD_CSV, index=False, float_format="%.17g")
summary.to_csv(SUMMARY_CSV, index=False, float_format="%.17g")
gap.to_csv(OUT / "ETF_Alert_Budget_Gap_Ratio_Tradeoff.csv", index=False, float_format="%.17g")

clean = lambda d: d.astype(object).where(pd.notna(d), None).to_dict("records")
result = {
    "analysis": "Alert-budget sensitivity on untouched walk-forward test predictions",
    "budgets_pct": [5, 10, 20],
    "tie_rule": "Select all rows above the cutoff and use expected positives for the remaining slots within the cutoff tie",
    "fold_metrics": clean(fold_metrics),
    "pooled_summary": clean(summary),
    "gap_ratio_tradeoff": clean(gap),
}
RESULTS_JSON.write_text(json.dumps(result, indent=2), encoding="utf-8")

# Pooled precision and recall curves.
core = ["Current Gap Ratio", "Logistic Regression", "Decision Tree", "Random Forest"]
colors = {"Current Gap Ratio": "#1F4E78", "Logistic Regression": "#4472C4",
          "Decision Tree": "#70AD47", "Random Forest": "#ED7D31"}
fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.7))
for model in core:
    p = summary[summary["Model"].eq(model)].sort_values("Budget_Pct")
    axes[0].plot(p["Budget_Pct"], p["Precision"] * 100, marker="o", label=model, color=colors[model])
    axes[1].plot(p["Budget_Pct"], p["Recall"] * 100, marker="o", label=model, color=colors[model])
axes[0].set_title("Precision decreases as alert capacity expands")
axes[1].set_title("Recall increases as alert capacity expands")
for ax, ylabel in zip(axes, ["Pooled precision (%)", "Pooled recall (%)"]):
    ax.set_xlabel("Alert budget (% of dates)"); ax.set_ylabel(ylabel)
    ax.set_xticks([5, 10, 20]); ax.grid(alpha=.25)
axes[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "ETF_Alert_Budget_Precision_Recall.png", dpi=220, bbox_inches="tight"); plt.close(fig)

# Gap-ratio capacity trade-off.
fig, ax = plt.subplots(figsize=(8.3, 4.8))
ax.plot(gap["Budget_Pct"], gap["Precision"] * 100, marker="o", linewidth=2.2, label="Precision")
ax.plot(gap["Budget_Pct"], gap["Recall"] * 100, marker="o", linewidth=2.2, label="Recall")
ax.set_title("Current gap ratio: operational capacity trade-off")
ax.set_xlabel("Alert budget (% of dates)"); ax.set_ylabel("Performance (%)")
ax.set_xticks([5, 10, 20]); ax.grid(alpha=.25); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIG / "ETF_Gap_Ratio_Budget_Tradeoff.png", dpi=220, bbox_inches="tight"); plt.close(fig)

print(json.dumps({
    "gap_ratio": gap[["Budget_Pct", "Total_Alerts", "Expected_True_Positives", "Precision", "Recall", "Lift", "Number_Needed_to_Review"]].to_dict("records"),
    "top_precision_model_by_budget": summary.loc[summary.groupby("Budget_Pct")["Precision"].idxmax(), ["Budget_Pct", "Model", "Precision"]].to_dict("records"),
}, indent=2))


{
  "gap_ratio": [
    {
      "Budget_Pct": 5.0,
      "Total_Alerts": 37,
      "Expected_True_Positives": 11.0,
      "Precision": 0.2972972972972973,
      "Recall": 0.16923076923076924,
      "Lift": 3.2108108108108113,
      "Number_Needed_to_Review": 3.3636363636363638
    },
    {
      "Budget_Pct": 10.0,
      "Total_Alerts": 72,
      "Expected_True_Positives": 19.0,
      "Precision": 0.2638888888888889,
      "Recall": 0.2923076923076923,
      "Lift": 2.85,
      "Number_Needed_to_Review": 3.789473684210526
    },
    {
      "Budget_Pct": 20.0,
      "Total_Alerts": 142,
      "Expected_True_Positives": 27.0,
      "Precision": 0.19014084507042253,
      "Recall": 0.4153846153846154,
      "Lift": 2.0535211267605633,
      "Number_Needed_to_Review": 5.2592592592592595
    }
  ],
  "top_precision_model_by_budget": [
    {
      "Budget_Pct": 5.0,
      "Model": "Current Gap Ratio",
      "Precision": 0.2972972972972973
    },
    {
      "Budget_Pct": 10.0,
      "Model":

## Step 7A — Measure incremental value from feature groups

Original repository script: `scripts/11_feature_group_ablation.py`


In [18]:
"""Prespecified feature-group ablation with purged walk-forward evaluation.

Each classifier is retuned on validation PR-AUC for every feature group and
fold. Test outcomes never enter preprocessing, hyperparameter selection or
feature-group construction.
"""
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore")
ROOT = PROJECT_ROOT
DATA = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
INPUT = DATA / "ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv"
METRICS_CSV = OUT / "ETF_Feature_Ablation_Metrics.csv"
SUMMARY_CSV = OUT / "ETF_Feature_Ablation_Summary.csv"
DELTAS_CSV = OUT / "ETF_Feature_Ablation_Deltas.csv"
TUNING_CSV = OUT / "ETF_Feature_Ablation_Tuning.csv"
RESULTS_JSON = OUT / "feature_ablation_results.json"

target = "Y_Abnormal_Dislocation_95Q"
folds = [
    {"Fold": "Fold 1", "Train_End": "2022-12-31", "Validation_Year": 2023, "Test_Year": 2024},
    {"Fold": "Fold 2", "Train_End": "2023-12-31", "Validation_Year": 2024, "Test_Year": 2025},
    {"Fold": "Fold 3", "Train_End": "2024-12-31", "Validation_Year": 2025, "Test_Year": 2026},
]

gap_only = [
    "X_Dislocation_Pct_t", "X_Abs_Dislocation_Pct_t", "X_Dislocation_Lag_1_Pct",
    "X_Dislocation_Lag_2_Pct", "X_Dislocation_Lag_5_Pct", "X_Premium_Flag_t",
]
gap_history_add = [
    "X_Abs_Dislocation_Mean_5D_Pct", "X_Abs_Dislocation_Mean_20D_Pct",
    "X_Abs_Dislocation_Std_20D_Pct", "X_Current_Abs_Gap_to_Threshold_Ratio",
]
market_stress_add = [
    "X_NIFTY50_Return_1D_Pct", "X_NIFTY50_Return_5D_Pct", "X_INDIA_VIX_Close_t",
    "X_INDIA_VIX_Change_1D_Pct", "X_VIX_Change_5D_Pct",
]
liquidity_add = [
    "X_Log_ETF_Volume_t", "X_Volume_to_20D_Median", "X_Log_ETF_Turnover_t",
    "X_Average_Trade_Value_INR_t", "X_Delivery_Pct_t", "X_Intraday_Range_Pct_t",
]

df = pd.read_csv(INPUT, parse_dates=["Prediction_Date", "Target_Date"])
all_features = [c for c in df.columns if c.startswith("X_")]
assert len(all_features) == 25
feature_groups = {
    "Gap Only": gap_only,
    "Gap History": gap_only + gap_history_add,
    "Gap + Market Stress": gap_only + gap_history_add + market_stress_add,
    "Gap + Liquidity": gap_only + gap_history_add + liquidity_add,
    "Full 25 Predictors": all_features,
}
for name, cols in feature_groups.items():
    assert len(cols) == len(set(cols)) and set(cols).issubset(all_features), name


def purge_prior(prior, following):
    if prior.empty or following.empty:
        return prior.copy(), 0
    first_next_prediction = following["Prediction_Date"].min()
    keep = prior["Target_Date"] < first_next_prediction
    return prior.loc[keep].copy(), int((~keep).sum())


def tie_adjusted_budget(y, score, budget=0.10):
    y = np.asarray(y, dtype=int)
    score = np.asarray(score, dtype=float)
    k = max(1, int(math.ceil(len(y) * budget)))
    order = np.argsort(-score, kind="stable")
    cutoff = score[order[k - 1]]
    above, tied = score > cutoff, score == cutoff
    slots = k - int(above.sum())
    expected_tp = float(y[above].sum() + slots * y[tied].mean())
    precision = expected_tp / k
    prevalence = float(y.mean())
    return {
        "Alerts_at_10Pct": k,
        "Expected_TP_at_10Pct": expected_tp,
        "Precision_at_10Pct": precision,
        "Recall_at_10Pct": expected_tp / max(1, int(y.sum())),
        "Lift_at_10Pct": precision / prevalence if prevalence else None,
        "Cutoff_Tie_Rows": int(tied.sum()),
    }


def metric_row(group, feature_count, fold, model, split, y, p):
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    row = {
        "Feature_Group": group, "Feature_Count": feature_count, "Fold": fold,
        "Model": model, "Split": split, "Observations": int(len(y)),
        "Events": int(y.sum()), "Event_Rate": float(y.mean()),
        "PR_AUC": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "Brier_Score": float(brier_score_loss(y, np.clip(p, 1e-6, 1 - 1e-6))),
    }
    row.update(tie_adjusted_budget(y, p))
    return row


def fit_models(train, valid, features, group, fold, tuning_rows):
    Xtr, ytr = train[features], train[target].astype(int)
    Xva, yva = valid[features], valid[target].astype(int)
    chosen = {}

    candidates = []
    for c in [0.01, 0.1, 1.0, 10.0]:
        model = Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(C=c, penalty="l2", solver="liblinear", max_iter=5000, random_state=42)),
        ])
        model.fit(Xtr, ytr); p = model.predict_proba(Xva)[:, 1]
        row = {"Feature_Group": group, "Fold": fold, "Model": "Logistic Regression", "Parameters": f"C={c}",
               "Validation_PR_AUC": float(average_precision_score(yva, p)), "Validation_Brier": float(brier_score_loss(yva, p))}
        tuning_rows.append(row); candidates.append((row, model))
    _, chosen["Logistic Regression"] = max(candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"]))

    candidates = []
    for depth in [2, 3, 4, 5]:
        for leaf in [10, 25, 50]:
            for weight in [None, "balanced"]:
                model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=leaf, class_weight=weight, random_state=42)
                model.fit(Xtr, ytr); p = model.predict_proba(Xva)[:, 1]
                row = {"Feature_Group": group, "Fold": fold, "Model": "Decision Tree",
                       "Parameters": f"depth={depth}; leaf={leaf}; weight={weight}",
                       "Validation_PR_AUC": float(average_precision_score(yva, p)), "Validation_Brier": float(brier_score_loss(yva, p))}
                tuning_rows.append(row); candidates.append((row, model))
    _, chosen["Decision Tree"] = max(candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"]))

    candidates = []
    for depth in [3, 5, None]:
        for leaf in [5, 10, 20]:
            for max_features in ["sqrt", 0.5]:
                for weight in [None, "balanced_subsample"]:
                    model = RandomForestClassifier(
                        n_estimators=400, max_depth=depth, min_samples_leaf=leaf,
                        max_features=max_features, class_weight=weight, random_state=42, n_jobs=-1,
                    )
                    model.fit(Xtr, ytr); p = model.predict_proba(Xva)[:, 1]
                    row = {"Feature_Group": group, "Fold": fold, "Model": "Random Forest",
                           "Parameters": f"depth={depth}; leaf={leaf}; max_features={max_features}; weight={weight}",
                           "Validation_PR_AUC": float(average_precision_score(yva, p)), "Validation_Brier": float(brier_score_loss(yva, p))}
                    tuning_rows.append(row); candidates.append((row, model))
    _, chosen["Random Forest"] = max(candidates, key=lambda x: (x[0]["Validation_PR_AUC"], -x[0]["Validation_Brier"]))
    return chosen


metric_rows, tuning_rows, audit_rows = [], [], []
for group, features in feature_groups.items():
    for spec in folds:
        fold = spec["Fold"]
        train = df[df["Target_Date"] <= pd.Timestamp(spec["Train_End"])].copy()
        valid = df[df["Target_Date"].dt.year.eq(spec["Validation_Year"])].copy()
        test = df[df["Target_Date"].dt.year.eq(spec["Test_Year"])].copy()
        train, purge_train = purge_prior(train, valid)
        valid, purge_valid = purge_prior(valid, test)
        feasible = all(part[target].nunique() == 2 for part in [train, valid, test])
        audit_rows.append({
            "Feature_Group": group, "Feature_Count": len(features), "Fold": fold,
            "Train_Rows": len(train), "Train_Events": int(train[target].sum()),
            "Validation_Rows": len(valid), "Validation_Events": int(valid[target].sum()),
            "Test_Rows": len(test), "Test_Events": int(test[target].sum()),
            "Purged_Train_Rows": purge_train, "Purged_Validation_Rows": purge_valid,
            "Feasible": feasible,
        })
        if not feasible:
            continue
        models = fit_models(train, valid, features, group, fold, tuning_rows)
        for split, part in [("Validation", valid), ("Test", test)]:
            y = part[target].astype(int).to_numpy()
            for model_name, model in models.items():
                p = model.predict_proba(part[features])[:, 1]
                metric_rows.append(metric_row(group, len(features), fold, model_name, split, y, p))

metrics = pd.DataFrame(metric_rows)
tuning = pd.DataFrame(tuning_rows)
audit = pd.DataFrame(audit_rows)
test = metrics[metrics["Split"].eq("Test")].copy()
summary = test.groupby(["Feature_Group", "Feature_Count", "Model"], as_index=False).agg(
    Folds=("Fold", "nunique"), Mean_PR_AUC=("PR_AUC", "mean"),
    Minimum_PR_AUC=("PR_AUC", "min"), Maximum_PR_AUC=("PR_AUC", "max"),
    Mean_ROC_AUC=("ROC_AUC", "mean"), Mean_Brier=("Brier_Score", "mean"),
    Mean_Precision_at_10Pct=("Precision_at_10Pct", "mean"),
    Mean_Recall_at_10Pct=("Recall_at_10Pct", "mean"), Mean_Lift_at_10Pct=("Lift_at_10Pct", "mean"),
)
summary["PR_AUC_Rank_within_Model"] = summary.groupby("Model")["Mean_PR_AUC"].rank(method="min", ascending=False).astype(int)

baseline = summary[summary["Feature_Group"].eq("Gap History")][
    ["Model", "Mean_PR_AUC", "Mean_Brier", "Mean_Precision_at_10Pct"]
].rename(columns={
    "Mean_PR_AUC": "Gap_History_PR_AUC", "Mean_Brier": "Gap_History_Brier",
    "Mean_Precision_at_10Pct": "Gap_History_Precision_at_10Pct",
})
deltas = summary.merge(baseline, on="Model", how="left")
deltas["PR_AUC_Delta_vs_Gap_History"] = deltas["Mean_PR_AUC"] - deltas["Gap_History_PR_AUC"]
deltas["Brier_Improvement_vs_Gap_History"] = deltas["Gap_History_Brier"] - deltas["Mean_Brier"]
deltas["Precision_Delta_vs_Gap_History"] = deltas["Mean_Precision_at_10Pct"] - deltas["Gap_History_Precision_at_10Pct"]

# Transparent reference recomputed on the same test folds.
gap_reference = []
for spec in folds:
    part = df[df["Target_Date"].dt.year.eq(spec["Test_Year"])]
    gap_reference.append(float(average_precision_score(part[target], part["X_Current_Abs_Gap_to_Threshold_Ratio"])))
gap_reference_mean = float(np.mean(gap_reference))

metrics.to_csv(METRICS_CSV, index=False, float_format="%.17g")
summary.to_csv(SUMMARY_CSV, index=False, float_format="%.17g")
deltas.to_csv(DELTAS_CSV, index=False, float_format="%.17g")
tuning.to_csv(TUNING_CSV, index=False, float_format="%.17g")
audit.to_csv(OUT / "ETF_Feature_Ablation_Fold_Audit.csv", index=False)

clean = lambda d: d.astype(object).where(pd.notna(d), None).to_dict("records")
result = {
    "analysis": "Feature-group ablation with validation-only retuning in each walk-forward fold",
    "feature_groups": feature_groups,
    "current_gap_ratio_reference_mean_pr_auc": gap_reference_mean,
    "current_gap_ratio_reference_fold_pr_auc": gap_reference,
    "fold_audit": clean(audit), "test_summary": clean(summary), "deltas_vs_gap_history": clean(deltas),
}
RESULTS_JSON.write_text(json.dumps(result, indent=2), encoding="utf-8")

group_order = list(feature_groups)
model_order = ["Logistic Regression", "Decision Tree", "Random Forest"]
plot = summary.pivot(index="Feature_Group", columns="Model", values="Mean_PR_AUC").reindex(group_order)[model_order]
ax = plot.plot(kind="bar", figsize=(11, 5.5), color=["#4472C4", "#70AD47", "#ED7D31"], width=.78)
ax.axhline(gap_reference_mean, color="#1F4E78", linestyle="--", linewidth=1.5, label="Current gap ratio reference")
ax.set_title("Feature-group ablation: mean walk-forward test PR-AUC")
ax.set_ylabel("Mean test PR-AUC"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=.25); ax.legend(frameon=False, ncol=2, fontsize=8)
fig = ax.get_figure(); fig.tight_layout(); fig.savefig(FIG / "ETF_Feature_Ablation_PR_AUC.png", dpi=220, bbox_inches="tight"); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(12.2, 4.4), sharey=True)
for ax, model in zip(axes, model_order):
    p = deltas[deltas["Model"].eq(model)].set_index("Feature_Group").reindex(group_order)
    colors = ["#70AD47" if x >= 0 else "#C0504D" for x in p["PR_AUC_Delta_vs_Gap_History"]]
    ax.barh(group_order, p["PR_AUC_Delta_vs_Gap_History"], color=colors)
    ax.axvline(0, color="#7F7F7F", linewidth=1); ax.set_title(model); ax.grid(axis="x", alpha=.2)
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5)); ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
    ax.tick_params(axis="x", labelsize=9)
axes[0].set_xlabel("PR-AUC difference vs gap history")
fig.suptitle("Incremental value relative to gap-history features", y=1.02)
fig.tight_layout(); fig.savefig(FIG / "ETF_Feature_Ablation_Deltas.png", dpi=220, bbox_inches="tight"); plt.close(fig)

best = summary.loc[summary.groupby("Model")["Mean_PR_AUC"].idxmax(), ["Model", "Feature_Group", "Mean_PR_AUC"]]
print(json.dumps({
    "gap_ratio_reference_mean_pr_auc": gap_reference_mean,
    "best_group_by_model": best.to_dict("records"),
    "market_stress_deltas": deltas[deltas["Feature_Group"].eq("Gap + Market Stress")][["Model", "PR_AUC_Delta_vs_Gap_History", "Brier_Improvement_vs_Gap_History"]].to_dict("records"),
    "liquidity_deltas": deltas[deltas["Feature_Group"].eq("Gap + Liquidity")][["Model", "PR_AUC_Delta_vs_Gap_History", "Brier_Improvement_vs_Gap_History"]].to_dict("records"),
}, indent=2))


{
  "gap_ratio_reference_mean_pr_auc": 0.23326676949033884,
  "best_group_by_model": [
    {
      "Model": "Decision Tree",
      "Feature_Group": "Gap Only",
      "Mean_PR_AUC": 0.17058046447920028
    },
    {
      "Model": "Logistic Regression",
      "Feature_Group": "Gap + Market Stress",
      "Mean_PR_AUC": 0.19500010602147275
    },
    {
      "Model": "Random Forest",
      "Feature_Group": "Full 25 Predictors",
      "Mean_PR_AUC": 0.1842833685680513
    }
  ],
  "market_stress_deltas": [
    {
      "Model": "Decision Tree",
      "PR_AUC_Delta_vs_Gap_History": 0.007095170985507104,
      "Brier_Improvement_vs_Gap_History": 0.031984513005373166
    },
    {
      "Model": "Logistic Regression",
      "PR_AUC_Delta_vs_Gap_History": 0.042055607245211096,
      "Brier_Improvement_vs_Gap_History": 0.002826360740205855
    },
    {
      "Model": "Random Forest",
      "PR_AUC_Delta_vs_Gap_History": -0.004853486952800107,
      "Brier_Improvement_vs_Gap_History": -0.015958988

## Step 7B — Diagnose temporal shift, calibration and alert instability

Original repository script: `scripts/12_temporal_shift_diagnostics.py`


In [20]:
"""Prespecified temporal-shift diagnostics for the enhanced ETF study.

Implements Section 13 of docs/ANALYSIS_PLAN_V2.md. Only validation-selected
walk-forward specifications are refitted. Test outcomes do not influence model
fitting or the locked alert cutoffs.
"""
from __future__ import annotations

from pathlib import Path
import json
import math
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


warnings.filterwarnings("ignore")
ROOT = PROJECT_ROOT
DATA, OUT, FIG = ROOT / "data" / "processed", ROOT / "outputs", ROOT / "figures"
OUT.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
INPUT = DATA / "ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv"
TUNING = OUT / "ETF_Walk_Forward_Tuning.csv"
TARGET = "Y_Abnormal_Dislocation_95Q"

FILES = {
    "year": OUT / "ETF_Temporal_Shift_Year_Summary.csv",
    "model": OUT / "ETF_Temporal_Shift_Model_Metrics.csv",
    "calibration": OUT / "ETF_Temporal_Shift_Calibration.csv",
    "locked": OUT / "ETF_Temporal_Shift_Locked_Alerts.csv",
    "feature_psi": OUT / "ETF_Temporal_Shift_Feature_PSI.csv",
    "score_psi": OUT / "ETF_Temporal_Shift_Score_PSI.csv",
    "json": OUT / "temporal_shift_results.json",
}

PROBABILITY_MODELS = {"Historical Prevalence", "Logistic Regression", "Decision Tree", "Random Forest"}
MODEL_ORDER = ["Historical Prevalence", "Current Absolute Gap", "Current Gap Ratio",
               "Previous Event Persistence", "Logistic Regression", "Decision Tree", "Random Forest"]
FOLDS = [
    {"Fold": "Fold 1", "Train_End": "2022-12-31", "Validation_Year": 2023, "Test_Year": 2024},
    {"Fold": "Fold 2", "Train_End": "2023-12-31", "Validation_Year": 2024, "Test_Year": 2025},
    {"Fold": "Fold 3", "Train_End": "2024-12-31", "Validation_Year": 2025, "Test_Year": 2026},
]


def purge_prior(prior, following):
    if prior.empty or following.empty:
        return prior.copy()
    return prior.loc[prior["Target_Date"] < following["Prediction_Date"].min()].copy()


def selected_parameter(tuning, fold, model):
    rows = tuning[(tuning["Fold"] == fold) & (tuning["Model"] == model)].copy()
    return str(rows.sort_values(["Validation_PR_AUC", "Validation_Brier"],
                                ascending=[False, True]).iloc[0]["Parameters"])


def make_model(name, parameters):
    if name == "Logistic Regression":
        c = float(parameters.split("=")[1])
        return Pipeline([("scale", StandardScaler()),
                         ("model", LogisticRegression(C=c, penalty="l2", solver="liblinear",
                                                      max_iter=5000, random_state=42))])
    fields = dict(part.split("=", 1) for part in parameters.split("; "))
    depth = None if fields["depth"] == "None" else int(fields["depth"])
    weight = None if fields["weight"] == "None" else fields["weight"]
    if name == "Decision Tree":
        return DecisionTreeClassifier(max_depth=depth, min_samples_leaf=int(fields["leaf"]),
                                      class_weight=weight, random_state=42)
    max_features = fields["max_features"]
    if re.fullmatch(r"\d+(\.\d+)?", max_features):
        max_features = float(max_features)
    return RandomForestClassifier(n_estimators=400, max_depth=depth,
                                  min_samples_leaf=int(fields["leaf"]), max_features=max_features,
                                  class_weight=weight, random_state=42, n_jobs=-1)


def five_bin_ece(y, p):
    work = pd.DataFrame({"y": y, "p": p})
    try:
        work["bin"] = pd.qcut(work["p"], q=5, duplicates="drop")
    except ValueError:
        return float(abs(work["y"].mean() - work["p"].mean()))
    if work["bin"].notna().sum() == 0:
        return float(abs(work["y"].mean() - work["p"].mean()))
    return float(sum(len(g) / len(work) * abs(g["y"].mean() - g["p"].mean())
                     for _, g in work.groupby("bin", observed=True)))


def calibration_stats(y, p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); y = np.asarray(y, int)
    mean_p, rate = float(p.mean()), float(y.mean())
    cil = float(np.log(rate / (1 - rate)) - np.log(mean_p / (1 - mean_p)))
    if np.ptp(p) < 1e-12:
        intercept, slope = cil, None
    else:
        logits = np.log(p / (1 - p)).reshape(-1, 1)
        recal = LogisticRegression(penalty=None, solver="lbfgs", max_iter=5000).fit(logits, y)
        intercept, slope = float(recal.intercept_[0]), float(recal.coef_[0, 0])
    return {"Mean_Predicted_Probability": mean_p, "Observed_Event_Rate": rate,
            "Calibration_in_the_Large": cil, "Calibration_Intercept": intercept,
            "Calibration_Slope": slope, "Five_Bin_ECE": five_bin_ece(y, p),
            "Brier_Score": float(brier_score_loss(y, p))}


def psi(reference, current, bins=10):
    reference, current = np.asarray(reference, float), np.asarray(current, float)
    reference, current = reference[np.isfinite(reference)], current[np.isfinite(current)]
    if not len(reference) or not len(current):
        return float("nan")
    edges = np.unique(np.quantile(reference, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        lo, hi = min(reference.min(), current.min()), max(reference.max(), current.max())
        if lo == hi:
            return 0.0
        edges = np.linspace(lo, hi, min(bins, 5) + 1)
    edges[0], edges[-1] = -np.inf, np.inf
    ref = np.clip(np.histogram(reference, bins=edges)[0] / len(reference), 1e-6, None)
    cur = np.clip(np.histogram(current, bins=edges)[0] / len(current), 1e-6, None)
    return float(np.sum((cur - ref) * np.log(cur / ref)))


def severity(value):
    if not np.isfinite(value): return "Unavailable"
    if value < .10: return "Low"
    if value < .25: return "Moderate"
    return "Large"


df = pd.read_csv(INPUT, parse_dates=["Prediction_Date", "Target_Date"])
tuning = pd.read_csv(TUNING)
features = [c for c in df.columns if c.startswith("X_")]
year_rows, model_rows, calibration_rows, locked_rows, feature_psi_rows, score_psi_rows = [], [], [], [], [], []

for spec in FOLDS:
    fold = spec["Fold"]
    train = df[df["Target_Date"] <= pd.Timestamp(spec["Train_End"])].copy()
    valid = df[df["Target_Date"].dt.year.eq(spec["Validation_Year"])].copy()
    test = df[df["Target_Date"].dt.year.eq(spec["Test_Year"])].copy()
    train, valid = purge_prior(train, valid), purge_prior(valid, test)
    Xtr, ytr = train[features], train[TARGET].astype(int).to_numpy()
    yva, yte = valid[TARGET].astype(int).to_numpy(), test[TARGET].astype(int).to_numpy()
    models = {}
    for name in ["Logistic Regression", "Decision Tree", "Random Forest"]:
        model = make_model(name, selected_parameter(tuning, fold, name)).fit(Xtr, ytr)
        models[name] = model
    prevalence = float(ytr.mean())

    def get_scores(part):
        result = {
            "Historical Prevalence": np.repeat(prevalence, len(part)),
            "Current Absolute Gap": part["X_Abs_Dislocation_Pct_t"].to_numpy(float),
            "Current Gap Ratio": part["X_Current_Abs_Gap_to_Threshold_Ratio"].to_numpy(float),
            "Previous Event Persistence": (part["X_Abs_Dislocation_Pct_t"] >
                                           part["Rolling_Threshold_95Pct"]).astype(float).to_numpy(),
        }
        for name, model in models.items(): result[name] = model.predict_proba(part[features])[:, 1]
        return result

    va_scores, te_scores = get_scores(valid), get_scores(test)
    abs_target = test["Target_Abs_Dislocation_Pct"].to_numpy(float)
    year_rows.append({"Fold": fold, "Test_Year": spec["Test_Year"], "Observations": len(test),
                      "Events": int(yte.sum()), "Event_Rate": float(yte.mean()),
                      "Mean_Abs_Dislocation_Pct": float(np.mean(abs_target)),
                      "Median_Abs_Dislocation_Pct": float(np.median(abs_target)),
                      "P95_Abs_Dislocation_Pct": float(np.quantile(abs_target, .95)),
                      "Maximum_Abs_Dislocation_Pct": float(np.max(abs_target))})

    for name in MODEL_ORDER:
        score, valid_score = te_scores[name], va_scores[name]
        model_rows.append({"Fold": fold, "Test_Year": spec["Test_Year"], "Model": name,
                           "Observations": len(test), "Events": int(yte.sum()),
                           "Event_Rate": float(yte.mean()),
                           "PR_AUC": float(average_precision_score(yte, score)),
                           "ROC_AUC": float(roc_auc_score(yte, score)),
                           "Mean_Score": float(np.mean(score)), "Score_SD": float(np.std(score, ddof=1))})
        if name in PROBABILITY_MODELS:
            calibration_rows.append({"Fold": fold, "Test_Year": spec["Test_Year"], "Model": name,
                                     **calibration_stats(yte, score)})
        k = max(1, int(math.ceil(.10 * len(valid_score))))
        cutoff = float(np.sort(valid_score)[::-1][k - 1])
        alert = score >= cutoff; tp = int(np.sum(alert & (yte == 1)))
        locked_rows.append({"Fold": fold, "Validation_Year": spec["Validation_Year"],
                            "Test_Year": spec["Test_Year"], "Model": name,
                            "Validation_Locked_Cutoff": cutoff, "Test_Alerts": int(alert.sum()),
                            "Test_Alert_Rate": float(alert.mean()),
                            "Alert_Rate_Deviation_from_10Pct": float(alert.mean() - .10),
                            "Test_True_Positives": tp,
                            "Test_Precision": float(tp / alert.sum()) if alert.sum() else None,
                            "Test_Recall": float(tp / yte.sum()) if yte.sum() else None})
        value = psi(valid_score, score)
        score_psi_rows.append({"Fold": fold, "Validation_Year": spec["Validation_Year"],
                               "Test_Year": spec["Test_Year"], "Model": name,
                               "Score_PSI": value, "PSI_Severity": severity(value)})
    for feature in features:
        value = psi(valid[feature], test[feature])
        feature_psi_rows.append({"Fold": fold, "Validation_Year": spec["Validation_Year"],
                                 "Test_Year": spec["Test_Year"], "Feature": feature,
                                 "Feature_PSI": value, "PSI_Severity": severity(value)})

year_summary, model_metrics = pd.DataFrame(year_rows), pd.DataFrame(model_rows)
calibration, locked = pd.DataFrame(calibration_rows), pd.DataFrame(locked_rows)
feature_psi, score_psi = pd.DataFrame(feature_psi_rows), pd.DataFrame(score_psi_rows)
model_metrics["PR_AUC_Rank"] = model_metrics.groupby("Fold")["PR_AUC"].rank(method="min", ascending=False).astype(int)

for frame, key in [(year_summary, "year"), (model_metrics, "model"), (calibration, "calibration"),
                   (locked, "locked"), (feature_psi, "feature_psi"), (score_psi, "score_psi")]:
    frame.to_csv(FILES[key], index=False, float_format="%.17g")

feature_fold = feature_psi.groupby(["Fold", "Test_Year"], as_index=False).agg(
    Median_Feature_PSI=("Feature_PSI", "median"), Maximum_Feature_PSI=("Feature_PSI", "max"),
    Features_Moderate_or_Large=("Feature_PSI", lambda s: int((s >= .10).sum())),
    Features_Large=("Feature_PSI", lambda s: int((s >= .25).sum())))
result = {"analysis": "Prespecified temporal-shift and calibration diagnostics",
          "psi_interpretation": {"low": "<0.10", "moderate": "0.10 to <0.25", "large": ">=0.25"},
          "year_summary": year_summary.to_dict("records"),
          "feature_shift_summary": feature_fold.to_dict("records"),
          "model_metrics": model_metrics.to_dict("records"),
          "calibration": calibration.replace({np.nan: None}).to_dict("records"),
          "locked_alerts": locked.replace({np.nan: None}).to_dict("records"),
          "score_psi": score_psi.to_dict("records")}
FILES["json"].write_text(json.dumps(result, indent=2), encoding="utf-8")

sns.set_theme(style="whitegrid")
fig, ax1 = plt.subplots(figsize=(8.6, 4.9)); x = np.arange(len(year_summary))
ax1.bar(x, 100 * year_summary["Event_Rate"], color="#1F4E78", width=.55)
ax1.set_ylabel("Abnormal-event prevalence (%)", color="#1F4E78")
ax1.set_xticks(x, year_summary["Test_Year"].astype(str)); ax1.set_xlabel("Untouched test year")
ax2 = ax1.twinx(); ax2.plot(x, year_summary["P95_Abs_Dislocation_Pct"], color="#ED7D31", marker="o", linewidth=2.2)
ax2.set_ylabel("95th percentile absolute dislocation (%)", color="#C65911")
ax1.set_title("Outcome regime changes across walk-forward test years")
fig.tight_layout(); fig.savefig(FIG / "ETF_Temporal_Shift_Outcomes.png", dpi=220, bbox_inches="tight"); plt.close(fig)

heat = model_metrics.pivot(index="Model", columns="Test_Year", values="PR_AUC").reindex(MODEL_ORDER)
fig, ax = plt.subplots(figsize=(7.3, 5.4)); sns.heatmap(heat, annot=True, fmt=".3f", cmap="Blues",
                                                         linewidths=.5, cbar_kws={"label": "PR-AUC"}, ax=ax)
ax.set_title("Model ranking changes across test years"); ax.set_xlabel("Test year"); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(FIG / "ETF_Temporal_Shift_PR_AUC.png", dpi=220, bbox_inches="tight"); plt.close(fig)

plot_cal = calibration[calibration["Model"].ne("Historical Prevalence")]
fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.8), sharex=True, sharey=True)
for ax, (model, part) in zip(axes, plot_cal.groupby("Model", sort=False)):
    ax.plot([0, .20], [0, .20], linestyle="--", color="#7F8C8D", linewidth=1)
    for _, row in part.iterrows():
        ax.scatter(row["Observed_Event_Rate"], row["Mean_Predicted_Probability"], s=65)
        ax.annotate(str(int(row["Test_Year"])), (row["Observed_Event_Rate"], row["Mean_Predicted_Probability"]),
                    xytext=(5, 4), textcoords="offset points", fontsize=8)
    ax.set_title(model); ax.set_xlabel("Observed event rate")
axes[0].set_ylabel("Mean predicted probability")
fig.suptitle("Probability calibration shifts by test year", y=1.03)
fig.tight_layout(); fig.savefig(FIG / "ETF_Temporal_Shift_Calibration.png", dpi=220, bbox_inches="tight"); plt.close(fig)

plot_locked = locked[~locked["Model"].isin(["Historical Prevalence", "Previous Event Persistence"])]
fig, ax = plt.subplots(figsize=(9.4, 5.0))
sns.lineplot(data=plot_locked, x="Test_Year", y="Test_Alert_Rate", hue="Model", marker="o", ax=ax)
ax.axhline(.10, linestyle="--", color="#7F8C8D", label="Intended 10% rate")
ax.set_ylabel("Test alert rate"); ax.set_xlabel("Test year")
ax.set_title("Validation-locked cutoffs do not preserve alert volume")
ax.set_xticks([2024, 2025, 2026], ["2024", "2025", "2026"])
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}"); ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout(); fig.savefig(FIG / "ETF_Temporal_Shift_Locked_Alerts.png", dpi=220, bbox_inches="tight"); plt.close(fig)

print(json.dumps({"year_summary": result["year_summary"],
                  "feature_shift_summary": result["feature_shift_summary"],
                  "largest_locked_alert_deviations": locked.loc[~locked["Model"].eq("Historical Prevalence")]
                  .assign(abs_dev=lambda d: d["Alert_Rate_Deviation_from_10Pct"].abs())
                  .nlargest(8, "abs_dev")[["Fold", "Test_Year", "Model", "Test_Alert_Rate",
                                            "Alert_Rate_Deviation_from_10Pct"]].to_dict("records")}, indent=2))


{
  "year_summary": [
    {
      "Fold": "Fold 1",
      "Test_Year": 2024,
      "Observations": 242,
      "Events": 26,
      "Event_Rate": 0.10743801652892562,
      "Mean_Abs_Dislocation_Pct": 0.13686204132231403,
      "Median_Abs_Dislocation_Pct": 0.118795,
      "P95_Abs_Dislocation_Pct": 0.27388574999999993,
      "Maximum_Abs_Dislocation_Pct": 3.009982
    },
    {
      "Fold": "Fold 2",
      "Test_Year": 2025,
      "Observations": 246,
      "Events": 5,
      "Event_Rate": 0.02032520325203252,
      "Mean_Abs_Dislocation_Pct": 0.10647492682926829,
      "Median_Abs_Dislocation_Pct": 0.1095845,
      "P95_Abs_Dislocation_Pct": 0.193423,
      "Maximum_Abs_Dislocation_Pct": 0.422978
    },
    {
      "Fold": "Fold 3",
      "Test_Year": 2026,
      "Observations": 214,
      "Events": 34,
      "Event_Rate": 0.1588785046728972,
      "Mean_Abs_Dislocation_Pct": 0.1527131121495327,
      "Median_Abs_Dislocation_Pct": 0.1230595,
      "P95_Abs_Dislocation_Pct": 0.36062415,

## Step 8A — Recreate core report figures

Original repository script: `scripts/04_make_report_figures.py`


In [22]:
"""Recreate the two figures included in the second-review report."""
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd


ROOT = PROJECT_ROOT
RESULTS = json.loads((ROOT / "outputs" / "model_results.json").read_text(encoding="utf-8"))
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
metrics = pd.DataFrame([r for r in RESULTS["metrics"] if r["Split"] == "Test"])
budget = pd.DataFrame([r for r in RESULTS["budget_metrics"] if r["Split"] == "Test"])

plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10})
fig, ax = plt.subplots(figsize=(7.2, 3.5))
ax.bar(metrics["Model"], metrics["PR_AUC"], color=["#A5A5A5", "#1F77B4", "#4472C4", "#70AD47", "#ED7D31"])
baseline = metrics.loc[metrics.Model.eq("Historical Prevalence"), "PR_AUC"].iloc[0]
ax.axhline(baseline, color="#7F7F7F", linestyle="--", linewidth=1, label="No-skill PR-AUC")
ax.set(ylabel="PR-AUC", title="Out-of-time test PR-AUC", ylim=(0, 0.34))
ax.grid(axis="y", alpha=.25); ax.tick_params(axis="x", rotation=18); ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(FIGURES / "ETF_Test_PR_AUC.png", dpi=220, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 3.5))
ax.bar(budget["Model"], budget["Precision_at_Budget"] * 100, color=["#1F77B4", "#4472C4", "#70AD47", "#ED7D31"])
ax.set(ylabel="Precision among alerts (%)", title="Test performance under a 10% alert budget", ylim=(0, 42))
ax.grid(axis="y", alpha=.25)
for i, (precision, events) in enumerate(zip(budget["Precision_at_Budget"] * 100, budget["True_Positives"])):
    ax.text(i, precision + 1, f"{precision:.1f}%\n{int(events)} events", ha="center", va="bottom", fontsize=9)
ax.tick_params(axis="x", rotation=12)
fig.tight_layout(); fig.savefig(FIGURES / "ETF_Alert_Budget.png", dpi=220, bbox_inches="tight"); plt.close(fig)
print("Figures recreated in figures/")


Figures recreated in figures/
Completed in 0.4 seconds.


## Step 8B — Verify reported counts and metrics

Original repository script: `scripts/05_verify_reported_results.py`


In [24]:
"""Fail loudly if a reproduced value differs from the second-review report."""
from pathlib import Path
import json

import pandas as pd


ROOT = PROJECT_ROOT
DATA = ROOT / "data" / "processed"
OUT = ROOT / "outputs"

market = pd.read_csv(DATA / "NIFTYBEES_EQ_Market_Data_2021_to_2026_Clean.csv")
nav = pd.read_csv(DATA / "NIFTYBEES_NAV_AMFI_official_2021_to_2026_Updated.csv")
master = pd.read_csv(DATA / "ETF_Master_Merged_2021_to_2026.csv")
analysis = pd.read_csv(DATA / "ETF_Analysis_Ready_2021_to_2026.csv")
model = pd.read_csv(DATA / "ETF_Modeling_Dataset_Leakage_Safe_2021_to_2026.csv")
results = json.loads((OUT / "model_results.json").read_text(encoding="utf-8"))

assert len(market) == 1456
assert len(nav) == 1451
assert len(master) == 1456
assert len(analysis) == 1446
assert len(model) == 1185
assert int(model["Y_Abnormal_Dislocation_95Q"].sum()) == 80

expected_splits = {"Train": (725, 41), "Validation": (246, 5), "Test": (214, 34)}
for split, (rows, events) in expected_splits.items():
    part = model[model["Chronological_Split"].eq(split)]
    assert len(part) == rows
    assert int(part["Y_Abnormal_Dislocation_95Q"].sum()) == events

test = {r["Model"]: r for r in results["metrics"] if r["Split"] == "Test"}
expected_pr_auc = {
    "Historical Prevalence": 0.159,
    "Current Gap Rule": 0.290,
    "Logistic Regression": 0.267,
    "Decision Tree": 0.206,
    "Random Forest": 0.202,
}
for name, expected in expected_pr_auc.items():
    # The report presents model metrics to three decimals. Rebuilding the merged
    # CSV can alter later floating-point digits without altering a reported value.
    assert round(test[name]["PR_AUC"], 3) == expected, (name, test[name]["PR_AUC"], expected)

assert round(test["Logistic Regression"]["Brier_Score"], 3) == 0.134
assert round(test["Decision Tree"]["Brier_Score"], 3) == 0.152
assert round(test["Random Forest"]["Brier_Score"], 3) == 0.162

budget = {r["Model"]: r for r in results["budget_metrics"] if r["Split"] == "Test"}
assert budget["Current Gap Rule"]["Alerts"] == 22
assert budget["Current Gap Rule"]["True_Positives"] == 8
assert abs(budget["Current Gap Rule"]["Precision_at_Budget"] - 8 / 22) < 1e-12
assert results["selected_model"] == "Random Forest"
print("PASS: all data counts, event counts, split counts and reported test results match the second-review draft.")


PASS: all data counts, event counts, split counts and reported test results match the second-review draft.
Completed in 0.1 seconds.


## Step 8C — Build the revised second-review Word report

Original repository script: `scripts/06_build_second_review_report.py`


In [26]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import pandas as pd
from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

ROOT = PROJECT_ROOT
OUT = ROOT / 'report'
OUT.mkdir(parents=True, exist_ok=True)
FIGURES = ROOT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS = json.loads((ROOT / 'outputs' / 'model_results.json').read_text(encoding='utf-8'))
DOCX = OUT / 'ETF_Project_Revised_Second_Review_Draft.docx'
FIG1 = FIGURES / 'ETF_Test_PR_AUC.png'
FIG2 = FIGURES / 'ETF_Alert_Budget.png'

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10})
test_metrics = pd.DataFrame([r for r in RESULTS['metrics'] if r['Split']=='Test'])
budget = pd.DataFrame([r for r in RESULTS['budget_metrics'] if r['Split']=='Test'])

fig, ax = plt.subplots(figsize=(7.2, 3.5))
colors = ['#A5A5A5','#1F77B4','#4472C4','#70AD47','#ED7D31']
ax.bar(test_metrics['Model'], test_metrics['PR_AUC'], color=colors)
ax.axhline(test_metrics.loc[test_metrics.Model.eq('Historical Prevalence'),'PR_AUC'].iloc[0], color='#7F7F7F', linestyle='--', linewidth=1, label='No-skill PR-AUC')
ax.set_ylabel('PR-AUC')
ax.set_title('Out-of-time test PR-AUC')
ax.set_ylim(0, 0.34)
ax.grid(axis='y', alpha=.25)
ax.tick_params(axis='x', rotation=18)
ax.legend(frameon=False, loc='upper right')
fig.tight_layout()
fig.savefig(FIG1, dpi=220, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(7.2, 3.5))
ax.bar(budget['Model'], budget['Precision_at_Budget']*100, color=['#1F77B4','#4472C4','#70AD47','#ED7D31'])
ax.set_ylabel('Precision among alerts (%)')
ax.set_title('Test performance under a 10% alert budget')
ax.set_ylim(0, 42)
ax.grid(axis='y', alpha=.25)
for i, (p, tp) in enumerate(zip(budget['Precision_at_Budget']*100, budget['True_Positives'])):
    ax.text(i, p+1, f'{p:.1f}%\n{int(tp)} events', ha='center', va='bottom', fontsize=9)
ax.tick_params(axis='x', rotation=12)
fig.tight_layout()
fig.savefig(FIG2, dpi=220, bbox_inches='tight')
plt.close(fig)

doc = Document()
sec = doc.sections[0]
sec.page_width = Inches(8.5)
sec.page_height = Inches(11)
sec.top_margin = Inches(0.72)
sec.bottom_margin = Inches(0.72)
sec.left_margin = Inches(0.78)
sec.right_margin = Inches(0.78)

styles = doc.styles
normal = styles['Normal']
normal.font.name = 'Aptos'
normal.font.size = Pt(10.5)
normal.font.color.rgb = RGBColor(0,0,0)
normal.paragraph_format.space_after = Pt(6)
normal.paragraph_format.line_spacing = 1.08

styles['Title'].font.name = 'Aptos Display'
styles['Title'].font.size = Pt(19)
styles['Title'].font.bold = True
styles['Title'].font.color.rgb = RGBColor(0,0,0)
styles['Title'].paragraph_format.space_after = Pt(10)

for name, size in [('Heading 1',14),('Heading 2',12),('Heading 3',11)]:
    st=styles[name]
    st.font.name='Aptos Display'
    st.font.size=Pt(size)
    st.font.bold=True
    st.font.color.rgb=RGBColor(0,0,0)
    st.paragraph_format.space_before=Pt(10)
    st.paragraph_format.space_after=Pt(5)
    st.paragraph_format.keep_with_next=True

def set_cell_shading(cell, fill):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = tcPr.find(qn('w:shd'))
    if shd is None:
        shd = OxmlElement('w:shd')
        tcPr.append(shd)
    shd.set(qn('w:fill'), fill)

def set_cell_margins(cell, top=90, start=90, bottom=90, end=90):
    tc = cell._tc
    tcPr = tc.get_or_add_tcPr()
    tcMar = tcPr.first_child_found_in('w:tcMar')
    if tcMar is None:
        tcMar = OxmlElement('w:tcMar')
        tcPr.append(tcMar)
    for m, v in [('top',top),('start',start),('bottom',bottom),('end',end)]:
        node = tcMar.find(qn(f'w:{m}'))
        if node is None:
            node=OxmlElement(f'w:{m}')
            tcMar.append(node)
        node.set(qn('w:w'), str(v)); node.set(qn('w:type'),'dxa')

def set_repeat_table_header(row):
    trPr = row._tr.get_or_add_trPr()
    tblHeader = OxmlElement('w:tblHeader')
    tblHeader.set(qn('w:val'),'true')
    trPr.append(tblHeader)

def add_table(headers, rows, widths=None, font_size=9):
    table = doc.add_table(rows=1, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.style = 'Table Grid'
    set_repeat_table_header(table.rows[0])
    for j,h in enumerate(headers):
        cell=table.rows[0].cells[j]
        cell.text=str(h)
        set_cell_shading(cell,'1F4E78')
        cell.vertical_alignment=WD_CELL_VERTICAL_ALIGNMENT.CENTER
        set_cell_margins(cell)
        for p in cell.paragraphs:
            p.alignment=WD_ALIGN_PARAGRAPH.CENTER
            for r in p.runs:
                r.font.name='Aptos'; r.font.size=Pt(font_size); r.font.bold=True; r.font.color.rgb=RGBColor(255,255,255)
    for i,row in enumerate(rows):
        cells=table.add_row().cells
        for j,val in enumerate(row):
            cells[j].text='' if val is None else str(val)
            set_cell_shading(cells[j], 'F3F6FA' if i%2 else 'FFFFFF')
            cells[j].vertical_alignment=WD_CELL_VERTICAL_ALIGNMENT.CENTER
            set_cell_margins(cells[j])
            for p in cells[j].paragraphs:
                p.paragraph_format.space_after=Pt(0)
                p.alignment=WD_ALIGN_PARAGRAPH.LEFT if j==0 else WD_ALIGN_PARAGRAPH.CENTER
                for r in p.runs:
                    r.font.name='Aptos'; r.font.size=Pt(font_size); r.font.color.rgb=RGBColor(0,0,0)
        if widths:
            for j,w in enumerate(widths):
                cells[j].width=Inches(w)
    doc.add_paragraph().paragraph_format.space_after=Pt(1)
    return table

def add_body(text, bold_lead=None):
    p=doc.add_paragraph()
    p.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY
    if bold_lead and text.startswith(bold_lead):
        p.add_run(bold_lead).bold=True
        p.add_run(text[len(bold_lead):])
    else:
        p.add_run(text)
    return p

def add_bullet(text):
    p=doc.add_paragraph(style='List Bullet')
    p.paragraph_format.space_after=Pt(3)
    p.add_run(text)
    return p

def add_caption(text):
    p=doc.add_paragraph()
    p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    p.paragraph_format.keep_with_next=True
    r=p.add_run(text); r.bold=True; r.font.size=Pt(9.5)

def add_page_number(paragraph):
    paragraph.alignment=WD_ALIGN_PARAGRAPH.CENTER
    run=paragraph.add_run()
    fldChar1=OxmlElement('w:fldChar'); fldChar1.set(qn('w:fldCharType'),'begin')
    instr=OxmlElement('w:instrText'); instr.set(qn('xml:space'),'preserve'); instr.text=' PAGE '
    fldChar2=OxmlElement('w:fldChar'); fldChar2.set(qn('w:fldCharType'),'end')
    run._r.extend([fldChar1,instr,fldChar2])

# Title page and abstract
p=doc.add_paragraph(style='Title')
p.alignment=WD_ALIGN_PARAGRAPH.CENTER
p.add_run('An Explainable Machine Learning Early Warning System for Price NAV Dislocations in Indian Equity ETFs')
p2=doc.add_paragraph()
p2.alignment=WD_ALIGN_PARAGRAPH.CENTER
r=p2.add_run('Evidence from NIFTYBEES')
r.font.size=Pt(14); r.bold=True

for line in [
    'Ansh Kotwal B2026006  Email insert official address',
    'Anish Mayya B2026005  Email insert official address',
    'Rohit Agarwal B2026037  Email insert official address',
    'Goa Institute of Management  Big Data Analytics',
    'Advanced Machine Learning for Business Transformation',
    'Second Review Draft  17 November 2026'
]:
    p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    run=p.add_run(line); run.font.size=Pt(10.5)

doc.add_paragraph()
doc.add_heading('Abstract', level=1)
add_body('This study develops and evaluates an explainable next trading day early warning system for abnormal differences between the market closing price and net asset value of Nippon India ETF Nifty 50 BeES. The analysis combines official NSE trading data, AMFI net asset values, NIFTY 50 market controls and India VIX observations from 1 January 2021 through 16 November 2026. After data reconciliation, 1,446 complete daily observations remain. An abnormal event is defined prospectively: the next day absolute price NAV dislocation must exceed the rolling 95th percentile estimated from the 252 complete observations available on the prediction date. The leakage safe modelling sample contains 1,185 forecasts, including 80 abnormal events. Transparent rules, logistic regression, a decision tree and a random forest are compared using chronological validation and an untouched 2026 test period. The current gap rule records the strongest test PR AUC of 0.290, compared with a no skill value of 0.159. Logistic regression is the best probability classifier, with a test PR AUC of 0.267 and Brier score of 0.134. Under a 10 percent alert budget, the current gap rule identifies 8 of 34 test events with 36.4 percent precision. The results show that a simple, auditable signal can outperform more complex classifiers when market conditions change. The system should therefore be interpreted as a risk monitoring aid rather than a trading recommendation.')
p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY
r=p.add_run('Keywords  '); r.bold=True
p.add_run('exchange traded funds, price NAV dislocation, NIFTYBEES, early warning, interpretable machine learning, PR AUC, calibration')

doc.add_page_break()

doc.add_heading('1 Introduction and Business Problem', level=1)
add_body('An exchange traded fund is priced in two connected markets. Its units trade continuously on an exchange, while its net asset value is calculated from the value of the underlying portfolio. Creation and redemption by authorized participants should limit large differences between these two values, but the mechanism is not frictionless. Trading costs, inventory constraints, thin secondary market liquidity, stale component prices and sudden order imbalances can allow the exchange price to trade temporarily above or below NAV.')
add_body('For an investor, a large premium or discount is an execution risk. A buyer may pay substantially more than the value represented by the underlying portfolio, while a seller may accept an unusually low price. For an ETF sponsor, trading desk or market maker, a widening price NAV gap may also signal liquidity stress or delayed arbitrage. The business problem is therefore to identify an elevated risk of abnormal dislocation before the next trading session. The objective is not to forecast general equity returns or promise an arbitrage profit.')
add_body('This project studies NIFTYBEES as a carefully audited pilot. The broader title refers to the intended ETF risk monitoring framework, but the empirical claims in this report apply only to NIFTYBEES. Restricting the first implementation to one scheme avoids combining funds before scheme identity, listing history, benchmark and NAV publication timing are reconciled.')

doc.add_heading('2 Literature Review', level=1)
doc.add_heading('2.1 ETF pricing and the arbitrage mechanism', level=2)
add_body('ETF market prices can deviate from reported NAV even though authorized participants can create or redeem units. Hilliard [8] models premiums and discounts as a mean reverting process with jumps and finds that pricing efficiency differs across asset classes. Petajisto [7] documents economically meaningful pricing bands and shows that larger deviations are associated with international and less liquid holdings. These studies establish that the arbitrage mechanism narrows price differences but does not eliminate them at every point in time.')
add_body('Liquidity is central to the speed and cost of correction. Bae and Kim [9] find that illiquid ETFs have larger tracking errors and that the effect is stronger when underlying assets are also illiquid. Rappoport and Tuzun [10] likewise show that market liquidity affects the efficacy of ETF arbitrage. Pan and Zeng [12] demonstrate how authorized participants balance their arbitrage role against dealer inventory and balance sheet constraints. Although some of this evidence comes from fixed income ETFs, the underlying implication is relevant to equity ETFs: observed premiums and discounts may reflect limits to immediate arbitrage as well as information differences.')

doc.add_heading('2.2 Evidence from India', level=2)
add_body('Indian studies provide direct evidence that price NAV deviations deserve separate analysis. Reddy and Dhabolkar [5] examine pricing efficiency using autoregressive and vector error correction methods and report persistent deviations and lead lag relationships between NAV and market prices. Goel and Ahluwalia [6] study 24 Indian equity ETFs and find that premiums, discounts and tracking errors remain material despite apparently low arbitrage constraints. They identify bid ask spreads, NAV momentum, market capitalization and operational constraints as relevant to pricing efficiency. This literature supports the use of liquidity, trading activity, underlying market movement and volatility as predictors.')
add_body('The existing Indian literature is mainly explanatory or contemporaneous. It measures tracking error, tests long run relationships, estimates the speed of correction or studies fund characteristics. These are important questions, but they do not directly answer whether information available after the current day NAV publication can warn of an abnormal dislocation on the next actual trading day.')

doc.add_heading('2.3 Volatility, stress and interpretability', level=2)
add_body('ETF arbitrage can transmit shocks between the fund and its underlying securities. Ben David, Franzoni and Moussawi [11] find that greater ETF ownership is associated with higher volatility and short horizon price reversals in underlying stocks. This motivates the inclusion of India VIX and NIFTY 50 movements as market stress controls rather than treating the ETF in isolation.')
add_body('The prediction task is imbalanced because abnormal dislocations are intentionally rare. Saito and Rehmsmeier [13] show why precision recall analysis is more informative than ROC analysis when positive observations are scarce. Probability quality also matters because an early warning system communicates risk rather than only a class label. Guo et al. [14] emphasize that predictive accuracy and probability calibration are different properties. The project therefore reports PR AUC, Brier score, calibration diagnostics and performance under a fixed alert budget. It also retains simple rules and logistic regression because transparent models are easier to audit when predictions influence financial decisions [15].')

doc.add_heading('2.4 Research gap and contribution', level=2)
add_body('Among the studies reviewed, we did not identify an Indian equity ETF study that combines a post NAV prediction timestamp, an ETF specific rolling event definition, purged chronological evaluation, probability calibration and a constrained alert budget. This project addresses that methodological gap through a single ETF pilot. Its contribution is a leakage controlled early warning design and an empirical comparison of transparent rules with classifiers. It does not claim that NIFTYBEES results generalize automatically to all Indian ETFs.')

add_caption('Table 1  Position of the study relative to selected literature')
add_table(
    ['Study','Market and method','Main relevance','Difference from this project'],
    [
        ['Reddy and Dhabolkar 2020','Indian ETFs; autoregression and VECM','Price NAV persistence and lead lag behaviour','Explains pricing efficiency; does not predict next day abnormal events'],
        ['Goel and Ahluwalia 2021','24 Indian equity ETFs; efficiency and fund characteristics','Indian discounts, tracking error and operational constraints','Cross fund explanatory analysis rather than post NAV warning'],
        ['Petajisto 2017','Multiple ETF asset classes; pricing bands','Mispricing persists despite arbitrage','Focuses pricing inefficiency and mean reversion'],
        ['Bae and Kim 2020','US ETFs; liquidity and tracking error','Illiquidity increases deviations and risk','Does not construct an event classifier'],
        ['Rappoport and Tuzun 2020','Equity and bond ETF panel; arbitrage and liquidity','Liquidity affects arbitrage efficacy','Studies mechanism rather than operational warning'],
        ['Current study','NIFTYBEES; rolling label and chronological classifiers','Next day warning, calibration and alert budget','Single fund pilot with leakage safe implementation'],
    ],
    widths=[1.4,1.8,2.0,2.1], font_size=8.3
)

doc.add_heading('3 Research Question and Objectives', level=1)
add_body('Research question: Can information available after the current trading day NAV publication predict whether NIFTYBEES will experience an abnormally large price NAV dislocation on the next actual NSE trading day?')
for item in [
    'Construct a reproducible daily panel using official market price, NAV, benchmark and volatility data.',
    'Define abnormal dislocations relative to the ETF own trailing history without using future information.',
    'Compare a simple alert rule with logistic regression, decision tree and random forest classifiers.',
    'Evaluate ranking, probability calibration and operational usefulness under a limited alert budget.',
    'Explain the main signals and document limitations that affect interpretation and deployment.'
]: add_bullet(item)

doc.add_heading('4 Data and Data Quality', level=1)
add_body('The observation unit is one NIFTYBEES trading date. Market price and liquidity fields come from official NSE security level reports. NAV observations use the AMFI historical NAV service for scheme code 140084, verified against the scheme name and ISIN. NIFTY 50 historical index data provide underlying market controls, and official NSE India VIX data provide the market volatility control. The study period runs from 1 January 2021 through 16 November 2026.')
add_caption('Table 2  Final data coverage')
add_table(
    ['Dataset','Rows','Coverage','Use'],
    [
        ['NIFTYBEES market data','1,456','1 Jan 2021 to 16 Nov 2026','Prices and liquidity'],
        ['Official AMFI NAV','1,451','1 Jan 2021 to 16 Nov 2026','Price NAV dislocation'],
        ['NIFTY 50','1,456','1 Jan 2021 to 16 Nov 2026','Underlying market control'],
        ['India VIX','1,455','1 Jan 2021 to 16 Nov 2026','Volatility control'],
        ['Complete case panel','1,446','1 Jan 2021 to 16 Nov 2026','Feature and label construction'],
    ], widths=[1.9,0.8,2.0,2.1], font_size=8.8
)
add_body('Eight special or Muhurat market sessions have exchange trading records but no corresponding AMFI NAV and are excluded when the next actual market session lacks a complete target. Three AMFI NAV reporting dates occur without an NSE market session. Two India VIX source zero anomalies, dated 12 February 2021 and 30 March 2021, are excluded. Extreme ETF price NAV differences were retained only after checking their source values; the largest absolute dislocation is 3.010 percent on 4 June 2024.')

doc.add_heading('5 Methodology', level=1)
doc.add_heading('5.1 Prediction timestamp and dislocation', level=2)
add_body('The forecast is made after AMFI publishes the NAV for trading date t. SEBI disclosure requirements specify that NAV is normally made available on AMFI and mutual fund websites by 11 p.m. on the business day [4]. At that point the ETF closing price, trading activity, NIFTY 50 close, India VIX and the date t NAV are available. The prediction concerns the next actual NSE trading date.')
p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
r=p.add_run('d_t = 100 x (ETF closing price_t - NAV_t) / NAV_t'); r.font.name='Cambria Math'; r.font.size=Pt(11); r.bold=True
add_body('The signed value identifies a premium or discount, while the absolute value measures the size of the dislocation regardless of direction.')

doc.add_heading('5.2 Forward label and leakage controls', level=2)
p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
r=p.add_run('y_(t+1) = 1 when |d_(t+1)| > rolling 95th percentile of the 252 complete observations ending on t'); r.font.name='Cambria Math'; r.font.size=Pt(10.5); r.bold=True
add_body('The target date cannot influence its own threshold. A row enters the modelling dataset only when date t is complete, the immediately following market session is complete, all selected predictors are available and 252 complete historical observations exist. Samples are assigned by target date. One overlapping label interval is purged at each train validation and validation test boundary. The final dataset contains 1,185 leakage safe observations.')

doc.add_heading('5.3 Predictors and models', level=2)
add_body('The predictor set contains 25 variables available by date t: current and lagged signed dislocations, current absolute dislocation, rolling five and 20 day dislocation summaries, ETF and NAV returns, NIFTY 50 returns, ETF minus index return differences, India VIX level and changes, volume, turnover, average trade value, delivery percentage and intraday range. Log transformations are applied to volume and turnover. The current gap rule uses the ratio of the current absolute dislocation to the rolling 95th percentile threshold as a transparent continuous risk score.')
add_body('Three classifiers are estimated: L2 regularized logistic regression, a bounded decision tree and a random forest. Hyperparameters are selected using validation PR AUC only. The models are trained on the training sample and remain locked for the test evaluation. The test labels are not used for feature selection, model selection or alert threshold selection.')

add_caption('Table 3  Chronological design')
add_table(
    ['Partition','Target period','Observations','Abnormal events','Event rate'],
    [
        ['Training','12 Jan 2022 to 30 Dec 2024','725','41','5.66%'],
        ['Validation','1 Jan 2025 to 30 Dec 2025','246','5','2.03%'],
        ['Testing','1 Jan 2026 to 16 Nov 2026','214','34','15.89%'],
        ['Total','12 Jan 2022 to 16 Nov 2026','1,185','80','6.75%'],
    ], widths=[1.2,2.2,1.0,1.1,1.0], font_size=8.8
)

doc.add_heading('5.4 Evaluation', level=2)
add_body('PR AUC is the primary ranking metric because the positive class is uncommon and the practical question is whether high risk scores concentrate genuine events. ROC AUC is reported as a secondary discrimination measure. Brier score, log loss, calibration intercept, calibration slope and five bin expected calibration error assess probability quality. Operational usefulness is evaluated by allowing alerts on only the highest scored 10 percent of dates. A second check freezes the validation score threshold and applies it unchanged to testing, revealing whether score distributions and alert volumes drift.')

doc.add_heading('6 Results', level=1)
doc.add_heading('6.1 Out of time model comparison', level=2)
add_caption('Table 4  Test performance')
add_table(
    ['Model','PR AUC','ROC AUC','Lift over no skill','Brier score','ECE'],
    [
        ['Historical prevalence','0.159','0.500','0.0%','0.144','0.102'],
        ['Current gap rule','0.290','0.642','82.8%','Not applicable','Not applicable'],
        ['Logistic regression','0.267','0.543','68.4%','0.134','0.079'],
        ['Decision tree','0.206','0.605','29.8%','0.152','0.094'],
        ['Random forest','0.202','0.599','27.1%','0.162','0.118'],
    ], widths=[1.6,0.8,0.8,1.1,1.0,0.8], font_size=8.7
)
add_body('Random forest achieved the highest validation PR AUC of 0.276 and was therefore the model selected under the predeclared rule. It did not retain that advantage in the untouched test period. The current gap rule achieved the highest test PR AUC of 0.290, followed by logistic regression at 0.267. Both substantially exceed the test event prevalence of 0.159, which is the no skill PR AUC. The finding does not justify replacing the validation selection rule after observing test outcomes. It instead shows that a simple signal generalized better under the 2026 conditions.')
add_caption('Figure 1  Model ranking in the untouched test period')
p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
p.add_run().add_picture(str(FIG1), width=Inches(6.7))

doc.add_heading('6.2 Alert budget performance', level=2)
add_caption('Table 5  Test alerts under a 10 percent budget')
add_table(
    ['Model','Alerts','True events identified','Precision','Recall'],
    [
        ['Current gap rule','22','8','36.4%','23.5%'],
        ['Logistic regression','22','7','31.8%','20.6%'],
        ['Decision tree','22','5','22.7%','14.7%'],
        ['Random forest','22','4','18.2%','11.8%'],
    ], widths=[1.9,0.8,1.5,1.0,1.0], font_size=8.8
)
add_body('The current gap rule identifies eight of the 34 abnormal test events while issuing 22 alerts. Its alert precision of 36.4 percent is more than twice the unconditional test event rate. Logistic regression identifies seven events with 31.8 percent precision. The result supports an operational design that ranks dates and limits review capacity instead of using a universal probability cutoff.')
add_caption('Figure 2  Precision when only 10 percent of test dates can be flagged')
p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
p.add_run().add_picture(str(FIG2), width=Inches(6.7))

doc.add_heading('6.3 Calibration and model explanations', level=2)
add_body('Logistic regression is the strongest probability classifier in testing. Its Brier score of 0.134 improves on the historical prevalence benchmark of 0.144, and its mean predicted probability of 15.12 percent is close to the actual test event rate of 15.89 percent. Its calibration slope of 0.355 nevertheless indicates that cross sectional probability differences are too extreme or unstable. Random forest predicts an average probability of 24.78 percent and has a worse Brier score of 0.162, showing overprediction during the shifted test regime.')
add_body('For logistic regression, the largest standardized coefficients are associated with ETF turnover, India VIX level and change, delivery percentage and recent dislocation values. Random forest gives its highest importance to the current gap to threshold ratio and the previous day signed dislocation. These patterns are consistent with persistence and liquidity channels discussed in the literature, but they are predictive associations rather than causal estimates. Correlated predictors also mean that individual coefficients and feature importances should not be interpreted in isolation.')

doc.add_heading('7 Discussion', level=1)
add_body('The most important empirical result is that added model complexity did not produce the best out of time warning performance. Random forest fit the sparse 2025 validation events well enough to win the selection stage, but the transparent current gap rule ranked 2026 outcomes more effectively. This result supports the project emphasis on interpretable baselines. A model comparison is informative only when the simple rule is treated as a genuine competitor rather than a weak formality.')
add_body('The shift in event frequency is equally important. Abnormal events represent 2.03 percent of validation dates and 15.89 percent of test dates. A score threshold calibrated to issue about 10 percent alerts in validation flags 39.3 percent of test dates for the current gap rule, 22.9 percent for logistic regression and 33.6 percent for random forest. The score distributions therefore changed materially. A live system should monitor event prevalence, probability calibration and alert volume and should recalibrate or retrain only under a documented schedule.')
add_body('The results also clarify how the system should be presented to a business user. It is a screening mechanism that prioritizes dates for review. A high score does not establish that an arbitrage is executable, because transaction costs, bid ask spreads, market impact, creation unit requirements and the timing of NAV publication are outside the current target. The warning can help an investor pause, compare current price with NAV or iNAV, inspect liquidity and use a limit order. It should not automatically trigger a trade.')

doc.add_heading('8 Managerial and Analytical Implications', level=1)
for item in [
    'Use the current gap ratio as the first line ranking signal because it is transparent and performed best in the test period.',
    'Use logistic regression when a probability estimate is required, but display calibration and drift warnings beside the probability.',
    'Cap alerts by review capacity, such as the highest risk 10 percent of dates, rather than relying only on a fixed score threshold.',
    'Recheck source timing and extreme values before escalating an alert, especially on special trading sessions and volatile market days.',
    'Treat feature importance as an explanation of model behaviour, not evidence that turnover, volatility or delivery activity causes dislocation.'
]: add_bullet(item)

doc.add_heading('9 Limitations and Future Work', level=1)
add_body('The study uses one highly traded broad market ETF. The result may not extend to sectoral, thematic, international, gold, debt or lower liquidity ETFs. A multi ETF panel should be added only after scheme codes, ISINs, benchmark mappings, listing histories, corporate actions and NAV publication schedules are reconciled for every fund.')
add_body('Daily closing price and end of day NAV may not capture intraday arbitrage dynamics. Future work could incorporate bid ask spreads, order book measures, iNAV, authorized participant flows and creation redemption activity where reliable data are available. The event definition is relative to the ETF recent history and is therefore suitable for monitoring, but sensitivity checks should compare the 95th percentile label with fixed economic thresholds such as 0.50 percent and 1.00 percent.')
add_body('Only five validation events are available, making hyperparameter choice and calibration estimates uncertain. Future evaluation should use walk forward folds across additional ETFs and market regimes. Statistical uncertainty should be reported through confidence intervals or bootstrap procedures that respect time dependence. Transaction costs and executable trading returns are outside this study and must be analysed separately before any investment interpretation.')

doc.add_heading('10 Conclusion', level=1)
add_body('This project converts an ETF pricing efficiency question into a leakage safe next trading day warning task. Official market, NAV, benchmark and volatility data support 1,446 complete NIFTYBEES observations and 1,185 eligible forecasts after the historical lookback and boundary purging rules are applied. The current gap rule produces the strongest test ranking performance, and logistic regression provides the most useful probability estimates among the classifiers. Decision tree and random forest add comparison evidence but do not improve the final out of time result.')
add_body('The central conclusion is practical: an explainable monitoring system is feasible, but stable alert capacity matters more than choosing the most complex model. The system should rank risk, limit alerts, expose its drivers and monitor calibration as market conditions change. The evidence supports a NIFTYBEES pilot and a carefully controlled expansion to other Indian ETFs; it does not support automated trading or general claims about the entire ETF market.')

doc.add_heading('11 Reproducibility', level=1)
add_body('Repository URL: insert public or institution accessible GitHub repository URL before submission.')
add_body('The repository should contain a README with the exact execution order, Python environment and file provenance. Raw downloaded files should remain unchanged in a source directory. Processing scripts should generate the cleaned market, NAV, NIFTY 50 and India VIX files, followed by the merged master panel, analysis ready panel, leakage safe modelling dataset and model comparison outputs.')
for item in [
    'data raw: original NSE, AMFI, NIFTY 50 and India VIX downloads.',
    'data processed: cleaned source files, merged panel and modelling dataset.',
    'scripts: data audit, feature and label construction, model comparison and report figure generation.',
    'outputs: audit workbooks, model metrics, row level predictions and figures.',
    'environment file: pinned versions of Python, pandas, NumPy, scikit learn and plotting packages.',
    'README: commands, expected row counts, split dates, random seed 42 and descriptions of every output.'
]: add_bullet(item)
add_body('Reproduction checks should confirm 1,456 market rows, 1,451 AMFI NAV rows, 1,446 complete cases, 1,185 modelling observations and 80 abnormal events. Model training uses random seed 42. The test set must remain isolated until all model and threshold choices are frozen.')

doc.add_heading('12 Artificial Intelligence Use Disclosure', level=1)
add_body('Generative artificial intelligence tools were used to assist literature discovery, code drafting, language editing and document formatting. The student authors selected the research design, reviewed the cited sources, verified the official data, executed and checked the analysis, and remain responsible for all interpretations and errors. AI generated text and code were not accepted without human review. No confidential personal data were provided to the tools.')

doc.add_heading('References', level=1)
refs = [
    '[1] National Stock Exchange of India, Security wise price, volume and deliverable position data, https://www.nseindia.com/report-detail/eq_security.',
    '[2] Association of Mutual Funds in India, Historical NAV download, https://www.amfiindia.com/net-asset-value/nav-download.',
    '[3] NSE Indices, Historical index data, https://www.niftyindices.com/reports.',
    '[4] Securities and Exchange Board of India, Master Circular for Mutual Funds, NAV disclosure requirements, 2026.',
    '[5] Y. V. Reddy and P. Dhabolkar, Pricing Efficiency of Exchange Traded Funds in India, Organizations and Markets in Emerging Economies, vol. 11, no. 1, pp. 244 to 268, 2020, doi:10.15388/omee.2020.11.33.',
    '[6] G. Goel and E. Ahluwalia, Do pricing efficiencies in Indian equity ETF market impact its performance, Global Finance Journal, vol. 49, art. 100654, 2021, doi:10.1016/j.gfj.2021.100654.',
    '[7] A. Petajisto, Inefficiencies in the Pricing of Exchange Traded Funds, Financial Analysts Journal, vol. 73, no. 1, pp. 24 to 54, 2017, doi:10.2469/faj.v73.n1.7.',
    '[8] J. Hilliard, Premiums and discounts in ETFs: An analysis of the arbitrage mechanism in domestic and international funds, Global Finance Journal, vol. 25, no. 2, pp. 90 to 107, 2014.',
    '[9] K. Bae and D. Kim, Liquidity risk and exchange traded fund returns, variances, and tracking errors, Journal of Financial Economics, vol. 138, no. 1, pp. 222 to 253, 2020, doi:10.1016/j.jfineco.2019.02.012.',
    '[10] D. E. Rappoport W. and T. Tuzun, Arbitrage and Liquidity: Evidence from a Panel of Exchange Traded Funds, Finance and Economics Discussion Series 2020 097, Federal Reserve Board, 2020, doi:10.17016/FEDS.2020.097.',
    '[11] I. Ben David, F. Franzoni, and R. Moussawi, Do ETFs Increase Volatility, Journal of Finance, vol. 73, no. 6, pp. 2471 to 2535, 2018, doi:10.1111/jofi.12727.',
    '[12] K. Pan and Y. Zeng, ETF Arbitrage Under Liquidity Mismatch, SSRN Working Paper 2895478, 2019, doi:10.2139/ssrn.2895478.',
    '[13] T. Saito and M. Rehmsmeier, The Precision Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets, PLOS ONE, vol. 10, no. 3, e0118432, 2015, doi:10.1371/journal.pone.0118432.',
    '[14] C. Guo, G. Pleiss, Y. Sun, and K. Q. Weinberger, On Calibration of Modern Neural Networks, Proceedings of the 34th International Conference on Machine Learning, vol. 70, pp. 1321 to 1330, 2017.',
    '[15] C. Rudin, Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead, Nature Machine Intelligence, vol. 1, pp. 206 to 215, 2019, doi:10.1038/s42256-019-0048-x.'
]
for ref in refs:
    p=doc.add_paragraph()
    p.paragraph_format.left_indent=Inches(0.2)
    p.paragraph_format.first_line_indent=Inches(-0.2)
    p.paragraph_format.space_after=Pt(3)
    r=p.add_run(ref); r.font.size=Pt(9)

for section in doc.sections:
    add_page_number(section.footer.paragraphs[0])

doc.core_properties.title = 'ETF Project Revised Second Review Draft'
doc.core_properties.subject = 'NIFTYBEES price NAV dislocation early warning system'
doc.core_properties.author = 'Ansh Kotwal, Anish Mayya, Rohit Agarwal'

OUT.mkdir(parents=True, exist_ok=True)
doc.save(DOCX)
print(DOCX)


/workspace/scratch/e2d669a31e97/Niftybees-Price-NAV-Early-Warning/report/ETF_Project_Revised_Second_Review_Draft.docx
Completed in 0.9 seconds.


## Step 8D — Build the final integrated Word report

Original repository script: `scripts/13_build_final_integrated_report.py`


In [28]:
"""Build the integrated final research report from verified project outputs."""
from __future__ import annotations

from pathlib import Path
from copy import deepcopy

import matplotlib.pyplot as plt
import pandas as pd
from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.table import WD_ALIGN_VERTICAL, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK, WD_LINE_SPACING
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor


ROOT = PROJECT_ROOT
OUT = ROOT / "outputs"
FIG = ROOT / "figures"
REPORT = ROOT / "report"
REPORT.mkdir(exist_ok=True)
TMP = ROOT / "outputs" / "final_report_assets"
TMP.mkdir(exist_ok=True)
DEST = REPORT / "ETF_Project_Final_Integrated_Draft.docx"


def set_repeat_table_header(row):
    tr_pr = row._tr.get_or_add_trPr()
    tbl_header = OxmlElement("w:tblHeader")
    tbl_header.set(qn("w:val"), "true")
    tr_pr.append(tbl_header)


def set_cell_shading(cell, fill):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd = tc_pr.find(qn("w:shd"))
    if shd is None:
        shd = OxmlElement("w:shd"); tc_pr.append(shd)
    shd.set(qn("w:fill"), fill)


def set_cell_margins(cell, top=80, start=90, bottom=80, end=90):
    tc = cell._tc; tc_pr = tc.get_or_add_tcPr()
    tc_mar = tc_pr.first_child_found_in("w:tcMar")
    if tc_mar is None:
        tc_mar = OxmlElement("w:tcMar"); tc_pr.append(tc_mar)
    for m, v in (("top", top), ("start", start), ("bottom", bottom), ("end", end)):
        node = tc_mar.find(qn(f"w:{m}"))
        if node is None:
            node = OxmlElement(f"w:{m}"); tc_mar.append(node)
        node.set(qn("w:w"), str(v)); node.set(qn("w:type"), "dxa")


def set_cell_width(cell, width_inches):
    tc_pr = cell._tc.get_or_add_tcPr()
    tc_w = tc_pr.find(qn("w:tcW"))
    if tc_w is None:
        tc_w = OxmlElement("w:tcW"); tc_pr.append(tc_w)
    tc_w.set(qn("w:w"), str(int(width_inches * 1440))); tc_w.set(qn("w:type"), "dxa")


def set_table_borders(table, color="D9D9D9", size="5"):
    tbl_pr = table._tbl.tblPr
    borders = tbl_pr.find(qn("w:tblBorders"))
    if borders is None:
        borders = OxmlElement("w:tblBorders"); tbl_pr.append(borders)
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        node = borders.find(qn(f"w:{edge}"))
        if node is None:
            node = OxmlElement(f"w:{edge}"); borders.append(node)
        node.set(qn("w:val"), "single"); node.set(qn("w:sz"), size)
        node.set(qn("w:space"), "0"); node.set(qn("w:color"), color)


def add_page_number(paragraph):
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = paragraph.add_run()
    fld_begin = OxmlElement("w:fldChar"); fld_begin.set(qn("w:fldCharType"), "begin")
    instr = OxmlElement("w:instrText"); instr.set(qn("xml:space"), "preserve"); instr.text = " PAGE "
    fld_end = OxmlElement("w:fldChar"); fld_end.set(qn("w:fldCharType"), "end")
    run._r.extend([fld_begin, instr, fld_end])


def add_hyperlink(paragraph, text, url):
    part = paragraph.part
    r_id = part.relate_to(url, "http://schemas.openxmlformats.org/officeDocument/2006/relationships/hyperlink", is_external=True)
    hyperlink = OxmlElement("w:hyperlink"); hyperlink.set(qn("r:id"), r_id)
    run = OxmlElement("w:r"); r_pr = OxmlElement("w:rPr")
    color = OxmlElement("w:color"); color.set(qn("w:val"), "000000")
    underline = OxmlElement("w:u"); underline.set(qn("w:val"), "single")
    r_pr.extend([color, underline]); run.append(r_pr)
    t = OxmlElement("w:t"); t.text = text; run.append(t); hyperlink.append(run)
    paragraph._p.append(hyperlink)


def add_body(doc, text, bold_lead=None):
    p = doc.add_paragraph(style="Body Text")
    if bold_lead and text.startswith(bold_lead):
        p.add_run(bold_lead).bold = True; p.add_run(text[len(bold_lead):])
    else:
        p.add_run(text)
    return p


def add_bullet(doc, text):
    return doc.add_paragraph(text, style="List Bullet")


def add_caption(doc, text):
    p = doc.add_paragraph(text, style="Caption")
    p.paragraph_format.keep_with_next = True
    return p


def add_table(doc, headers, rows, widths, font_size=8.5, aligns=None):
    table = doc.add_table(rows=1, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER; table.autofit = False
    set_table_borders(table); set_repeat_table_header(table.rows[0])
    hdr = table.rows[0]
    for i, h in enumerate(headers):
        cell = hdr.cells[i]; set_cell_width(cell, widths[i]); set_cell_shading(cell, "1F4E78")
        set_cell_margins(cell); cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
        p = cell.paragraphs[0]; p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r = p.add_run(str(h)); r.bold = True; r.font.color.rgb = RGBColor(255,255,255); r.font.size = Pt(font_size)
    for ridx, row in enumerate(rows):
        cells = table.add_row().cells
        for i, value in enumerate(row):
            set_cell_width(cells[i], widths[i]); set_cell_margins(cells[i]); cells[i].vertical_alignment = WD_ALIGN_VERTICAL.CENTER
            if ridx % 2 == 1: set_cell_shading(cells[i], "F3F6FA")
            p = cells[i].paragraphs[0]
            p.alignment = aligns[i] if aligns else (WD_ALIGN_PARAGRAPH.LEFT if i == 0 else WD_ALIGN_PARAGRAPH.CENTER)
            r = p.add_run(str(value)); r.font.size = Pt(font_size)
    doc.add_paragraph().paragraph_format.space_after = Pt(0)
    return table


def add_figure(doc, path, caption, width=6.45, note=None):
    add_caption(doc, caption)
    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run().add_picture(str(path), width=Inches(width))
    p.paragraph_format.space_after = Pt(2)
    if note:
        n = doc.add_paragraph(note, style="Source Note"); n.paragraph_format.space_after = Pt(8)


def add_reference(doc, number, text, url=None):
    p = doc.add_paragraph(style="Reference")
    p.add_run(f"[{number}] {text}")
    if url:
        p.add_run(" "); add_hyperlink(p, url, url)


def equation_png(text, output, width=10):
    fig, ax = plt.subplots(figsize=(width, .75)); ax.axis("off")
    ax.text(.5, .5, text, ha="center", va="center", fontsize=17)
    fig.savefig(output, dpi=240, transparent=True, bbox_inches="tight", pad_inches=.05); plt.close(fig)


def page_break(doc):
    doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)


doc = Document()
sec = doc.sections[0]
sec.page_width, sec.page_height = Inches(8.5), Inches(11)
sec.top_margin = sec.bottom_margin = Inches(.68)
sec.left_margin = sec.right_margin = Inches(.78)
add_page_number(sec.footer.paragraphs[0])

styles = doc.styles
normal = styles["Normal"]; normal.font.name = "Times New Roman"; normal.font.size = Pt(10.5); normal.font.color.rgb = RGBColor(0,0,0)
normal.paragraph_format.space_after = Pt(5); normal.paragraph_format.line_spacing = 1.13
for name, size, before, after in [("Title",21,0,12),("Heading 1",15,12,6),("Heading 2",12.5,9,4),("Heading 3",11,7,3)]:
    st=styles[name];st.font.name="Arial";st.font.size=Pt(size);st.font.bold=True;st.font.color.rgb=RGBColor(0,0,0)
    st.paragraph_format.space_before=Pt(before);st.paragraph_format.space_after=Pt(after);st.paragraph_format.keep_with_next=True
styles["Body Text"].font.name="Times New Roman";styles["Body Text"].font.size=Pt(10.5)
styles["Body Text"].paragraph_format.first_line_indent=Inches(.25);styles["Body Text"].paragraph_format.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY
styles["Body Text"].paragraph_format.line_spacing=1.13;styles["Body Text"].paragraph_format.space_after=Pt(5)
styles["List Bullet"].font.name="Times New Roman";styles["List Bullet"].font.size=Pt(10.5);styles["List Bullet"].paragraph_format.space_after=Pt(3)
styles["Caption"].font.name="Arial";styles["Caption"].font.size=Pt(9);styles["Caption"].font.bold=True;styles["Caption"].font.color.rgb=RGBColor(0,0,0)
styles["Caption"].paragraph_format.alignment=WD_ALIGN_PARAGRAPH.CENTER;styles["Caption"].paragraph_format.space_before=Pt(7);styles["Caption"].paragraph_format.space_after=Pt(3)
for custom in ["Source Note","Reference"]:
    if custom not in styles: styles.add_style(custom, 1)
styles["Source Note"].font.name="Times New Roman";styles["Source Note"].font.size=Pt(8.5);styles["Source Note"].font.italic=True
styles["Source Note"].paragraph_format.alignment=WD_ALIGN_PARAGRAPH.CENTER;styles["Source Note"].paragraph_format.space_after=Pt(7)
styles["Reference"].font.name="Times New Roman";styles["Reference"].font.size=Pt(9.2);styles["Reference"].paragraph_format.left_indent=Inches(.25)
styles["Reference"].paragraph_format.first_line_indent=Inches(-.25);styles["Reference"].paragraph_format.space_after=Pt(3)

# Cover page
p=doc.add_paragraph(style="Title");p.alignment=WD_ALIGN_PARAGRAPH.CENTER
p.add_run("Does Model Complexity Improve Next Day Price NAV Dislocation Warnings")
sub=doc.add_paragraph();sub.alignment=WD_ALIGN_PARAGRAPH.CENTER
r=sub.add_run("A Leakage Controlled Walk Forward Study of NIFTYBEES");r.bold=True;r.font.name="Arial";r.font.size=Pt(16)
doc.add_paragraph()
for line in ["Ansh Kotwal  B2026006","Anish Mayya  B2026005","Rohit Agarwal  B2026037"]:
    p=doc.add_paragraph(line);p.alignment=WD_ALIGN_PARAGRAPH.CENTER;p.runs[0].font.size=Pt(12)
doc.add_paragraph()
for line in ["Goa Institute of Management","PGDM Big Data Analytics","Advanced Machine Learning for Business Transformation"]:
    p=doc.add_paragraph(line);p.alignment=WD_ALIGN_PARAGRAPH.CENTER;p.runs[0].font.size=Pt(11.5)
doc.add_paragraph()
p=doc.add_paragraph("Final Integrated Draft  18 November 2026");p.alignment=WD_ALIGN_PARAGRAPH.CENTER;p.runs[0].bold=True
doc.add_paragraph()
p=doc.add_paragraph("Study period  1 January 2021 to 16 November 2026");p.alignment=WD_ALIGN_PARAGRAPH.CENTER
page_break(doc)

doc.add_heading("Abstract", level=1)
add_body(doc,"This study evaluates whether complex machine learning models provide more reliable next trading day warnings of abnormal NIFTYBEES price NAV dislocations than transparent benchmark rules. Official NSE market data, AMFI net asset values, NIFTY 50 observations and India VIX data are reconciled for 1 January 2021 through 16 November 2026. The final merged panel contains 1,446 complete observations. A prospective abnormal event occurs when the next trading day's absolute dislocation exceeds the rolling 95th percentile estimated from the 252 complete observations available on the prediction date. The leakage controlled modelling sample contains 1,185 forecasts and 80 events. Three purged expanding window evaluations compare historical prevalence, current gap rules, previous event persistence, logistic regression, a decision tree and a random forest. The current gap to threshold ratio achieves the highest mean walk forward PR AUC of 0.233, compared with 0.169 for logistic regression, 0.156 for the decision tree and 0.184 for random forest. A paired moving block bootstrap establishes a 0.078 advantage over the decision tree, while intervals against logistic regression and random forest include zero. Market stress variables improve logistic regression from 0.153 to 0.195, but liquidity variables do not generate consistent gains. The gap ratio has the highest pooled precision at 5, 10 and 20 percent alert budgets. Event prevalence changes from 10.7 percent in 2024 to 2.0 percent in 2025 and 15.9 percent in 2026, producing substantial calibration and alert volume instability. The evidence supports a transparent capacity constrained monitoring system with continuing drift checks. It does not support automatic trading or universal superiority of one model.")
p=doc.add_paragraph();p.add_run("Keywords  ").bold=True;p.add_run("exchange traded funds, price NAV dislocation, NIFTYBEES, early warning, walk forward validation, interpretable machine learning, calibration, distribution shift")

doc.add_heading("Executive Summary", level=1)
add_body(doc,"The project began as an ETF pricing analysis and has been reframed as a prospective risk screening problem. After the NAV for trading date t becomes available, the system ranks the risk that the next actual NSE trading session will exhibit an unusually large price NAV dislocation. The design prevents the next day outcome from entering the event threshold, feature construction, model tuning or alert cutoff selection.")
add_bullet(doc,"The transparent current gap ratio records the highest average walk forward PR AUC, 0.233, and wins two of three test years.")
add_bullet(doc,"The uncertainty analysis supports superiority over the decision tree, but does not establish superiority over logistic regression or random forest at the 95 percent level.")
add_bullet(doc,"Market stress information improves logistic regression in every test fold; adding liquidity information or all available predictors does not reliably improve results.")
add_bullet(doc,"At a 10 percent review capacity, the gap ratio finds 19 of 65 pooled test events with 26.4 percent precision and 29.2 percent recall.")
add_bullet(doc,"Fixed score cutoffs fail under changing conditions. A validation cutoff intended for 10 percent alerts produces 0.0 to 5.3 percent alerts in 2025 and 22.9 to 39.3 percent in 2026.")
add_body(doc,"The recommended implementation is therefore a transparent ranking system that reviews a fixed percentage of dates, displays probability calibration when a classifier is used and monitors event prevalence, predictor shift, score shift and alert volume.")
page_break(doc)

doc.add_heading("1 Introduction and Business Problem", level=1)
add_body(doc,"An exchange traded fund is priced in two connected settings. Units trade continuously on an exchange, while net asset value is calculated from the value of the underlying portfolio. Creation and redemption by authorized participants should constrain large price differences, but trading costs, inventory limits, liquidity conditions, asynchronous prices and order imbalances can prevent immediate correction [8-13]. A premium can cause a buyer to pay more than the portfolio value represented by the unit; a discount can expose a seller to an unusually low execution price.")
add_body(doc,"This study asks a decision focused question: after the current day's official NAV becomes available, which dates deserve review for the risk of an abnormal price NAV dislocation on the next actual trading session? The objective is risk monitoring. It does not forecast general equity returns, prove an executable arbitrage strategy or recommend a trade.")
add_body(doc,"NIFTYBEES is used as a carefully audited pilot. It is a broad market Indian equity ETF with sufficient history to construct rolling thresholds and multiple chronological tests. All empirical claims therefore apply directly to NIFTYBEES. Expansion to other ETFs requires separate reconciliation of scheme identity, listing history, benchmark, corporate actions and NAV publication timing.")

doc.add_heading("2 Literature Review and Research Gap", level=1)
doc.add_heading("2.1 ETF pricing and limits to arbitrage", level=2)
add_body(doc,"Hilliard [9] models ETF premiums and discounts as a mean reverting process with jumps, while Petajisto [8] documents economically meaningful pricing bands that are wider for international and illiquid assets. Bae and Kim [10] show that ETF liquidity affects tracking error and volatility. Rappoport and Tuzun [11] find that market liquidity changes the efficacy of ETF arbitrage, and Pan and Zeng [13] explain how liquidity mismatch and dealer inventory can restrict arbitrage capacity. Together, these studies imply that current dislocation, recent dislocation history, market stress and trading activity are plausible predictive signals.")
doc.add_heading("2.2 Evidence from Indian ETFs", level=2)
add_body(doc,"Reddy and Dhabolkar [6] use autoregression and vector error correction methods to study Indian ETF price NAV deviations and lead lag relationships. Goel and Ahluwalia [7] examine 24 Indian equity ETFs and report meaningful discounts and tracking errors despite apparently low arbitrage constraints. Their evidence motivates direct analysis of Indian pricing efficiency, but it is primarily explanatory or contemporaneous. It does not evaluate a post NAV, next trading day warning system with leakage controls and a fixed review budget.")
doc.add_heading("2.3 Prediction, calibration and temporal evaluation", level=2)
add_body(doc,"The abnormal outcome is rare by construction, so PR AUC is more informative than ROC AUC for the primary comparison [14]. Probability quality is evaluated separately because discrimination does not guarantee calibration [15,20]. The Brier score is a proper scoring rule for probabilistic forecasts [19]. Cerqueira, Torgo and Mozetic [17] show that repeated out of sample evaluation is particularly appropriate when time series are nonstationary. Moving block resampling follows the dependent data logic established by Künsch [18]. These methodological findings support purged chronological folds, calibration reporting and block bootstrap uncertainty rather than random cross validation.")
doc.add_heading("2.4 Interpretability and research gap", level=2)
add_body(doc,"Rudin [16] argues that interpretable models should be treated as serious alternatives in consequential settings rather than explained only after fitting a black box. This principle is relevant to an ETF warning system because analysts must understand what triggers review. Among the studies examined, we did not identify an Indian equity ETF study that combines a post NAV prediction timestamp, a rolling ETF specific event definition, purged walk forward evaluation, paired dependent data uncertainty, calibration, feature ablation, label sensitivity and operational alert budgets. The contribution is this integrated design and its evidence, not a claim that price NAV deviations or machine learning are individually new.")
add_caption(doc,"Table 1  Position of the study relative to selected literature")
add_table(doc,["Study","Market and method","Relevant finding","Difference from this study"],[
    ["Reddy and Dhabolkar [6]","Indian ETFs; AR and VECM","Persistent deviations and lead lag behaviour","Explains pricing efficiency; no prospective event warning"],
    ["Goel and Ahluwalia [7]","24 Indian equity ETFs","Discounts, tracking error and operating constraints","Cross fund explanatory design"],
    ["Petajisto [8]","Multiple ETF asset classes","Pricing bands and short horizon mean reversion","No operational classifier or alert budget"],
    ["Bae and Kim [10]","US ETFs; liquidity and tracking error","Illiquidity increases deviation risk","Different market and dependent variable"],
    ["Rappoport and Tuzun [11]","ETF panel; arbitrage and liquidity","Liquidity affects arbitrage efficacy","Studies mechanism rather than next day warnings"],
    ["Current study","NIFTYBEES; purged walk forward models","Ranking, calibration, uncertainty and drift","Single fund pilot with prospective label"]
],[1.25,1.75,1.8,2.2],font_size=7.8,aligns=[WD_ALIGN_PARAGRAPH.LEFT]*4)

doc.add_heading("3 Research Questions and Hypotheses", level=1)
add_body(doc,"The primary research question is whether complex machine learning models provide more reliable next trading day warnings of abnormal NIFTYBEES price NAV dislocations than transparent benchmark rules when market conditions change over time.")
add_caption(doc,"Table 2  Prespecified hypotheses and decision evidence")
add_table(doc,["Hypothesis","Statement","Primary evidence"],[
    ["H1 Persistence","A larger current dislocation is associated with higher next day abnormal risk.","Gap rules, persistence benchmark and 10 percent alert lift"],
    ["H2 Incremental information","Market stress and liquidity add value beyond gap history.","Feature group ablation"],
    ["H3 Complexity","Nonlinear classifiers do not consistently beat transparent rules.","Walk forward PR AUC and paired bootstrap"],
    ["H4 Temporal shift","Prevalence and distribution shifts affect calibration and alert volume.","Yearly calibration, PSI and locked cutoffs"],
    ["H5 Operational utility","Limited capacity ranking identifies events above random review.","5, 10 and 20 percent alert budgets"]
],[1.05,3.15,2.8],font_size=8.2,aligns=[WD_ALIGN_PARAGRAPH.CENTER,WD_ALIGN_PARAGRAPH.LEFT,WD_ALIGN_PARAGRAPH.LEFT])

doc.add_heading("4 Data and Data Quality", level=1)
add_body(doc,"The observation unit is one NIFTYBEES trading date. Security level price, volume, turnover, trade count, delivery and intraday range fields come from NSE [1]. Official NAV observations come from AMFI scheme code 140084 and ISIN INF204KB14I2 [2]. NIFTY 50 data provide the underlying market control [3], and official India VIX observations provide market stress information [4]. SEBI's March 2026 Master Circular states that the NAV for domestic schemes is normally disclosed by 11 p.m. on the same business day [5]. The forecast timestamp is therefore after the date t NAV becomes available.")
add_caption(doc,"Table 3  Final data coverage")
add_table(doc,["Dataset","Rows","Coverage","Use"],[
    ["NIFTYBEES market data","1,456","1 Jan 2021 to 16 Nov 2026","Price and liquidity"],
    ["Official AMFI NAV","1,451","1 Jan 2021 to 16 Nov 2026","Price NAV dislocation"],
    ["NIFTY 50","1,456","1 Jan 2021 to 16 Nov 2026","Underlying market"],
    ["India VIX","1,455","1 Jan 2021 to 16 Nov 2026","Market stress"],
    ["Complete merged panel","1,446","1 Jan 2021 to 16 Nov 2026","Feature and label construction"],
    ["Leakage safe forecasts","1,185","12 Jan 2022 to 16 Nov 2026","Model evaluation"]
],[2.0,.75,2.05,2.2],font_size=8.5)
add_body(doc,"Eight special or Muhurat trading sessions have an NSE record but no matching AMFI NAV. Three AMFI NAV dates have no NSE market session. India VIX values of zero on 12 February 2021 and 30 March 2021 are treated as source anomalies and excluded from complete cases. Extreme dislocations are retained after source verification; the largest is 3.010 percent on 4 June 2024. These decisions are recorded at row level in the audit workbook and processing scripts.")

doc.add_heading("5 Methodology", level=1)
doc.add_heading("5.1 Prediction timestamp and prospective label", level=2)
add_body(doc,"For prediction date t, signed dislocation measures the percentage difference between the ETF closing price and official NAV. Positive values indicate a premium and negative values a discount.")
eq1=TMP/"equation_dislocation.png"; equation_png(r"$d_t = 100 \times \frac{P_t - NAV_t}{NAV_t}$",eq1)
p=doc.add_paragraph();p.alignment=WD_ALIGN_PARAGRAPH.CENTER;p.add_run().add_picture(str(eq1),width=Inches(3.2))
add_body(doc,"The abnormal event is relative to the ETF's own recent history. The threshold uses only information available through t.")
eq2=TMP/"equation_label.png"; equation_png(r"$Y_{t+1}=1\;\;\mathrm{if}\;\;|d_{t+1}|>Q_{0.95}(|d_{t-251}|,\ldots,|d_t|)$",eq2)
p=doc.add_paragraph();p.alignment=WD_ALIGN_PARAGRAPH.CENTER;p.add_run().add_picture(str(eq2),width=Inches(5.5))
add_body(doc,"A row enters modelling only when 252 complete historical observations are available, the following actual NSE session has a complete target and all predictors are available on t. The future target cannot influence its own threshold. One overlapping label interval is purged at each train validation and validation test boundary.")
doc.add_heading("5.2 Predictors, benchmarks and classifiers", level=2)
add_body(doc,"The 25 predictors cover current and lagged dislocations, five and 20 day dislocation summaries, ETF and NAV returns, NIFTY 50 returns, ETF minus index return differences, India VIX level and changes, volume, turnover, trade value, delivery percentage and intraday range. Volume and turnover are log transformed. Logistic scaling is fitted on training data only.")
add_body(doc,"Benchmarks include historical prevalence, current absolute gap, current gap divided by its rolling threshold and previous event persistence. Classifiers are L2 regularized logistic regression, a bounded decision tree and a 400 tree random forest. Validation PR AUC selects hyperparameters, with Brier score as a tie breaker. Random seed 42 is fixed.")
doc.add_heading("5.3 Purged walk forward evaluation", level=2)
add_caption(doc,"Table 4  Expanding window design")
add_table(doc,["Fold","Training targets","Validation targets","Test targets","Test rows and events"],[
    ["1","Through 2022","2023","2024","242 and 26"],
    ["2","Through 2023","2024","2025","246 and 5"],
    ["3","Through 2024","2025","2026 to 16 Nov","214 and 34"]
],[.55,1.55,1.45,1.55,1.9],font_size=8.5)
add_body(doc,"Each classifier is retuned in every fold using the validation period. The subsequent test period is evaluated once. This repeated subsequent year design is appropriate for assessing nonstationary temporal data [17].")
doc.add_heading("5.4 Evaluation and robustness analyses", level=2)
add_body(doc,"PR AUC is the primary ranking metric. ROC AUC is secondary. Brier score, log loss, calibration intercept, calibration slope and five equal frequency bin expected calibration error assess probability quality. Operational performance is measured by precision, recall, lift and number needed to review at 5, 10 and 20 percent capacities. Cutoff ties use expected selection within the tied group.")
add_body(doc,"Uncertainty is estimated with 2,000 paired moving block resamples using five observation blocks and a ten observation sensitivity check [18]. Robustness analyses cover rolling 90th, 95th and 97.5th percentile labels; fixed 0.50 and 1.00 percent labels; five feature groups; and validation locked cutoffs applied to the subsequent test year. Population Stability Index is used as a descriptive monitoring statistic, with values below 0.10 labelled low, 0.10 to below 0.25 moderate and at least 0.25 large.")

page_break(doc)
doc.add_heading("6 Results", level=1)
doc.add_heading("6.1 Walk forward model comparison", level=2)
add_caption(doc,"Table 5  Mean untouched test performance across three walk forward folds")
add_table(doc,["Model","Mean PR AUC","Minimum","Maximum","10 percent precision","10 percent recall"],[
    ["Historical prevalence","0.096","0.020","0.159","9.6%","10.3%"],
    ["Current absolute gap","0.218","0.125","0.277","22.4%","29.5%"],
    ["Current gap ratio","0.233","0.116","0.293","26.8%","32.7%"],
    ["Previous event persistence","0.154","0.056","0.214","27.9%","39.7%"],
    ["Logistic regression","0.169","0.035","0.267","18.6%","14.6%"],
    ["Decision tree","0.156","0.100","0.206","20.9%","39.3%"],
    ["Random forest","0.184","0.077","0.274","18.1%","26.2%"]
],[1.8,1.0,.8,.8,1.3,1.3],font_size=8.2)
add_body(doc,"The current gap ratio has the highest mean PR AUC and wins the 2024 and 2026 test folds. The current absolute gap wins the sparse 2025 fold. Random forest is validation selected in every fold but wins no subsequent test fold. H1 is supported because current gap information consistently ranks future events above prevalence. H3 is also supported at the consistency level: model complexity does not reliably dominate transparent rules.")
add_figure(doc,FIG/"ETF_Walk_Forward_PR_AUC.png","Figure 1  Purged walk forward test PR AUC by fold",6.4,"Source  Authors' calculations from the leakage safe modelling dataset.")

doc.add_heading("6.2 Dependent data uncertainty", level=2)
add_caption(doc,"Table 6  Paired five observation block bootstrap comparison")
add_table(doc,["Comparison","Observed PR AUC difference","95 percent interval","Interpretation"],[
    ["Gap ratio minus logistic regression","0.064","-0.012 to 0.164","Interval includes zero"],
    ["Gap ratio minus decision tree","0.078","0.019 to 0.168","Positive at 95 percent"],
    ["Gap ratio minus random forest","0.049","-0.024 to 0.128","Interval includes zero"]
],[2.1,1.3,1.35,2.25],font_size=8.3,aligns=[WD_ALIGN_PARAGRAPH.LEFT,WD_ALIGN_PARAGRAPH.CENTER,WD_ALIGN_PARAGRAPH.CENTER,WD_ALIGN_PARAGRAPH.LEFT])
add_body(doc,"The gap ratio mean PR AUC interval is 0.160 to 0.349. Its advantage over the decision tree remains positive with both five and ten observation blocks. Intervals against logistic regression and random forest include zero. The appropriate conclusion is therefore that the gap ratio has the best point estimate and a demonstrated advantage over the tree, while superiority over the other two classifiers remains uncertain.")
add_figure(doc,FIG/"ETF_Bootstrap_Paired_Differences.png","Figure 2  Paired block bootstrap differences in mean PR AUC",6.2,"Bars show percentile 95 percent intervals from 2,000 valid five observation block resamples.")

doc.add_heading("6.3 Label sensitivity", level=2)
add_caption(doc,"Table 7  Winner under alternative rolling event definitions")
add_table(doc,["Label","Events in modelling data","Highest mean PR AUC","Gap ratio 10 percent precision","Interpretation"],[
    ["Rolling 90th percentile","182","Random forest 0.299","35.2%","Forest leads by 0.010"],
    ["Rolling 95th percentile","80","Gap ratio 0.233","26.8%","Primary result"],
    ["Rolling 97.5th percentile","37","Random forest 0.286","21.1%","Forest leads by 0.002"],
    ["Fixed 0.50 percent","12","Infeasible","Not estimated","Too few events"],
    ["Fixed 1.00 percent","2","Infeasible","Not estimated","Too few events"]
],[1.5,1.25,1.35,1.45,1.45],font_size=8.0)
add_body(doc,"The exact PR AUC winner changes with the rolling percentile. However, the current gap ratio has the highest mean precision, recall and lift at the 10 percent alert budget for all three feasible rolling definitions. This separates ranking sensitivity from operational usefulness. The fixed thresholds are reported as infeasible rather than discarded, because their event counts cannot support the planned fold structure.")
add_figure(doc,FIG/"ETF_Label_Sensitivity_PR_AUC.png","Figure 3  Mean walk forward PR AUC under alternative labels",6.4,"Fixed thresholds are excluded from model comparison because their positive counts are insufficient.")

doc.add_heading("6.4 Alert budget and workload trade off", level=2)
add_caption(doc,"Table 8  Pooled current gap ratio performance by review capacity")
add_table(doc,["Budget","Alerts","Events found","Precision","Recall","Lift","Marginal precision"],[
    ["5%","37","11","29.7%","16.9%","3.21x","Not applicable"],
    ["10%","72","19","26.4%","29.2%","2.85x","22.9%"],
    ["20%","142","27","19.0%","41.5%","2.05x","11.4%"]
],[.7,.75,.95,.9,.9,.75,1.25],font_size=8.4)
add_body(doc,"The current gap ratio has the highest pooled precision at every prespecified capacity. Moving from 5 to 10 percent adds 35 reviews and eight events, while moving from 10 to 20 percent adds 70 reviews and only eight additional events. H5 is supported because precision remains about two to three times prevalence. A 10 percent capacity provides a defensible balance between workload and coverage, although the decision maker can choose another capacity based on resources.")
add_figure(doc,FIG/"ETF_Gap_Ratio_Budget_Tradeoff.png","Figure 4  Current gap ratio workload and coverage trade off",6.15,"Cutoff ties are handled through expected selection rather than row order.")

doc.add_heading("6.5 Feature group ablation", level=2)
add_caption(doc,"Table 9  Mean PR AUC by feature group")
add_table(doc,["Feature group","Predictors","Logistic","Tree","Forest"],[
    ["Gap only","6","0.160","0.171","0.169"],
    ["Gap history","10","0.153","0.153","0.178"],
    ["Gap plus market stress","15","0.195","0.160","0.173"],
    ["Gap plus liquidity","16","0.150","0.163","0.176"],
    ["Full model","25","0.169","0.156","0.184"],
    ["Gap ratio reference","1","0.233","0.233","0.233"]
],[2.25,.9,1.15,1.15,1.15],font_size=8.5)
add_body(doc,"H2 receives partial support. Adding NIFTY 50 and India VIX variables to gap history raises logistic regression mean PR AUC from 0.153 to 0.195 and improves it in all three folds. The mean 10 percent precision increases from 15.4 to 23.9 percent. The effect does not generalize to random forest, and liquidity variables do not produce consistent PR AUC or Brier improvements. More predictors therefore do not automatically improve out of time performance.")
add_figure(doc,FIG/"ETF_Feature_Ablation_PR_AUC.png","Figure 5  Classifier PR AUC by feature group",6.25,"Every classifier is retuned within each feature group and fold.")

doc.add_heading("6.6 Temporal shift and calibration", level=2)
add_caption(doc,"Table 10  Outcome and operational regime changes")
add_table(doc,["Test year","Events","Prevalence","PR AUC winner","Median predictor PSI","Locked alert range"],[
    ["2024","26","10.7%","Gap ratio 0.293","0.141","15.7 to 28.1%"],
    ["2025","5","2.0%","Absolute gap 0.125","0.191","0.0 to 5.3%"],
    ["2026","34","15.9%","Gap ratio 0.290","0.222","22.9 to 39.3%"]
],[.75,.65,.9,1.55,1.35,1.5],font_size=8.3)
add_body(doc,"H4 is supported. Event prevalence changes by a factor of 7.8 between the lowest and highest years. Predictor and score distributions shift, rankings change and no classifier remains well calibrated in every period. Logistic regression predicts 32.3 percent mean risk against 10.7 percent observed in 2024, 2.8 percent against 2.0 percent in 2025 and 15.1 percent against 15.9 percent in 2026. Its calibration slope remains below one in every year.")
add_body(doc,"A validation score cutoff intended for 10 percent alerts issues no logistic alerts in 2025 but 22.9 percent in 2026. The gap ratio cutoff issues 1.6 percent in 2025 and 39.3 percent in 2026. The operational system should therefore rank a fixed capacity rather than rely on a permanent score threshold.")
add_figure(doc,FIG/"ETF_Temporal_Shift_Outcomes.png","Figure 6  Outcome regime changes across test years",6.4,"The 2026 period ends on 16 November 2026.")
add_figure(doc,FIG/"ETF_Temporal_Shift_Locked_Alerts.png","Figure 7  Alert volume after applying validation locked cutoffs",6.4,"Constant and binary score benchmarks are omitted because complete ties make percentile cutoffs operationally uninformative.")

page_break(doc)
doc.add_heading("7 Discussion and Scholarly Comparison", level=1)
doc.add_heading("7.1 Persistence and pricing efficiency", level=2)
add_body(doc,"The strong current gap result is consistent with Reddy and Dhabolkar's evidence of persistent Indian ETF deviations [6] and with the short horizon pricing bands and mean reversion described by Petajisto [8] and Hilliard [9]. The present study adds a prospective interpretation: the current gap contains information about whether the next session will exceed an ETF specific abnormality threshold. This remains an association and does not show that a profitable trade is executable.")
doc.add_heading("7.2 Market stress and liquidity", level=2)
add_body(doc,"The market stress improvement in logistic regression is directionally consistent with evidence that ETF activity and volatility interact [12]. The absence of a broad liquidity gain differs from the stronger cross sectional liquidity relationships reported by Bae and Kim [10], Goel and Ahluwalia [7] and Rappoport and Tuzun [11]. The difference is credible rather than contradictory: this study examines one liquid broad market ETF, daily public proxies and a next day event, whereas those studies use broader panels, different liquidity measures and explanatory outcomes. The result suggests that daily volume, turnover, delivery and range do not add stable incremental warning information after gap history for this fund.")
doc.add_heading("7.3 Complexity, calibration and uncertainty", level=2)
add_body(doc,"The random forest's repeated validation selection but absence of test wins shows why model complexity must be judged out of time. The result supports Rudin's argument that transparent alternatives should be evaluated directly in consequential applications [16]. It also aligns with the wider forecasting evidence that no model is universally best and that evaluation design matters when data shift [17]. Probability calibration varies independently of PR AUC, as calibration research predicts [15,20].")
add_body(doc,"The uncertainty analysis prevents an inflated conclusion. The gap ratio has the best observed mean, but only its advantage over the decision tree excludes zero at the 95 percent level. Reporting this limitation increases the credibility of the main result. The evidence supports the gap ratio as the preferred operational benchmark, while logistic regression and random forest remain statistically plausible alternatives.")
doc.add_heading("7.4 What is original in this project", level=2)
add_body(doc,"The project's originality lies in the combination of decisions and controls. It defines an actionable post NAV timestamp, constructs a leakage safe next session label, treats transparent rules as genuine competitors, evaluates repeated subsequent years, quantifies dependent data uncertainty, tests label and feature sensitivity, measures calibration and connects model rankings to a fixed review capacity. The contribution is demonstrated on NIFTYBEES and should be described as a rigorous pilot rather than a universal ETF model.")

doc.add_heading("8 Contribution and Managerial Implications", level=1)
doc.add_heading("8.1 Methodological contribution", level=2)
add_body(doc,"The study provides a reproducible template for prospective financial warning tasks in which labels depend on rolling history. Its boundary purging, validation only tuning, repeated test periods and dependent data intervals reduce several common sources of optimistic bias.")
doc.add_heading("8.2 Empirical contribution", level=2)
add_body(doc,"The evidence shows that current price NAV dislocation is a stronger and more stable warning signal than unrestricted feature expansion. Market stress adds value to a parsimonious logistic model, but public daily liquidity measures do not contribute consistently. The results also quantify how rapidly probability and workload interpretations can fail when event prevalence changes.")
doc.add_heading("8.3 Operational recommendation", level=2)
add_bullet(doc,"Use the current gap to rolling threshold ratio as the first line ranking signal.")
add_bullet(doc,"Review a fixed capacity, with 10 percent as the study's primary balance between precision and recall.")
add_bullet(doc,"Use logistic regression when a probability is required, accompanied by calibration and drift indicators.")
add_bullet(doc,"Monitor event prevalence, feature PSI, score PSI, Brier score and realised alert volume by period.")
add_bullet(doc,"Investigate source timing and extreme values before escalating a warning, particularly on special sessions.")
add_body(doc,"A high score should prompt price NAV comparison, liquidity review and cautious execution. It should not automatically initiate a trade because bid ask spread, market impact, creation unit constraints, intraday iNAV and transaction costs are outside the current target.")

doc.add_heading("9 Limitations and Future Work", level=1)
add_body(doc,"The study covers one broad market ETF and three test years. The 2025 test fold contains only five events, making its ranking and calibration estimates imprecise. Bootstrap intervals remain wide, and most paired classifier comparisons are inconclusive. These limitations are reported rather than hidden.")
add_body(doc,"Closing price and end of day NAV do not capture intraday arbitrage. Bid ask spreads, order book depth, iNAV, authorized participant flows, creation and redemption activity, market impact and transaction costs are unavailable. Consequently, the study cannot estimate executable trading profitability.")
add_body(doc,"The rolling label is useful for risk monitoring but is relative to recent history. Fixed 0.50 and 1.00 percent definitions are economically intuitive but too sparse in the current sample. Future work should extend the history and add carefully reconciled ETFs before revisiting those thresholds.")
add_body(doc,"A multi ETF panel is the most valuable extension. Each fund should be mapped by scheme code and ISIN, with listing history, benchmark, corporate actions, NAV timing and asset class checked before pooling. External validation should test whether the gap ratio remains strong in sectoral, thematic, international, gold and debt ETFs. More frequent data could then support transaction cost and intraday execution analysis.")

doc.add_heading("10 Conclusion", level=1)
add_body(doc,"This project converts ETF pricing efficiency into a leakage controlled next trading day warning problem. Official sources support 1,446 complete NIFTYBEES observations and 1,185 eligible forecasts after the rolling history and boundary rules are applied. Across three purged walk forward tests, the current gap ratio produces the highest mean PR AUC and the best pooled precision at every alert budget. Its advantage over the decision tree is supported by the paired block bootstrap, while advantages over logistic regression and random forest remain uncertain.")
add_body(doc,"The evidence answers the primary question: added model complexity does not provide a more reliable warning system in this sample. Market stress improves one parsimonious classifier, but liquidity and unrestricted feature expansion do not produce general gains. Rankings, calibration and alert volumes vary sharply across years, so a permanent probability threshold is unsuitable.")
add_body(doc,"A feasible implementation is a transparent capacity based monitor that ranks dates, limits the number reviewed, exposes the drivers of risk and tracks drift. This conclusion is useful precisely because it is bounded. It supports a NIFTYBEES pilot and controlled external validation; it does not support automatic trading or a universal claim about Indian ETFs.")

doc.add_heading("11 Reproducibility and Research Integrity", level=1)
add_body(doc,"The repository contains cleaned official inputs, the merged and leakage safe analytical datasets, numbered scripts, frozen configuration, row level predictions, machine readable results, audit workbooks and figures. Running the documented workflow recreates data reconciliation, model comparison, walk forward validation, block bootstrap intervals, label sensitivity, alert budgets, feature ablation and temporal shift diagnostics.")
add_bullet(doc,"Expected counts: 1,456 market rows, 1,451 NAV rows, 1,446 complete cases, 1,185 modelling observations and 80 primary events.")
add_bullet(doc,"Environment: Python 3.11 or 3.12 with pinned package versions in requirements.txt.")
add_bullet(doc,"Execution: python run_all.py from the repository root.")
add_bullet(doc,"Randomness: random seed 42 for model fitting and bootstrap sampling.")
add_bullet(doc,"Test isolation: no test outcome enters preprocessing, feature selection, hyperparameter selection or validation cutoffs.")
p=doc.add_paragraph(style="Body Text");p.add_run("Repository URL  ").bold=True;p.add_run("Insert the public or institution accessible GitHub URL after uploading the final repository package.")
doc.add_heading("11.1 Artificial intelligence use disclosure", level=2)
add_body(doc,"Generative artificial intelligence tools assisted literature discovery, code drafting, language editing and document formatting. The student authors selected the research design, reviewed the cited sources, verified official data, executed the analysis and remain responsible for all interpretations and errors. AI generated text and code were not accepted without human review. No confidential personal data were supplied to the tools.")

page_break(doc)
doc.add_heading("References", level=1)
refs=[
 (1,"National Stock Exchange of India. Security wise price, volume and deliverable position data.","https://www.nseindia.com/report-detail/eq_security"),
 (2,"Association of Mutual Funds in India. Historical NAV download.","https://www.amfiindia.com/net-asset-value/nav-download"),
 (3,"NSE Indices. Historical index reports.","https://www.niftyindices.com/reports"),
 (4,"National Stock Exchange of India. Historical index data including India VIX.","https://www.nseindia.com/reports-indices-historical-index-data"),
 (5,"Securities and Exchange Board of India. Master Circular for Mutual Funds as on 20 March 2026, Chapter 9.","https://www.sebi.gov.in/sebi_data/attachdocs/mar-2026/1774024028162.pdf"),
 (6,"Reddy, Y. V., and Dhabolkar, P. 2020. Pricing Efficiency of Exchange Traded Funds in India. Organizations and Markets in Emerging Economies 11(1), 244-268. doi:10.15388/omee.2020.11.33.","https://doi.org/10.15388/omee.2020.11.33"),
 (7,"Goel, G., and Ahluwalia, E. 2021. Do pricing efficiencies in Indian equity ETF market impact its performance? Global Finance Journal 49, 100654. doi:10.1016/j.gfj.2021.100654.","https://doi.org/10.1016/j.gfj.2021.100654"),
 (8,"Petajisto, A. 2017. Inefficiencies in the Pricing of Exchange Traded Funds. Financial Analysts Journal 73(1), 24-54. doi:10.2469/faj.v73.n1.7.","https://doi.org/10.2469/faj.v73.n1.7"),
 (9,"Hilliard, J. 2014. Premiums and discounts in ETFs: An analysis of the arbitrage mechanism in domestic and international funds. Global Finance Journal 25(2), 90-107.",None),
 (10,"Bae, K., and Kim, D. 2020. Liquidity risk and exchange traded fund returns, variances, and tracking errors. Journal of Financial Economics 138(1), 222-253. doi:10.1016/j.jfineco.2019.02.012.","https://doi.org/10.1016/j.jfineco.2019.02.012"),
 (11,"Rappoport, D. E., and Tuzun, T. 2020. Arbitrage and Liquidity: Evidence from a Panel of Exchange Traded Funds. Finance and Economics Discussion Series 2020-097. doi:10.17016/FEDS.2020.097.","https://doi.org/10.17016/FEDS.2020.097"),
 (12,"Ben-David, I., Franzoni, F., and Moussawi, R. 2018. Do ETFs Increase Volatility? Journal of Finance 73(6), 2471-2535. doi:10.1111/jofi.12727.","https://doi.org/10.1111/jofi.12727"),
 (13,"Pan, K., and Zeng, Y. 2019. ETF Arbitrage Under Liquidity Mismatch. SSRN Working Paper 2895478. doi:10.2139/ssrn.2895478.","https://doi.org/10.2139/ssrn.2895478"),
 (14,"Saito, T., and Rehmsmeier, M. 2015. The Precision Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets. PLOS ONE 10(3), e0118432. doi:10.1371/journal.pone.0118432.","https://doi.org/10.1371/journal.pone.0118432"),
 (15,"Guo, C., Pleiss, G., Sun, Y., and Weinberger, K. Q. 2017. On Calibration of Modern Neural Networks. Proceedings of ICML 2017, 1321-1330.","https://proceedings.mlr.press/v70/guo17a.html"),
 (16,"Rudin, C. 2019. Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. Nature Machine Intelligence 1, 206-215. doi:10.1038/s42256-019-0048-x.","https://doi.org/10.1038/s42256-019-0048-x"),
 (17,"Cerqueira, V., Torgo, L., and Mozetic, I. 2020. Evaluating time series forecasting models: An empirical study on performance estimation methods. Machine Learning 109, 1997-2028. doi:10.1007/s10994-020-05910-7.","https://doi.org/10.1007/s10994-020-05910-7"),
 (18,"Kunsch, H. R. 1989. The jackknife and the bootstrap for general stationary observations. The Annals of Statistics 17(3), 1217-1241. doi:10.1214/aos/1176347265.","https://doi.org/10.1214/aos/1176347265"),
 (19,"Gneiting, T., and Raftery, A. E. 2007. Strictly Proper Scoring Rules, Prediction, and Estimation. Journal of the American Statistical Association 102(477), 359-378. doi:10.1198/016214506000001437.","https://doi.org/10.1198/016214506000001437"),
 (20,"Niculescu-Mizil, A., and Caruana, R. 2005. Predicting good probabilities with supervised learning. Proceedings of the 22nd International Conference on Machine Learning, 625-632. doi:10.1145/1102351.1102430.","https://doi.org/10.1145/1102351.1102430"),
]
for n,text,url in refs: add_reference(doc,n,text,url)

doc.add_heading("Appendix A Reproduction Outputs", level=1)
add_table(doc,["Analysis","Primary reproducible output"],[
    ["Data audit","ETF_Label_and_Leakage_Audit_2021_to_2026.xlsx"],
    ["Walk forward validation","ETF_Walk_Forward_Validation.xlsx and row level predictions"],
    ["Bootstrap uncertainty","ETF_Block_Bootstrap_Uncertainty.xlsx"],
    ["Label sensitivity","ETF_Label_Sensitivity.xlsx"],
    ["Alert budgets","ETF_Alert_Budget_Sensitivity.xlsx"],
    ["Feature ablation","ETF_Feature_Ablation.xlsx"],
    ["Temporal shift","ETF_Temporal_Shift_Diagnostics.xlsx"],
    ["Machine readable results","JSON and CSV files in outputs"],
],[2.2,4.8],font_size=8.7,aligns=[WD_ALIGN_PARAGRAPH.LEFT,WD_ALIGN_PARAGRAPH.LEFT])
add_body(doc,"All reported counts and headline metrics are checked by executable verification scripts. The final GitHub package should be uploaded without changing the relative directory structure.")

# Prevent rows from splitting across pages and normalize paragraph widows.
for table in doc.tables:
    for row in table.rows:
        tr_pr = row._tr.get_or_add_trPr(); cant_split = OxmlElement("w:cantSplit"); tr_pr.append(cant_split)
for p in doc.paragraphs:
    p_pr = p._p.get_or_add_pPr(); widow = OxmlElement("w:widowControl"); p_pr.append(widow)

doc.core_properties.title = "Does Model Complexity Improve Next Day Price NAV Dislocation Warnings"
doc.core_properties.subject = "Leakage controlled walk forward study of NIFTYBEES"
doc.core_properties.author = "Ansh Kotwal, Anish Mayya, Rohit Agarwal"
doc.core_properties.keywords = "NIFTYBEES, ETF, price NAV dislocation, machine learning, walk forward validation"
doc.save(DEST)
print(DEST)


/workspace/scratch/e2d669a31e97/Niftybees-Price-NAV-Early-Warning/report/ETF_Project_Final_Integrated_Draft.docx
Completed in 0.7 seconds.


## Step 8E — Build the two-column Overleaf-style Word report

Original repository script: `scripts/14_build_overleaf_style_report.py`


In [30]:
from copy import deepcopy
from pathlib import Path
import re
from docx import Document
from docx.enum.section import WD_SECTION
from docx.enum.style import WD_STYLE_TYPE
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

ROOT = PROJECT_ROOT
SRC = ROOT / 'report' / 'ETF_Project_Final_Integrated_Draft.docx'
OUT = ROOT / 'report' / 'ETF_Project_Final_Overleaf_Style.docx'
doc=Document(SRC)

# Helpers
def set_cell_margins(cell, top=35, start=45, bottom=35, end=45):
    tc=cell._tc; tcPr=tc.get_or_add_tcPr(); tcMar=tcPr.first_child_found_in('w:tcMar')
    if tcMar is None:
        tcMar=OxmlElement('w:tcMar'); tcPr.append(tcMar)
    for m,v in [('top',top),('start',start),('bottom',bottom),('end',end)]:
        e=tcMar.find(qn('w:'+m))
        if e is None: e=OxmlElement('w:'+m); tcMar.append(e)
        e.set(qn('w:w'),str(v)); e.set(qn('w:type'),'dxa')

def set_columns(sectPr, num, space=360):
    cols=sectPr.find(qn('w:cols'))
    if cols is None:
        cols=OxmlElement('w:cols'); sectPr.append(cols)
    cols.set(qn('w:num'),str(num)); cols.set(qn('w:space'),str(space))

def continuous_sectpr(num):
    sp=OxmlElement('w:sectPr')
    typ=OxmlElement('w:type'); typ.set(qn('w:val'),'continuous'); sp.append(typ)
    pg=OxmlElement('w:pgSz'); pg.set(qn('w:w'),'12240'); pg.set(qn('w:h'),'15840'); sp.append(pg)
    mar=OxmlElement('w:pgMar')
    for k,v in [('top','720'),('right','720'),('bottom','720'),('left','720'),('header','360'),('footer','360'),('gutter','0')]: mar.set(qn('w:'+k),v)
    sp.append(mar); set_columns(sp,num); return sp

def add_break_before(el, current_cols):
    p=OxmlElement('w:p'); pPr=OxmlElement('w:pPr'); pPr.append(continuous_sectpr(current_cols)); p.append(pPr); el.addprevious(p)

def add_break_after(el, current_cols):
    p=OxmlElement('w:p'); pPr=OxmlElement('w:pPr'); pPr.append(continuous_sectpr(current_cols)); p.append(pPr); el.addnext(p)

def remove_page_breaks(p):
    for br in list(p._p.iter(qn('w:br'))):
        if br.get(qn('w:type')) in (None,'page'):
            br.getparent().remove(br)

# Page setup. Title section is one column; body will become two columns.
sec=doc.sections[0]
sec.page_width=Inches(8.5); sec.page_height=Inches(11)
sec.top_margin=Inches(.5); sec.bottom_margin=Inches(.5); sec.left_margin=Inches(.5); sec.right_margin=Inches(.5)
sec.header_distance=Inches(.25); sec.footer_distance=Inches(.25)
for footer in [sec.footer, sec.first_page_footer, sec.even_page_footer]:
    for p in footer.paragraphs: p._element.getparent().remove(p._element)

# Base typography.
styles=doc.styles
for sname in ['Normal','Body Text','List Bullet','Source Note','Caption']:
    if sname in styles:
        st=styles[sname]; st.font.name='Times New Roman'; st._element.rPr.rFonts.set(qn('w:eastAsia'),'Times New Roman')
styles['Normal'].font.size=Pt(10)
styles['Normal'].paragraph_format.space_after=Pt(0)
styles['Normal'].paragraph_format.line_spacing=1.0
styles['Body Text'].font.size=Pt(10)
styles['Body Text'].paragraph_format.first_line_indent=Inches(.16)
styles['Body Text'].paragraph_format.space_after=Pt(2)
styles['Body Text'].paragraph_format.line_spacing=1.0
styles['List Bullet'].font.size=Pt(9.5)
styles['List Bullet'].paragraph_format.left_indent=Inches(.16)
styles['List Bullet'].paragraph_format.first_line_indent=Inches(-.12)
styles['List Bullet'].paragraph_format.space_after=Pt(1)
styles['Caption'].font.size=Pt(8)
styles['Caption'].font.bold=False
styles['Caption'].paragraph_format.alignment=WD_ALIGN_PARAGRAPH.CENTER
styles['Caption'].paragraph_format.space_before=Pt(3)
styles['Caption'].paragraph_format.space_after=Pt(2)
styles['Source Note'].font.size=Pt(7.5)
styles['Source Note'].font.italic=True
styles['Source Note'].paragraph_format.alignment=WD_ALIGN_PARAGRAPH.CENTER
styles['Source Note'].paragraph_format.space_after=Pt(3)

# Title block.
title=doc.paragraphs[0]
title.alignment=WD_ALIGN_PARAGRAPH.CENTER
title.paragraph_format.space_before=Pt(10); title.paragraph_format.space_after=Pt(2)
for r in title.runs: r.font.name='Times New Roman'; r.font.size=Pt(23); r.font.bold=False
# Remove the blue divider used by the earlier one-column report cover.
pBdr=title._p.get_or_add_pPr().find(qn('w:pBdr'))
if pBdr is not None: title._p.get_or_add_pPr().remove(pBdr)
title_style_bdr=styles['Title']._element.pPr.find(qn('w:pBdr')) if styles['Title']._element.pPr is not None else None
if title_style_bdr is not None: styles['Title']._element.pPr.remove(title_style_bdr)
subtitle=doc.paragraphs[1]; subtitle.alignment=WD_ALIGN_PARAGRAPH.CENTER
subtitle.paragraph_format.space_after=Pt(15)
for r in subtitle.runs: r.font.name='Times New Roman'; r.font.size=Pt(12); r.font.italic=False
for idx in [3,4,5]:
    p=doc.paragraphs[idx]; p.alignment=WD_ALIGN_PARAGRAPH.CENTER; p.paragraph_format.space_after=Pt(0)
    for r in p.runs: r.font.name='Times New Roman'; r.font.size=Pt(10.5)
for idx in [7,8,9,11,13]:
    p=doc.paragraphs[idx]; p.alignment=WD_ALIGN_PARAGRAPH.CENTER; p.paragraph_format.space_after=Pt(0)
    for r in p.runs: r.font.name='Times New Roman'; r.font.size=Pt(9.5); r.font.bold=False
# Compact/remove blank title spacer paragraphs.
for idx in [2,6,10,12,14]:
    p=doc.paragraphs[idx]; remove_page_breaks(p); p.paragraph_format.space_after=Pt(0); p.paragraph_format.space_before=Pt(0)
    for r in p.runs: r.font.size=Pt(1)
# End the full-width title block and start two columns before Abstract.
p14=doc.paragraphs[14]
pPr=p14._p.get_or_add_pPr(); pPr.append(continuous_sectpr(1))
set_columns(doc._element.body.sectPr,2)
# Letter setup on final section properties.
body_sp=doc._element.body.sectPr
pg=body_sp.find(qn('w:pgSz')); pg.set(qn('w:w'),'12240'); pg.set(qn('w:h'),'15840')
mar=body_sp.find(qn('w:pgMar'))
for k,v in [('top','720'),('right','720'),('bottom','720'),('left','720'),('header','360'),('footer','360'),('gutter','0')]: mar.set(qn('w:'+k),v)

# Remove forced page breaks from the former cover and before the Introduction.
for p in doc.paragraphs:
    remove_page_breaks(p)

# Merge Abstract label into its text, matching the Overleaf run-in style.
p_abs_h=next(p for p in doc.paragraphs if p.text=='Abstract')
abs_idx=next(i for i,p in enumerate(doc.paragraphs) if p.text=='Abstract')
p_abs=doc.paragraphs[abs_idx+1]
for r in list(p_abs.runs): r._element.getparent().remove(r._element)
r=p_abs.add_run('Abstract—'); r.bold=True; r.italic=True
r=p_abs.add_run('This study evaluates whether complex machine learning models provide more reliable next trading day warnings of abnormal NIFTYBEES price NAV dislocations than transparent benchmark rules. Official NSE market data, AMFI net asset values, NIFTY 50 observations and India VIX data are reconciled for 1 January 2021 through 16 November 2026. The final merged panel contains 1,446 complete observations. A prospective abnormal event occurs when the next trading day’s absolute dislocation exceeds the rolling 95th percentile estimated from the 252 complete observations available on the prediction date. The leakage controlled modelling sample contains 1,185 forecasts and 80 events. Three purged expanding window evaluations compare historical prevalence, current gap rules, previous event persistence, logistic regression, a decision tree and a random forest. The current gap to threshold ratio achieves the highest mean walk forward PR AUC of 0.233, compared with 0.169 for logistic regression, 0.156 for the decision tree and 0.184 for random forest. A paired moving block bootstrap establishes a 0.078 advantage over the decision tree, while intervals against logistic regression and random forest include zero. Market stress variables improve logistic regression from 0.153 to 0.195, but liquidity variables do not generate consistent gains. The gap ratio has the highest pooled precision at 5, 10 and 20 percent alert budgets. Event prevalence changes from 10.7 percent in 2024 to 2.0 percent in 2025 and 15.9 percent in 2026, producing substantial calibration and alert volume instability. The evidence supports a transparent capacity constrained monitoring system with continuing drift checks. It does not support automatic trading or universal superiority of one model.')
p_abs.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY; p_abs.paragraph_format.first_line_indent=Inches(.16); p_abs.paragraph_format.space_after=Pt(2)
p_abs_h._element.getparent().remove(p_abs_h._element)
# Index terms styling.
kw=next(p for p in doc.paragraphs if p.text.startswith('Keywords'))
text=kw.text.replace('Keywords  ','')
for r in list(kw.runs): r._element.getparent().remove(r._element)
r=kw.add_run('Index Terms—'); r.bold=True; r.italic=True
kw.add_run(text)
kw.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY; kw.paragraph_format.space_after=Pt(6)

# IEEE-like section and subsection headings.
roman={1:'I',2:'II',3:'III',4:'IV',5:'V',6:'VI',7:'VII',8:'VIII',9:'IX',10:'X',11:'XI'}
current_main=None; sub_no=0
for p in doc.paragraphs:
    if p.style.name=='Heading 1':
        m=re.match(r'^(\d+)\s+(.*)$',p.text)
        if m:
            current_main=int(m.group(1)); sub_no=0
            p.text=f"{roman[current_main]}.  {m.group(2).upper()}"
        elif p.text=='Executive Summary': p.text='EXECUTIVE SUMMARY'
        elif p.text=='References': p.text='REFERENCES'
        elif p.text.startswith('Appendix A'): p.text='APPENDIX A.  REPRODUCTION OUTPUTS'
        p.alignment=WD_ALIGN_PARAGRAPH.CENTER
        p.paragraph_format.space_before=Pt(7); p.paragraph_format.space_after=Pt(3); p.paragraph_format.keep_with_next=True
        for r in p.runs:
            r.font.name='Times New Roman'; r.font.size=Pt(10); r.font.bold=False; r.font.small_caps=True
    elif p.style.name=='Heading 2':
        sub_no+=1
        txt=re.sub(r'^\d+\.\d+\s+','',p.text)
        p.text=f"{chr(64+sub_no)}.  {txt}"
        p.alignment=WD_ALIGN_PARAGRAPH.LEFT
        p.paragraph_format.space_before=Pt(5); p.paragraph_format.space_after=Pt(1); p.paragraph_format.keep_with_next=True
        for r in p.runs:
            r.font.name='Times New Roman'; r.font.size=Pt(10); r.font.bold=True; r.font.italic=True

# General body paragraph formatting and remove large spacing.
for pi,p in enumerate(doc.paragraphs):
    if p.style.name in ('Body Text','Normal') and pi >= 15:
        p.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY
        p.paragraph_format.line_spacing=1.0
        if p.style.name=='Normal' and p.text and not p.text.startswith(('Index Terms','Repository URL')):
            p.paragraph_format.first_line_indent=Inches(.16)
        p.paragraph_format.space_after=Pt(2)
        for r in p.runs:
            r.font.name='Times New Roman'; r.font.size=Pt(10); r.font.color.rgb=RGBColor(0,0,0)
    if p.style.name=='List Bullet':
        p.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY
        for r in p.runs: r.font.name='Times New Roman'; r.font.size=Pt(9.5); r.font.color.rgb=RGBColor(0,0,0)

# Caption conversion: TABLE I / Fig. 1.
for p in doc.paragraphs:
    if p.style.name=='Caption':
        m=re.match(r'Table\s+(\d+)\s+(.*)',p.text)
        if m:
            n=int(m.group(1)); rn=['','I','II','III','IV','V','VI','VII','VIII','IX','X','XI'][n]
            p.text=f"TABLE {rn}\n{m.group(2).upper()}"
        else:
            m=re.match(r'Figure\s+(\d+)\s+(.*)',p.text)
            if m: p.text=f"Fig. {m.group(1)}.  {m.group(2)}"
        p.alignment=WD_ALIGN_PARAGRAPH.CENTER; p.paragraph_format.keep_with_next=True
        for r in p.runs: r.font.name='Times New Roman'; r.font.size=Pt(8); r.font.bold=False

# Tables. Wide tables span both columns; smaller tables remain column-width.
wide_indices=set(range(len(doc.tables)))
for ti,table in enumerate(doc.tables):
    table.alignment=WD_TABLE_ALIGNMENT.CENTER; table.autofit=True
    table._tbl.tblPr.tblStyle.set(qn('w:val'),'TableGrid') if table._tbl.tblPr.tblStyle is not None else None
    fs=Pt(7.2 if ti in wide_indices else 6.5)
    for ri,row in enumerate(table.rows):
        for cell in row.cells:
            cell.vertical_alignment=WD_CELL_VERTICAL_ALIGNMENT.CENTER; set_cell_margins(cell)
            for p in cell.paragraphs:
                p.paragraph_format.space_before=Pt(0); p.paragraph_format.space_after=Pt(0); p.paragraph_format.line_spacing=.9
                for r in p.runs:
                    r.font.name='Times New Roman'; r.font.size=fs; r.font.color.rgb=RGBColor(0,0,0)
                    if ri==0: r.font.bold=True
    # Remove blue shading; retain simple black rules.
    for cell in table.rows[0].cells:
        shd=cell._tc.get_or_add_tcPr().find(qn('w:shd'))
        if shd is not None: shd.set(qn('w:fill'),'FFFFFF')
    if ti in wide_indices:
        # Find the caption paragraph directly preceding the table (ignoring blank/break paragraphs).
        prev=table._tbl.getprevious()
        while prev is not None and prev.tag!=qn('w:p'): prev=prev.getprevious()
        if prev is not None: add_break_before(prev,2)
        # The final appendix table can remain in the full-width section; returning
        # to columns after it would strand the closing sentence on a blank page.
        if ti != len(doc.tables)-1:
            add_break_after(table._tbl,1)

# Figures and equations. Charts fit one column; formulas remain compact.
for i,s in enumerate(doc.inline_shapes):
    if i==0:
        if s.width>Inches(3.15):
            ratio=s.height/s.width; s.width=Inches(3.15); s.height=int(s.width*ratio)
    elif i==1:
        ratio=s.height/s.width; s.width=Inches(3.15); s.height=int(s.width*ratio)
    else:
        ratio=s.height/s.width; s.width=Inches(3.15); s.height=int(s.width*ratio)

# References: compact numbered list with hanging indentation.
start=next(i for i,p in enumerate(doc.paragraphs) if p.text=='REFERENCES')+1
for p in doc.paragraphs[start:]:
    if p.text.startswith('APPENDIX A.'): break
    if re.match(r'^\[\d+\]',p.text):
        p.alignment=WD_ALIGN_PARAGRAPH.JUSTIFY
        p.paragraph_format.left_indent=Inches(.16); p.paragraph_format.first_line_indent=Inches(-.16)
        p.paragraph_format.space_after=Pt(1); p.paragraph_format.line_spacing=.9
        for r in p.runs: r.font.name='Times New Roman'; r.font.size=Pt(7.5); r.font.color.rgb=RGBColor(0,0,0)

# Keep title/captions/headings together; remove page numbers and header artifacts.
for p in doc.paragraphs:
    if p.style.name.startswith('Heading') or p.style.name=='Caption': p.paragraph_format.keep_with_next=True
for section in doc.sections:
    for hf in [section.header,section.footer,section.first_page_header,section.first_page_footer,section.even_page_header,section.even_page_footer]:
        for p in hf.paragraphs: p.clear()

# Metadata and save.
doc.core_properties.title='Does Model Complexity Improve Next Day Price NAV Dislocation Warnings'
doc.core_properties.subject='Overleaf-style two-column final integrated report'
doc.save(OUT)
print(OUT)


/workspace/scratch/e2d669a31e97/Niftybees-Price-NAV-Early-Warning/report/ETF_Project_Final_Overleaf_Style.docx
Completed in 0.6 seconds.


## Generated deliverables

After successful execution:

- Cleaned and model-ready datasets are in `data/processed/`.
- Audit workbooks, predictions and machine-readable metrics are in `outputs/`.
- Charts are in `figures/`.
- The revised, integrated and Overleaf-style Word reports are in `report/`.
- Supporting methodology, results and source records are in `docs/`.

The verification cell should print `PASS` before the report-building cells execute.


## Step 9 — Additional operational and robustness analyses
Uses the frozen folds and existing data; no label or test period is redefined.

In [ ]:
# Dislocation-size analysis
%run scripts/15_dislocation_size_analysis.py


[
  {
    "Group": "Not alerted",
    "Observations": 630,
    "Events": 46,
    "Mean_Abs_Dislocation_bps": 12.489355079365076,
    "Median_Abs_Dislocation_bps": 11.325000000000001,
    "P90_Abs_Dislocation_bps": 21.175880000000003,
    "P95_Abs_Dislocation_bps": 26.610134999999982
  },
  {
    "Group": "Alerted",
    "Observations": 72,
    "Events": 19,
    "Mean_Abs_Dislocation_bps": 18.48765972222222,
    "Median_Abs_Dislocation_bps": 17.195699999999995,
    "P90_Abs_Dislocation_bps": 36.55692999999999,
    "P95_Abs_Dislocation_bps": 43.344770000000004
  }
]


In [ ]:
# Delayed-NAV stress test
%run scripts/16_nav_timing_stress_test.py


[
  {
    "Model": "Current Gap Ratio",
    "Folds": 3,
    "Mean_PR_AUC": 0.1600043703188501,
    "Minimum_PR_AUC": 0.03827564398723238,
    "Maximum_PR_AUC": 0.22591396082808168,
    "Mean_Precision_at_10Pct": 0.1709090909090909,
    "Mean_Recall_at_10Pct": 0.18959276018099547
  },
  {
    "Model": "Decision Tree",
    "Folds": 3,
    "Mean_PR_AUC": 0.10969173870689274,
    "Minimum_PR_AUC": 0.056639566395663955,
    "Maximum_PR_AUC": 0.15592150199077082,
    "Mean_Precision_at_10Pct": 0.14424242424242426,
    "Mean_Recall_at_10Pct": 0.21779788838612368
  },
  {
    "Model": "Logistic Regression",
    "Folds": 3,
    "Mean_PR_AUC": 0.14844735845864995,
    "Minimum_PR_AUC": 0.10670670037926676,
    "Maximum_PR_AUC": 0.21839953569897042,
    "Mean_Precision_at_10Pct": 0.14424242424242426,
    "Mean_Recall_at_10Pct": 0.21779788838612368
  },
  {
    "Model": "Random Forest",
    "Folds": 3,
    "Mean_PR_AUC": 0.1610735755506459,
    "Minimum_PR_AUC": 0.05079801151229722,
    "Maximum_P

In [ ]:
# Operational timeline
%run scripts/17_operational_timeline_figure.py


figures/ETF_Operational_Timeline.png


In [ ]:
# Gradient-boosting comparison
%run scripts/18_gradient_boosting_comparison.py


{
  "Mean_PR_AUC": 0.14471129579481548,
  "Minimum_PR_AUC": 0.037210651846330496,
  "Maximum_PR_AUC": 0.21205190973200475,
  "Mean_Precision_at_10Pct": 0.1309090909090909,
  "Mean_Recall_at_10Pct": 0.09728506787330317
}
